# MALLORN: Reproducible TDE Classification

**Ehsan Cheraghi — Python, astronomical time series and rare-event machine learning**

Source: https://github.com/Ehsan-Ch/mallorn-tde-classifier

This notebook contains the project source, a synthetic software check, and optional
official-data training. It does not submit predictions to a competition.

## Recorded results (historical, not generated by this notebook run)

| Evaluation | F1 |
| --- | ---: |
| Kaggle private — completed late submission, 4 October 2026 | **0.6648** |
| Kaggle public — same prediction file | 0.6825 |
| Original nested local baseline | 0.5254 |
| Frozen improvement: development / reserved audit | 0.6912 / 0.5833 |

The top private benchmark was 0.6824. This project does **not** claim to beat it.
Later local selection scores are not comparable private-test scores. The recovered
cycle-5 method is included, but its original trained artifacts were lost in a
workspace rollback; see the repository recovery note. There are no model weights,
raw competition data or fabricated execution outputs embedded here.

## Method

Per-band statistics, dust correction, rest-frame timing, common-phase colours,
GP/thermal and flare features; LightGBM/CatBoost models, auxiliary class targets,
training-fold feature selection, nested evaluation and F1 threshold selection.
Inspired by the published second- and third-place solutions, independently implemented.

CPU is sufficient. Enable Internet for the initial installation. Add the official
MALLORN competition data through **Add Input** after accepting its rules if you
want official training. The default run performs only a short software check.


In [ ]:
SOURCE_FILES = {'src/mallorn/__init__.py': '"""Independent, top-solution-inspired MALLORN research pipeline."""\n\n__version__ = "0.1.0"\n', 'src/mallorn/checkpointing.py': '"""Atomic checkpoints and an OS-owned lock for restartable local experiments."""\nfrom __future__ import annotations\n\nfrom contextlib import contextmanager\nimport json\nimport os\nfrom pathlib import Path\nimport tempfile\n\nimport joblib\n\n\nclass AlreadyRunning(RuntimeError):\n    pass\n\n\n@contextmanager\ndef exclusive_lock(path):\n    """A crashed process releases the lock; a stale filename never blocks work."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\'a+b\') as handle:\n        handle.seek(0)\n        if os.name == \'nt\':\n            import msvcrt\n            if path.stat().st_size == 0:\n                handle.write(b\'0\')\n                handle.flush()\n            handle.seek(0)\n            try:\n                msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)\n            except OSError as exc:\n                raise AlreadyRunning(str(path)) from exc\n        else:\n            import fcntl\n            try:\n                fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\n            except BlockingIOError as exc:\n                raise AlreadyRunning(str(path)) from exc\n        try:\n            yield\n        finally:\n            if os.name == \'nt\':\n                handle.seek(0)\n                msvcrt.locking(handle.fileno(), msvcrt.LK_UNLCK, 1)\n            else:\n                fcntl.flock(handle.fileno(), fcntl.LOCK_UN)\n\n\ndef _atomic(path, writer):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, tmp = tempfile.mkstemp(prefix=path.name + \'.\', suffix=\'.pending\', dir=path.parent)\n    try:\n        with os.fdopen(fd, \'wb\') as handle:\n            writer(handle)\n            handle.flush()\n            os.fsync(handle.fileno())\n        os.replace(tmp, path)\n        if os.name != \'nt\':\n            fd = os.open(path.parent, os.O_RDONLY)\n            try:\n                os.fsync(fd)\n            finally:\n                os.close(fd)\n    finally:\n        if os.path.exists(tmp):\n            os.unlink(tmp)\n\n\ndef atomic_json(path, value):\n    payload = json.dumps(value, indent=2, allow_nan=False).encode() + b\'\\n\'\n    _atomic(path, lambda handle: handle.write(payload))\n\n\ndef atomic_model(path, value):\n    _atomic(path, lambda handle: joblib.dump(value, handle, compress=3))\n\n\ndef freeze_json(path, value):\n    path = Path(path)\n    if path.exists():\n        if json.loads(path.read_text()) != value:\n            raise ValueError(f\'Frozen configuration differs: {path}. Create a new experiment.\')\n    else:\n        atomic_json(path, value)\n', 'src/mallorn/cli.py': '"""Run locally on authorized MALLORN files; never uploads to Kaggle."""\nfrom __future__ import annotations\n\nimport argparse\nfrom dataclasses import asdict\nfrom datetime import datetime,timezone\nfrom importlib.metadata import version\nfrom pathlib import Path\nimport tempfile\n\nimport joblib\nimport pandas as pd\n\nfrom .data import load_dataset,load_groups,sha256,validate_submission\nfrom .features import FeatureConfig,extract_features\nfrom .fixtures import make_fixture\nfrom .modeling import TrainConfig,nested_evaluate,predict_bundle,refit\nfrom .reporting import bootstrap_interval,report,write_json\n\n\ndef prepare(root,feature_cfg):\n    train,test,photo,manifest=load_dataset(root)\n    # Concatenation here only enables per-object extraction; no across-object fit.\n    metadata=pd.concat([train,test],ignore_index=True)\n    x=extract_features(metadata,photo,feature_cfg)\n    return train,test,x,manifest\n\n\ndef run(root: Path,output: Path,cfg: TrainConfig,feature_cfg: FeatureConfig,groups_path=None):\n    if output.exists() and any(output.iterdir()):\n        raise FileExistsError("Output directory is not empty; use a new run directory to preserve evidence")\n    train,test,x,manifest=prepare(root,feature_cfg)\n    groups=load_groups(groups_path,train.object_id)\n    train_x=x.loc[train.object_id]\n    output.mkdir(parents=True,exist_ok=True)\n    print(f"Prepared {len(train)} training and {len(test)} test objects; {x.shape[1]} features",flush=True)\n    metrics,oof,audit=nested_evaluate(train_x,train.target.to_numpy(),cfg,groups)\n    synthetic=(root/"SYNTHETIC_FIXTURE.json").is_file()\n    metrics.update(data_kind="synthetic_fixture" if synthetic else "user_supplied_mallorn_format",\n                   generated_at_utc=datetime.now(timezone.utc).isoformat(),feature_count=x.shape[1],\n                   feature_config=feature_cfg.metadata(),\n                   environment={p:version(p) for p in ("numpy","pandas","scipy","scikit-learn","catboost","lightgbm","extinction","joblib")})\n    metrics["conditional_f1_interval"]=bootstrap_interval(oof.target,oof.selected_prediction,groups,cfg.seed)\n    bundle=refit(train_x,train.target.to_numpy(),metrics,cfg)\n    bundle.update(feature_config=asdict(feature_cfg),data_kind=metrics["data_kind"])\n    probabilities,predictions=predict_bundle(bundle,x.loc[test.object_id])\n    sample=pd.read_csv(root/"sample_submission.csv")\n    indexed=pd.Series(predictions,index=test.object_id)\n    submission=sample[["object_id"]].copy()\n    submission["prediction"]=submission.object_id.map(indexed).astype(int)\n    validate_submission(submission,test.object_id)\n    name="synthetic_submission.csv" if synthetic else "submission.csv"\n    submission.to_csv(output/name,index=False)\n    pd.DataFrame({"object_id":test.object_id,"probability":probabilities}).to_csv(output/"test_probabilities.csv",index=False,float_format="%.12g")\n    oof.to_csv(output/"oof.csv",index=False,float_format="%.12g")\n    x.to_csv(output/"features.csv",float_format="%.12g")\n    joblib.dump(bundle,output/"model.joblib")\n    manifest.update(feature_names=list(x.columns),feature_config=feature_cfg.metadata(),\n                    feature_matrix_sha256=sha256(output/"features.csv"),\n                    groups_sha256=sha256(groups_path) if groups_path is not None else None)\n    write_json(output/"manifest.json",manifest)\n    write_json(output/"fold_audit.json",audit)\n    write_json(output/"metrics.json",metrics)\n    report(output,metrics,oof)\n    print(f"Completed {metrics[\'data_kind\']}; nested local F1={metrics[\'primary\'][\'f1\']:.4f}. Not a Kaggle score.",flush=True)\n    return metrics\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    subs=parser.add_subparsers(dest="command",required=True)\n    train=subs.add_parser("run",help="Validate files, engineer features, run nested CV, refit and generate a local submission")\n    train.add_argument("--data",type=Path,required=True)\n    train.add_argument("--output",type=Path,required=True)\n    train.add_argument("--groups",type=Path,help="Optional real source-template mapping: object_id,group_id")\n    train.add_argument("--outer-folds",type=int,default=5)\n    train.add_argument("--inner-folds",type=int,default=3)\n    train.add_argument("--iterations",type=int,default=500)\n    train.add_argument("--seed",type=int,default=42)\n    train.add_argument("--threads",type=int,default=2)\n    train.add_argument("--backend",choices=["lightgbm","hybrid"],default="lightgbm",help="hybrid adds CatBoost and a fixed blend; needs a compatible host")\n    train.add_argument("--no-deredden",action="store_true")\n    smoke=subs.add_parser("smoke",help="Run an explicitly synthetic software test, not a benchmark")\n    smoke.add_argument("--output",type=Path,required=True)\n    prediction=subs.add_parser("predict",help="Replay test predictions from a locally trusted model")\n    prediction.add_argument("--data",type=Path,required=True)\n    prediction.add_argument("--model",type=Path,required=True)\n    prediction.add_argument("--output",type=Path,required=True)\n    check=subs.add_parser("check-submission")\n    check.add_argument("--submission",type=Path,required=True)\n    check.add_argument("--test-log",type=Path,required=True)\n    args=parser.parse_args()\n    if args.command=="run":\n        cfg=TrainConfig(args.outer_folds,args.inner_folds,args.iterations,args.seed,args.threads,args.backend)\n        cfg.validate()\n        run(args.data,args.output,cfg,FeatureConfig(deredden=not args.no_deredden),args.groups)\n    elif args.command=="smoke":\n        with tempfile.TemporaryDirectory(prefix="mallorn_synthetic_") as temporary:\n            root=Path(temporary)\n            make_fixture(root)\n            run(root,args.output,TrainConfig(3,2,12,42,1),FeatureConfig())\n    elif args.command=="predict":\n        if args.output.exists():\n            raise FileExistsError("Prediction file exists; choose a new output")\n        # joblib uses pickle. Never load a model from an untrusted third party.\n        bundle=joblib.load(args.model)\n        _,test,x,_=prepare(args.data,FeatureConfig(**bundle["feature_config"]))\n        _,pred=predict_bundle(bundle,x.loc[test.object_id])\n        sample=pd.read_csv(args.data/"sample_submission.csv")\n        sample["prediction"]=sample.object_id.map(pd.Series(pred,index=test.object_id)).astype(int)\n        validate_submission(sample,test.object_id)\n        args.output.parent.mkdir(parents=True,exist_ok=True)\n        sample.to_csv(args.output,index=False)\n        print(f"Wrote {len(sample)} predictions; model data kind: {bundle[\'data_kind\']}")\n    else:\n        validate_submission(pd.read_csv(args.submission),pd.read_csv(args.test_log).object_id)\n        print("Submission schema, object set and binary predictions are valid")\n\n\nif __name__=="__main__":\n    main()\n', 'src/mallorn/data.py': '"""Strict data contracts for the official MALLORN CSV layout."""\nfrom __future__ import annotations\n\nimport hashlib\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nBANDS = tuple("ugrizy")\nPHOTOMETRY = ("object_id", "Time (MJD)", "Flux", "Flux_err", "Filter")\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef validate_log(frame: pd.DataFrame, train: bool) -> pd.DataFrame:\n    required = {"object_id", "Z", "EBV"} | ({"target"} if train else set())\n    if not required.issubset(frame):\n        raise ValueError(f"Missing metadata columns: {sorted(required - set(frame))}")\n    frame = frame.copy()\n    if frame.empty or frame.object_id.isna().any() or frame.object_id.duplicated().any():\n        raise ValueError("Metadata IDs must be nonempty, unique and non-null")\n    frame["object_id"] = frame.object_id.astype(str)\n    if frame.object_id.str.strip().eq("").any() or frame.object_id.duplicated().any():\n        raise ValueError("Empty or duplicate normalized object ID")\n    for col in ("Z", "EBV"):\n        frame[col] = pd.to_numeric(frame[col], errors="raise")\n        invalid = (frame[col] <= -1) if col == "Z" else (frame[col] < 0)\n        if not np.isfinite(frame[col]).all() or invalid.any():\n            raise ValueError(f"{col} must be finite; require Z > -1 and EBV >= 0")\n    if train:\n        if frame.target.isna().any() or not frame.target.isin([0, 1]).all():\n            raise ValueError("target must contain binary 0/1 labels")\n        if frame.target.nunique() != 2:\n            raise ValueError("Training data must contain both classes")\n        frame["target"] = frame.target.astype(int)\n    return frame\n\n\ndef validate_photometry(frame: pd.DataFrame) -> pd.DataFrame:\n    if not set(PHOTOMETRY).issubset(frame):\n        raise ValueError(f"Missing photometry columns: {sorted(set(PHOTOMETRY) - set(frame))}")\n    frame = frame.loc[:, list(PHOTOMETRY)].copy()\n    if frame.empty or frame.object_id.isna().any() or frame.Filter.isna().any():\n        raise ValueError("Photometry is empty or contains null IDs/bands")\n    frame["object_id"] = frame.object_id.astype(str)\n    if frame.object_id.str.strip().eq("").any():\n        raise ValueError("Empty photometry object ID")\n    frame["Filter"] = frame.Filter.astype(str).str.strip().str.lower()\n    if not set(frame.Filter).issubset(BANDS):\n        raise ValueError("Unexpected filter; expected u,g,r,i,z,y")\n    for col in ("Time (MJD)", "Flux", "Flux_err"):\n        frame[col] = pd.to_numeric(frame[col], errors="raise")\n        values = frame[col].dropna() if col == "Flux" else frame[col]\n        if not np.isfinite(values).all():\n            raise ValueError(f"Non-finite photometry: {col}")\n    if (frame.Flux_err <= 0).any():\n        raise ValueError("Flux uncertainties must be strictly positive")\n    if frame.duplicated(["object_id", "Filter", "Time (MJD)"]).any():\n        raise ValueError("Duplicate object/filter/time observations; resolve explicitly")\n    # Official files contain absent flux measurements. Never impute a measured\n    # flux from other objects; omit these observations and audit counts in loader.\n    frame = frame.dropna(subset=["Flux"])\n    if frame.empty:\n        raise ValueError("No measured flux observations remain")\n    return frame.sort_values(["object_id", "Filter", "Time (MJD)"], kind="stable").reset_index(drop=True)\n\n\ndef load_dataset(root: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:\n    root = root.resolve()\n    for name in ("train_log.csv", "test_log.csv", "sample_submission.csv"):\n        if not (root / name).is_file():\n            raise FileNotFoundError(f"Missing {root / name}. Download and extract the official data after accepting Kaggle rules.")\n    train = validate_log(pd.read_csv(root / "train_log.csv"), train=True)\n    test = validate_log(pd.read_csv(root / "test_log.csv"), train=False)\n    if set(train.object_id) & set(test.object_id):\n        raise ValueError("Train/test object IDs overlap")\n    sample = pd.read_csv(root / "sample_submission.csv")\n    validate_submission(sample, test.object_id)\n    frames, files = [], [root / n for n in ("train_log.csv", "test_log.csv", "sample_submission.csv")]\n    cleaning = {}\n    for mode, log in (("train", train), ("test", test)):\n        paths = sorted(root.rglob(f"{mode}_full_lightcurves.csv"))\n        if not paths:\n            raise FileNotFoundError(f"No {mode}_full_lightcurves.csv under {root}")\n        raw = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)\n        photo = validate_photometry(raw)\n        cleaning[mode] = {"input_rows": len(raw), "missing_flux_rows_removed": len(raw)-len(photo),\n                          "retained_rows": len(photo), "negative_redshifts_floored_in_features": int((log.Z < 0).sum())}\n        if set(photo.object_id) != set(log.object_id):\n            raise ValueError(f"{mode} photometry/metadata object sets differ")\n        if "split" in log:\n            for path in paths:\n                ids = set(pd.read_csv(path, usecols=["object_id"]).object_id.astype(str))\n                expected = set(log.loc[log.split.astype(str).eq(path.parent.name), "object_id"])\n                if ids != expected:\n                    raise ValueError(f"Metadata split disagrees with folder: {path.parent.name}")\n        frames.append(photo)\n        files.extend(paths)\n    manifest = {"files": [{"path": str(p.relative_to(root)), "bytes": p.stat().st_size,\n                            "sha256": sha256(p)} for p in files],\n                "train_objects": len(train), "test_objects": len(test),\n                "positive_objects": int(train.target.sum()),\n                "photometry_rows": sum(map(len, frames)), "cleaning": cleaning}\n    return train, test, pd.concat(frames, ignore_index=True), manifest\n\n\ndef validate_submission(frame: pd.DataFrame, expected_ids) -> None:\n    if list(frame.columns) != ["object_id", "prediction"]:\n        raise ValueError("Submission columns must be object_id,prediction in that order")\n    if frame.object_id.isna().any() or frame.object_id.duplicated().any():\n        raise ValueError("Submission IDs must be unique and non-null")\n    if set(frame.object_id.astype(str)) != set(map(str, expected_ids)):\n        raise ValueError("Submission IDs do not exactly match the test objects")\n    if frame.prediction.isna().any() or not frame.prediction.isin([0, 1]).all():\n        raise ValueError("Submission requires binary predictions, not probabilities")\n\n\ndef load_groups(path: Path | None, ids: pd.Series) -> np.ndarray | None:\n    if path is None:\n        return None\n    frame = pd.read_csv(path, dtype={"object_id": str, "group_id": str})\n    if not {"object_id", "group_id"}.issubset(frame):\n        raise ValueError("Group mapping requires object_id,group_id")\n    if frame.object_id.duplicated().any() or frame[["object_id", "group_id"]].isna().any().any():\n        raise ValueError("Invalid group mapping")\n    if set(frame.object_id) != set(ids):\n        raise ValueError("Group mapping must cover exactly the training IDs")\n    return frame.set_index("object_id").loc[ids, "group_id"].to_numpy()\n', 'src/mallorn/evaluation.py': '"""Backend-independent metrics, identical to the frozen cycle-2 definitions.\n\nKept separate so LightGBM research never imports the optional native CatBoost\nbackend. The archived modeling module stays unchanged for hash verification.\n"""\nimport numpy as np\nfrom sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,\n                             precision_recall_curve, precision_score, recall_score, roc_auc_score)\n\ndef check_probability(y, p):\n    y, p = np.asarray(y), np.asarray(p, dtype=float)\n    if y.ndim != 1 or p.shape != y.shape or not len(y):\n        raise ValueError("Mismatched or empty probability vectors")\n    if not np.isin(y, [0,1]).all() or not np.isfinite(p).all() or ((p<0)|(p>1)).any():\n        raise ValueError("Expected binary labels and finite probabilities in [0,1]")\n    return y, p\n\ndef best_threshold(y, p) -> tuple[float, float]:\n    y, p = check_probability(y, p)\n    if len(np.unique(y)) != 2:\n        raise ValueError("Threshold fitting requires both classes")\n    precision, recall, thresholds = precision_recall_curve(y, p)\n    f1 = 2*precision[:-1]*recall[:-1]/np.maximum(precision[:-1]+recall[:-1], 1e-15)\n    # Deterministic tie break: largest threshold, i.e. fewest positive calls.\n    best = np.flatnonzero(np.isclose(f1, np.max(f1), atol=1e-12, rtol=0))[-1]\n    threshold = float(thresholds[best])\n    return threshold, float(f1_score(y, p>=threshold, zero_division=0))\n\ndef score(y, p, predictions) -> dict:\n    y, p = check_probability(y, p)\n    predictions = np.asarray(predictions)\n    if predictions.shape != y.shape or not np.isin(predictions,[0,1]).all():\n        raise ValueError("Expected aligned binary decisions")\n    return {"n": int(len(y)), "positives": int(y.sum()), "predicted_positives": int(predictions.sum()),\n            "f1": float(f1_score(y,predictions,zero_division=0)),\n            "precision": float(precision_score(y,predictions,zero_division=0)),\n            "recall": float(recall_score(y,predictions,zero_division=0)),\n            "average_precision": float(average_precision_score(y,p)),\n            "roc_auc": float(roc_auc_score(y,p)) if len(np.unique(y))==2 else None,\n            "confusion_matrix_tn_fp_fn_tp": confusion_matrix(y,predictions,labels=[0,1]).ravel().tolist()}\n', 'src/mallorn/features.py': '"""Label-free, per-object features for sparse, irregular six-band light curves.\n\nNo fitting across objects, extrapolation across seasons, or use of SpecType.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom itertools import combinations\n\nimport extinction\nimport numpy as np\nimport pandas as pd\nfrom scipy.signal import lombscargle\nfrom scipy.stats import skew\n\nfrom .data import BANDS, validate_photometry\n\n# Representative wavelengths, Angstrom. Monochromatic approximation, not full\n# throughput/SED integration; retained in the manifest for reproducibility.\nWAVELENGTHS = dict(zip(BANDS, [3670., 4825., 6222., 7545., 8691., 9710.]))\nPHASE_EDGES = [-np.inf, -60, -30, -10, 0, 10, 30, 60, 120, np.inf]\nCOLOR_PHASES = [-10, 0, 15, 30, 60]\n\n\n@dataclass(frozen=True)\nclass FeatureConfig:\n    deredden: bool = True\n    max_color_gap_days: float = 60.0\n    min_color_snr: float = 2.0\n\n    def metadata(self) -> dict:\n        return {**asdict(self), "wavelengths_angstrom": WAVELENGTHS,\n                "extinction_law": "Fitzpatrick99; Rv=3.1; monochromatic approximation",\n                "redshift_policy": "Preserve input Z; floor negative noisy estimates to zero for all features; exclude Z_err"}\n\n\ndef dust_scale(ebv: float, band: str) -> float:\n    a_lambda = extinction.fitzpatrick99(np.array([WAVELENGTHS[band]], dtype=float), 3.1 * ebv, 3.1)[0]\n    scale = float(10 ** (0.4 * a_lambda))\n    if not np.isfinite(scale):\n        raise ValueError("Extinction correction overflow")\n    return scale\n\n\ndef safe_skew(values: np.ndarray) -> float:\n    return float(skew(values, bias=False)) if len(values) >= 3 and np.std(values) > 1e-12 else np.nan\n\n\ndef interpolate_observed(t, f, e, query, max_gap, min_snr):\n    """Linear interpolation only between confident, nearby detections."""\n    if not len(t):\n        return np.nan\n    j = int(np.searchsorted(t, query))\n    if j < len(t) and t[j] == query:\n        return float(f[j]) if f[j] / e[j] >= min_snr else np.nan\n    if j == 0 or j == len(t) or t[j] - t[j - 1] > max_gap:\n        return np.nan\n    if min(f[j] / e[j], f[j - 1] / e[j - 1]) < min_snr:\n        return np.nan\n    return float(np.interp(query, t[j-1:j+1], f[j-1:j+1]))\n\n\ndef band_features(t, f, e, peak, z) -> dict:\n    base_names = ["n", "span", "max_gap", "mean", "std", "min", "max", "q10", "q30", "q50", "q90",\n                  "skew", "weighted_mean", "chi2_dof", "snr_max", "snr_mean", "negative_fraction",\n                  "detected_fraction", "width_snr5", "peak_lag", "q90_q50_scale", "q100_q50_scale",\n                  "decay_slope", "decay_r2", "period_rest_days", "period_power"]\n    out = {name: np.nan for name in base_names}\n    for i in range(len(PHASE_EDGES)-1):\n        out.update({f"phase{i}_mean_norm": np.nan, f"phase{i}_snr_mean": np.nan, f"phase{i}_n": 0.})\n    out["n"] = float(len(t))\n    if not len(t):\n        return out\n    rest_t = (t - peak) / (1 + z)\n    snr = f / e\n    scale = max(float(np.max(np.abs(f))), 1e-12)\n    weights = 1 / e**2\n    weighted_mean = float(np.average(f, weights=weights))\n    q10, q30, q50, q90 = np.quantile(f, [.1, .3, .5, .9])\n    detected = snr > 5\n    out.update(n=float(len(t)), span=float(np.ptp(rest_t)),\n               max_gap=float(np.max(np.diff(rest_t))) if len(t) > 1 else np.nan,\n               mean=float(np.mean(f)), std=float(np.std(f)), min=float(min(f)), max=float(max(f)),\n               q10=q10, q30=q30, q50=q50, q90=q90, skew=safe_skew(f), weighted_mean=weighted_mean,\n               chi2_dof=float(np.sum(weights*(f-weighted_mean)**2)/(len(t)-1)) if len(t)>1 else np.nan,\n               snr_max=float(max(snr)), snr_mean=float(np.mean(snr)), negative_fraction=float(np.mean(f<0)),\n               detected_fraction=float(np.mean(detected)),\n               width_snr5=float(np.ptp(rest_t[detected])) if detected.sum()>=2 else np.nan,\n               peak_lag=float(rest_t[np.argmax(f)]), q90_q50_scale=float((q90-q50)/scale),\n               q100_q50_scale=float((max(f)-q50)/scale))\n    for i, (left, right) in enumerate(zip(PHASE_EDGES[:-1], PHASE_EDGES[1:])):\n        mask = (rest_t >= left) & (rest_t < right)\n        out[f"phase{i}_n"] = float(mask.sum())\n        if mask.any():\n            out[f"phase{i}_mean_norm"] = float(np.mean(f[mask])/scale)\n            out[f"phase{i}_snr_mean"] = float(np.mean(snr[mask]))\n    # A descriptive log-log decline statistic, not a fitted physical TDE model.\n    mask = (rest_t >= 5) & (snr > 2)\n    if mask.sum() >= 3 and np.ptp(rest_t[mask]) > 5:\n        x, y = np.log(rest_t[mask]), np.log(f[mask])\n        coef = np.polyfit(x, y, 1, w=snr[mask])\n        residual = np.sum((y - np.polyval(coef, x))**2)\n        total = np.sum((y-y.mean())**2)\n        out["decay_slope"] = float(coef[0])\n        out["decay_r2"] = float(1-residual/total) if total>1e-12 else np.nan\n    if len(t) >= 8 and np.ptp(rest_t) > 20 and np.std(f) > 1e-10:\n        # Uneven cadence: use weighted Lomb-Scargle, not an FFT of raw samples.\n        periods = np.geomspace(5, max(6, np.ptp(rest_t)), 128)\n        power = lombscargle(rest_t, f, 2*np.pi/periods, weights=weights,\n                           floating_mean=True, normalize=True)\n        if np.isfinite(power).any():\n            best = int(np.nanargmax(power))\n            out["period_rest_days"], out["period_power"] = float(periods[best]), float(power[best])\n    return out\n\n\ndef extract_features(log: pd.DataFrame, photo: pd.DataFrame, config: FeatureConfig = FeatureConfig()) -> pd.DataFrame:\n    if log.object_id.duplicated().any():\n        raise ValueError("Duplicate metadata IDs")\n    photo = validate_photometry(photo)\n    if set(photo.object_id) != set(log.object_id):\n        raise ValueError("Feature input object sets must match exactly")\n    rows = {}\n    metadata = log.set_index("object_id")\n    for object_id, lc in photo.groupby("object_id", sort=True):\n        row = metadata.loc[object_id]\n        z, ebv = float(row.Z), float(row.EBV)\n        if not np.isfinite([z, ebv]).all() or z <= -1 or ebv < 0:\n            raise ValueError("Invalid redshift/extinction")\n        # The official test log includes a slightly negative photometric estimate.\n        # Keep the raw file unchanged; apply the same fixed physical floor to\n        # metadata features, rest-frame timing and colours for every object.\n        z = max(z, 0.0)\n        bands = {}\n        for band in BANDS:\n            g = lc.loc[lc.Filter.eq(band)].sort_values("Time (MJD)")\n            factor = dust_scale(ebv, band) if config.deredden else 1.\n            bands[band] = (g["Time (MJD)"].to_numpy(), g.Flux.to_numpy()*factor, g.Flux_err.to_numpy()*factor)\n        # Reference peak from highest-SNR observation in g/r/i, or all bands if absent.\n        reference = lc[lc.Filter.isin(list("gri"))]\n        if reference.empty:\n            reference = lc\n        peak = float(reference.loc[(reference.Flux/reference.Flux_err).idxmax(), "Time (MJD)"])\n        out = {"Z": z, "EBV": ebv, "log1p_Z": float(np.log1p(z)),\n               "total_observations": float(len(lc)), "bands_observed": float(lc.Filter.nunique())}\n        detected_t = lc.loc[(lc.Flux/lc.Flux_err)>5, "Time (MJD)"].to_numpy()\n        out["all_width_snr5"] = float(np.ptp(detected_t)/(1+z)) if len(detected_t)>=2 else np.nan\n        for band, (t, f, e) in bands.items():\n            out.update({f"{band}__{key}": value for key, value in band_features(t, f, e, peak, z).items()})\n        # Common-phase colors avoid ratios of maxima from different dates.\n        for a, b in combinations(BANDS, 2):\n            colors = []\n            for phase in COLOR_PHASES:\n                query = peak + phase * (1+z)\n                fluxes = [interpolate_observed(*bands[c], query, config.max_color_gap_days, config.min_color_snr) for c in (a,b)]\n                color = float(-2.5*np.log10(fluxes[0]/fluxes[1])) if min(fluxes)>0 and np.isfinite(fluxes).all() else np.nan\n                out[f"color_{a}_{b}__p{phase}"] = color\n                colors.append(color)\n            out[f"color_{a}_{b}__evolution_0_30"] = colors[3]-colors[1]\n        rows[object_id] = out\n    result = pd.DataFrame.from_dict(rows, orient="index").reindex(log.object_id)\n    result.index.name = "object_id"\n    result = result.reindex(sorted(result.columns), axis=1).astype(float)\n    if np.isinf(result.to_numpy()).any():\n        raise ValueError("Infinite engineered features")\n    return result\n\n\ndef basic_columns(columns) -> list[str]:\n    return [c for c in columns if not any(term in c for term in ("phase", "decay_", "period_", "color_"))]\n', 'src/mallorn/fixtures.py': '"""Small invented light curves for software checks ONLY; no astronomical validity."""\nfrom pathlib import Path\nimport json\n\nimport numpy as np\nimport pandas as pd\n\n\ndef make_fixture(root: Path, train_count=72, test_count=12, seed=917):\n    root.mkdir(parents=True,exist_ok=True)\n    rng=np.random.default_rng(seed)\n    logs={"train":[],"test":[]}\n    curves={"train":[],"test":[]}\n    for mode,count in (("train",train_count),("test",test_count)):\n        for i in range(count):\n            label=int(i%4==0)\n            object_id=f"synthetic_{mode}_{i:04d}"\n            split=f"split_{i%2+1:02d}"\n            row={"object_id":object_id,"Z":float(rng.uniform(.05,.7)),"Z_err":np.nan if mode=="train" else .1,\n                 "EBV":float(rng.uniform(0,.12)),"split":split,"English Translation":"invented fixture"}\n            if mode=="train": row.update(target=label,SpecType="TDE" if label else "AGN")\n            logs[mode].append(row)\n            peak=rng.uniform(59020,59040)\n            for band_no,band in enumerate("ugrizy"):\n                if i%9==0 and band=="u": continue\n                time=np.sort(rng.uniform(58950,59200,18))\n                shape=(15*np.exp(-((time-peak)/(45 if label else 15))**2)\n                       if i%3 else 5+5*np.sin((time-peak)/22))\n                error=rng.uniform(.6,1.5,len(time))\n                flux=shape*(1+band_no*(.12 if label else .35))+rng.normal(0,error)\n                curves[mode].extend({"object_id":object_id,"Time (MJD)":float(t),"Flux":float(f),\n                                      "Flux_err":float(e),"Filter":band,"split":split}\n                                     for t,f,e in zip(time,flux,error))\n        pd.DataFrame(logs[mode]).to_csv(root/f"{mode}_log.csv",index=False)\n        frame=pd.DataFrame(curves[mode])\n        for split,g in frame.groupby("split"):\n            folder=root/split\n            folder.mkdir(exist_ok=True)\n            g.drop(columns="split").to_csv(folder/f"{mode}_full_lightcurves.csv",index=False)\n    pd.DataFrame({"object_id":[r["object_id"] for r in reversed(logs["test"])],"prediction":0}).to_csv(root/"sample_submission.csv",index=False)\n    (root/"SYNTHETIC_FIXTURE.json").write_text(json.dumps({"synthetic":True,"purpose":"Software tests; no competition evidence"}))\n', 'src/mallorn/modeling.py': '"""Nested object-level CV: model AND F1 threshold chosen inside each outer fold."""\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\n\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostClassifier\nfrom lightgbm import LGBMClassifier\nfrom sklearn.base import BaseEstimator, ClassifierMixin\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,\n                             precision_recall_curve, precision_score, recall_score, roc_auc_score)\nfrom sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold\nfrom sklearn.pipeline import make_pipeline\n\nfrom .features import basic_columns\n\n\nclass PipelineCatBoostClassifier(ClassifierMixin, CatBoostClassifier, BaseEstimator):\n    """Supply sklearn estimator tags for the pinned CatBoost 1.2.8 pipeline.\n\n    CatBoost\'s native fitting is unchanged. sklearn 1.8 requires the public\n    estimator tags API; the mixins follow sklearn\'s documented MRO ordering.\n    """\n\n\n@dataclass(frozen=True)\nclass TrainConfig:\n    outer_folds: int = 5\n    inner_folds: int = 3\n    iterations: int = 500\n    seed: int = 42\n    threads: int = 2\n    backend: str = "lightgbm"\n\n    def validate(self):\n        if min(self.outer_folds, self.inner_folds) < 2 or self.iterations < 1 or self.threads < 1:\n            raise ValueError("Require folds>=2, iterations>=1, threads>=1")\n        if self.backend not in ("lightgbm","hybrid"):\n            raise ValueError("backend must be lightgbm or hybrid")\n\n\ndef base_models(cfg):\n    return ("lgb_basic","lgb_full") + (("cat_full",) if cfg.backend=="hybrid" else ())\n\n\ndef candidate_names(cfg):\n    return ("prior",*base_models(cfg)) + (("blend",) if cfg.backend=="hybrid" else ())\n\n\ndef check_probability(y, p):\n    y, p = np.asarray(y), np.asarray(p, dtype=float)\n    if y.ndim != 1 or p.shape != y.shape or not len(y):\n        raise ValueError("Mismatched or empty probability vectors")\n    if not np.isin(y, [0,1]).all() or not np.isfinite(p).all() or ((p<0)|(p>1)).any():\n        raise ValueError("Expected binary labels and finite probabilities in [0,1]")\n    return y, p\n\n\ndef best_threshold(y, p) -> tuple[float, float]:\n    y, p = check_probability(y, p)\n    if len(np.unique(y)) != 2:\n        raise ValueError("Threshold fitting requires both classes")\n    precision, recall, thresholds = precision_recall_curve(y, p)\n    f1 = 2*precision[:-1]*recall[:-1]/np.maximum(precision[:-1]+recall[:-1], 1e-15)\n    # Deterministic tie break: largest threshold, i.e. fewest positive calls.\n    best = np.flatnonzero(np.isclose(f1, np.max(f1), atol=1e-12, rtol=0))[-1]\n    threshold = float(thresholds[best])\n    return threshold, float(f1_score(y, p>=threshold, zero_division=0))\n\n\ndef score(y, p, predictions) -> dict:\n    y, p = check_probability(y, p)\n    predictions = np.asarray(predictions)\n    if predictions.shape != y.shape or not np.isin(predictions,[0,1]).all():\n        raise ValueError("Expected aligned binary decisions")\n    return {"n": int(len(y)), "positives": int(y.sum()), "predicted_positives": int(predictions.sum()),\n            "f1": float(f1_score(y,predictions,zero_division=0)),\n            "precision": float(precision_score(y,predictions,zero_division=0)),\n            "recall": float(recall_score(y,predictions,zero_division=0)),\n            "average_precision": float(average_precision_score(y,p)),\n            "roc_auc": float(roc_auc_score(y,p)) if len(np.unique(y))==2 else None,\n            "confusion_matrix_tn_fp_fn_tp": confusion_matrix(y,predictions,labels=[0,1]).ravel().tolist()}\n\n\ndef folds(y, groups, count, seed):\n    y = np.asarray(y)\n    if min(np.bincount(y.astype(int), minlength=2)) < count:\n        raise ValueError("Too few examples of each class for requested folds")\n    if groups is None:\n        iterator = StratifiedKFold(count,shuffle=True,random_state=seed).split(np.zeros(len(y)),y)\n    else:\n        groups = np.asarray(groups)\n        if len(groups)!=len(y) or len(np.unique(groups))<count:\n            raise ValueError("Insufficient or misaligned source groups")\n        iterator = StratifiedGroupKFold(count,shuffle=True,random_state=seed).split(np.zeros(len(y)),y,groups)\n    result = list(iterator)\n    seen = np.zeros(len(y), dtype=int)\n    for train, val in result:\n        if set(train)&set(val) or len(np.unique(y[train]))!=2 or len(np.unique(y[val]))!=2:\n            raise ValueError("Invalid fold: overlap or a missing class")\n        if groups is not None and set(groups[train])&set(groups[val]):\n            raise ValueError("Source group leakage")\n        seen[val]+=1\n    if not np.all(seen==1):\n        raise ValueError("Each object must be held out once")\n    return result\n\n\ndef fit_model(name, x, y, cfg: TrainConfig, seed):\n    columns = basic_columns(x.columns) if name.endswith("_basic") else list(x.columns)\n    if name.startswith("cat_"):\n        estimator = PipelineCatBoostClassifier(iterations=cfg.iterations,depth=5,learning_rate=.04,\n            l2_leaf_reg=6,loss_function="Logloss",random_seed=seed,thread_count=cfg.threads,\n            allow_writing_files=False,verbose=False)\n    elif name.startswith("lgb_"):\n        estimator = LGBMClassifier(n_estimators=cfg.iterations,num_leaves=15,max_depth=-1,\n            learning_rate=.035,min_child_samples=20,reg_lambda=5.,colsample_bytree=.85,\n            random_state=seed,n_jobs=cfg.threads,verbosity=-1,deterministic=True,force_col_wise=True)\n    else:\n        raise ValueError(f"Unknown model {name}")\n    # Every imputer (including its missing indicators) is fitted only on fit rows.\n    imputer=SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True).set_output(transform="pandas")\n    pipeline = make_pipeline(imputer,estimator)\n    pipeline.fit(x.loc[:,columns], y)\n    return {"columns":columns,"pipeline":pipeline}\n\n\ndef predict_model(model, x):\n    return model["pipeline"].predict_proba(x.loc[:,model["columns"]])[:,1]\n\n\ndef fit_candidates(x, y, cfg, seed):\n    return {"prior":float(np.mean(y)), **{name:fit_model(name,x,y,cfg,seed) for name in base_models(cfg)}}\n\n\ndef predict_candidates(models, x):\n    result = {"prior":np.full(len(x),models["prior"])}\n    result.update({name:predict_model(model,x) for name,model in models.items() if name!="prior"})\n    if "cat_full" in result:\n        result["blend"] = .5*result["cat_full"]+.5*result["lgb_full"]\n    return result\n\n\ndef choose_candidate(y, probabilities):\n    tuning = {name:dict(zip(("threshold","tuning_f1"),best_threshold(y,p))) for name,p in probabilities.items()}\n    # Prefer the predeclared simpler candidate on an exact F1 tie.\n    selected = max(probabilities,key=lambda name:tuning[name]["tuning_f1"])\n    return selected,tuning\n\n\ndef nested_evaluate(x: pd.DataFrame, y, cfg: TrainConfig, groups=None):\n    cfg.validate()\n    y = np.asarray(y)\n    if not np.isin(y,[0,1]).all():\n        raise ValueError("Labels must be binary before conversion")\n    y = y.astype(int)\n    if len(y)!=len(x) or x.index.duplicated().any() or len(set(x.columns))!=len(x.columns):\n        raise ValueError("Feature rows, IDs or columns invalid")\n    if not np.isin(y,[0,1]).all() or np.isinf(x.to_numpy()).any():\n        raise ValueError("Invalid labels or features")\n    outer = folds(y,groups,cfg.outer_folds,cfg.seed)\n    names=candidate_names(cfg)\n    oof = {name:np.full(len(y),np.nan) for name in names}\n    decisions = {name:np.zeros(len(y),dtype=int) for name in names}\n    selected_p,selected_pred = np.full(len(y),np.nan),np.zeros(len(y),dtype=int)\n    assignments, audits = np.full(len(y),-1), []\n    for fold,(fit_idx,test_idx) in enumerate(outer):\n        print(f"Outer fold {fold+1}/{cfg.outer_folds}: {len(fit_idx)} fit, {len(test_idx)} test",flush=True)\n        fit_x,fit_y = x.iloc[fit_idx],y[fit_idx]\n        fit_groups = np.asarray(groups)[fit_idx] if groups is not None else None\n        inner = folds(fit_y,fit_groups,cfg.inner_folds,cfg.seed+fold+1)\n        inner_p = {name:np.full(len(fit_y),np.nan) for name in names}\n        inner_audit = []\n        for inner_fold,(a,b) in enumerate(inner):\n            models = fit_candidates(fit_x.iloc[a],fit_y[a],cfg,cfg.seed+fold*100+inner_fold)\n            for name,p in predict_candidates(models,fit_x.iloc[b]).items():\n                inner_p[name][b]=p\n            inner_audit.append({"fit_ids":fit_x.index[a].tolist(),"validation_ids":fit_x.index[b].tolist()})\n        selected,tuning = choose_candidate(fit_y,inner_p)\n        models = fit_candidates(fit_x,fit_y,cfg,cfg.seed+fold)\n        predictions = predict_candidates(models,x.iloc[test_idx])\n        for name,p in predictions.items():\n            oof[name][test_idx]=p\n            decisions[name][test_idx]=(p>=tuning[name]["threshold"]).astype(int)\n        selected_p[test_idx] = predictions[selected]\n        selected_pred[test_idx] = decisions[selected][test_idx]\n        assignments[test_idx]=fold\n        audits.append({"fold":fold,"fit_ids":fit_x.index.tolist(),"test_ids":x.index[test_idx].tolist(),\n                       "inner_splits":inner_audit,"selected":selected,"inner_tuning":tuning,\n                       "test_metrics":score(y[test_idx],selected_p[test_idx],selected_pred[test_idx])})\n    metrics = {"validation":"Nested cross-validation; model and threshold selected inside outer training folds",\n               "config":asdict(cfg),"grouping":"source_group" if groups is not None else "object_only",\n               "primary":score(y,selected_p,selected_pred),\n               "candidate_outer_metrics":{name:score(y,oof[name],decisions[name]) for name in names},\n               "baselines":{"all_negative":score(y,np.zeros(len(y)),np.zeros(len(y),int)),\n                            "all_positive":score(y,np.ones(len(y)),np.ones(len(y),int))}}\n    # Deployment selection sees all OOF labels; this tuning score is NOT the unbiased result.\n    final_name,final_tuning = choose_candidate(y,oof)\n    metrics["deployment"] = {"candidate":final_name,**final_tuning[final_name],\n                             "tuning_f1_is_performance_estimate":False}\n    table = pd.DataFrame({"object_id":x.index,"target":y,"outer_fold":assignments,\n                          "selected_probability":selected_p,"selected_prediction":selected_pred})\n    for name in names:\n        table[f"{name}_probability"] = oof[name]\n        table[f"{name}_prediction"] = decisions[name]\n    return metrics,table,audits\n\n\ndef refit(x,y,metrics,cfg):\n    name = metrics["deployment"]["candidate"]\n    names = ("cat_full","lgb_full") if name=="blend" else (() if name=="prior" else (name,))\n    return {"candidate":name,"threshold":metrics["deployment"]["threshold"],\n            "features":list(x.columns),"prior":float(np.mean(y)),\n            "models":{n:fit_model(n,x,y,cfg,cfg.seed) for n in names}}\n\n\ndef predict_bundle(bundle,x):\n    if list(x.columns)!=bundle["features"]:\n        raise ValueError("Inference feature schema/order differs from training")\n    name=bundle["candidate"]\n    if name=="prior":\n        p=np.full(len(x),bundle["prior"])\n    elif name=="blend":\n        p=.5*predict_model(bundle["models"]["cat_full"],x)+.5*predict_model(bundle["models"]["lgb_full"],x)\n    else:\n        p=predict_model(bundle["models"][name],x)\n    check_probability(np.zeros(len(x)),p)\n    return p,(p>=bundle["threshold"]).astype(int)\n', 'src/mallorn/morphology_features.py': '"""Label-free descriptive flare fits and stacked multi-band shape summaries."""\nfrom itertools import combinations\nimport numpy as np\nfrom scipy.optimize import least_squares\nfrom scipy.stats import skew,kurtosis\nfrom .data import BANDS\nfrom .features import dust_scale\n\n\ndef object_features(task):\n    oid,z,ebv,time,flux,error,band=task\n    z=max(float(z),0.)\n    correction=np.array([dust_scale(ebv,b) for b in BANDS])\n    flux=flux*correction[band];error=error*correction[band]\n    core=np.isin(band,[1,2,3]);peak=time[np.flatnonzero(core)[np.argmax((flux/error)[core])]] if core.any() else time[np.argmax(flux/error)]\n    t=(time-peak)/(1+z);out={};statistics={};normalized=np.zeros(len(flux));normerror=np.zeros(len(flux))\n    for i,b in enumerate(BANDS):\n        m=band==i;f=flux[m];e=error[m];tt=t[m]\n        if not len(f):continue\n        scale=max(np.max(f),np.median(e));normalized[m]=f/scale;normerror[m]=e/scale\n        statistics[b]={\'max\':np.max(f),\'q90\':np.quantile(f,.9),\'mean\':np.mean(f),\'std\':np.std(f)}\n        for k in [2,3,5,10,20]:\n            use=f/e>k\n            out[f\'morph_{b}_snr{k}_count\']=float(use.sum())\n            out[f\'morph_{b}_snr{k}_fraction\']=float(use.mean())\n            out[f\'morph_{b}_snr{k}_span\']=float(np.ptp(tt[use])) if use.sum()>1 else np.nan\n        order=np.argsort(tt);tt,f,e=tt[order],f[order],e[order]\n        for channel,values in [(\'flux\',f),(\'snr\',f/e)]:\n            q=np.quantile(values,[.05,.1,.25,.5,.75,.9,.95])\n            amp=max(q[-1]-q[0],1e-12)\n            out[f\'morph_{b}_{channel}_tail_asymmetry\']=float((q[-1]+q[0]-2*q[3])/amp)\n            out[f\'morph_{b}_{channel}_iqr_ratio\']=float((q[4]-q[2])/amp)\n            out[f\'morph_{b}_{channel}_kurtosis\']=float(kurtosis(values)) if np.std(values)>1e-12 else np.nan\n        for side,m in [(\'before\',tt<0),(\'after\',tt>=0),(\'late\',tt>100)]:\n            out[f\'morph_{b}_{side}_mean_norm\']=float(np.mean(f[m])/scale) if m.any() else np.nan\n            out[f\'morph_{b}_{side}_detected_fraction\']=float(np.mean(f[m]/e[m]>3)) if m.any() else np.nan\n    for a,b in combinations(BANDS,2):\n        if a not in statistics or b not in statistics:continue\n        for name in [\'max\',\'q90\',\'mean\',\'std\']:\n            fa,fb=statistics[a][name],statistics[b][name]\n            out[f\'morph_ratio_{a}_{b}_{name}\']=float(np.log(fa/fb)) if min(fa,fb)>0 else np.nan\n    # Morphological shape after independent per-band amplitude normalization.\n    # Fit uses only this object\'s observations, never class labels.\n    use=core&(t>-150)&(t<400)\n    tt,ff,ee=t[use],normalized[use],normerror[use]\n    out[\'morph_stacked_skew\']=float(skew(ff)) if len(ff)>3 and np.std(ff)>1e-12 else np.nan\n    out[\'morph_stacked_kurtosis\']=float(kurtosis(ff)) if len(ff)>3 and np.std(ff)>1e-12 else np.nan\n    if len(tt)>=10:\n        # Floor tiny uncertainties to avoid allowing a single point to dominate\n        # a descriptive fit with deliberately simplified spectral evolution.\n        ee=np.maximum(ee,.03)\n        for kind in [\'exponential\',\'powerlaw\']:\n            def predict(params):\n                center,rise,fall,amplitude,baseline=params\n                phase=tt-center\n                before=np.exp(-.5*(np.minimum(phase,0)/rise)**2)\n                after=np.exp(-np.maximum(phase,0)/fall) if kind==\'exponential\' else (1+np.maximum(phase,0)/fall)**(-5/3)\n                return baseline+amplitude*before*after\n            fit=least_squares(lambda p:(predict(p)-ff)/ee,[0,20,50,1,0],\n                              bounds=([-60,1,1,0,-.5],[60,300,1000,3,1]),max_nfev=100,loss=\'soft_l1\')\n            for name,value in zip([\'center\',\'rise\',\'fall\',\'amplitude\',\'baseline\'],fit.x):out[f\'flare_{kind}_{name}\']=float(value)\n            out[f\'flare_{kind}_chi2\']=float(np.mean(((predict(fit.x)-ff)/ee)**2))\n            out[f\'flare_{kind}_asymmetry\']=float(fit.x[2]/fit.x[1])\n            out[f\'flare_{kind}_success\']=float(fit.success)\n        out[\'flare_powerlaw_exp_chi2_ratio\']=out[\'flare_powerlaw_chi2\']/max(out[\'flare_exponential_chi2\'],1e-12)\n    return oid,out\n', 'src/mallorn/physics_features.py': '"""Independent per-object GP and thermal-shape features for local experiments.\n\nInspired by Avocado and published MALLORN approaches; no external feature\ntables/code are used. Monochromatic thermal fits are descriptive approximations.\n"""\nfrom __future__ import annotations\n\nfrom itertools import combinations\n\nimport numpy as np\nfrom scipy.integrate import quad\nfrom scipy.linalg import cho_factor, cho_solve\nfrom scipy.optimize import minimize_scalar\nfrom scipy.stats import skew, kurtosis\n\nfrom .data import BANDS\nfrom .features import WAVELENGTHS, dust_scale\n\nPHASES = (-20, -10, 0, 10, 15, 30, 60, 90)\n\n\ndef luminosity_distance_cm(z):\n    """Flat LCDM, H0=70 km/s/Mpc, Omega_m=.3, radiation neglected."""\n    if z <= 0:\n        return np.nan\n    integral = quad(lambda zz: 1 / np.sqrt(.3*(1+zz)**3+.7), 0, z)[0]\n    return (1+z)*299792.458/70*integral*3.085677581491367e24\n\n\ndef kernel(t1, w1, t2, w2, tau):\n    # Matern 3/2 over time and log wavelength; fixed wavelength scale .4.\n    distance = np.sqrt(((t1[:,None]-t2[None,:])/tau)**2 +\n                       ((w1[:,None]-w2[None,:])/.4)**2)\n    r = np.sqrt(3)*distance\n    return (1+r)*np.exp(-r)\n\n\ndef thermal_fit(flux, error, z, distance):\n    valid = np.isfinite(flux)&np.isfinite(error)&(flux>2*error)&(error>0)\n    if valid.sum()<3 or not np.isfinite(distance):\n        return np.full(4,np.nan)\n    # f_nu=(1+z)*pi*B_nu((1+z)nu_obs,T)*(R/D_L)^2.\n    nu = 2.99792458e18/np.array([WAVELENGTHS[b] for b in BANDS])[valid]*(1+z)\n    f, e = flux[valid]*1e-29, error[valid]*1e-29  # microJy -> cgs\n    temperatures = np.geomspace(3500,100000,96)\n    bb = 2*6.62607015e-27*nu[None,:]**3/(2.99792458e10**2)/np.expm1(\n        6.62607015e-27*nu[None,:]/(1.380649e-16*temperatures[:,None]))\n    shape = np.pi*bb*(1+z)\n    amplitude = np.sum(shape*f/e**2,axis=1)/np.sum(shape**2/e**2,axis=1)\n    chi = np.sum(((f-shape*amplitude[:,None])/e)**2,axis=1)\n    i = int(np.argmin(chi))\n    radius = distance*np.sqrt(amplitude[i])\n    luminosity = 4*np.pi*radius**2*5.670374419e-5*temperatures[i]**4\n    return np.array([np.log10(temperatures[i]),np.log10(radius),np.log10(luminosity),chi[i]/max(1,valid.sum()-2)])\n\n\ndef object_features(task):\n    object_id,z,ebv,time,flux,error,band = task\n    z=max(float(z),0.)\n    factors=np.array([dust_scale(ebv,b) for b in BANDS])\n    flux=flux*factors[band];error=error*factors[band]\n    snr=flux/error\n    reference=np.flatnonzero(np.isin(band,[1,2,3]))\n    if not len(reference):reference=np.arange(len(time))\n    peak=time[reference[np.argmax(snr[reference])]]\n    t=(time-peak)/(1+z)\n    wavelengths=np.log(np.array([WAVELENGTHS[b] for b in BANDS]))\n    distance=luminosity_distance_cm(z)\n    out={\'phys_log_distance\':float(np.log10(distance)) if distance>0 else np.nan}\n    for bi,b in enumerate(BANDS):\n        mask=band==bi; f=flux[mask];tt=t[mask];ee=error[mask]\n        if not len(f):continue\n        order=np.argsort(tt);f=f[order];tt=tt[order];ee=ee[order]\n        peak_flux=max(float(np.max(f)),1e-10)\n        for name,val in [(\'mean\',np.mean(f)),(\'q90\',np.quantile(f,.9)),(\'peak\',np.max(f))]:\n            out[f\'phys_{b}_logL_{name}\']=float(np.log10(4*np.pi*distance**2*val*1e-29/(1+z))) if val>0 and distance>0 else np.nan\n        out[f\'phys_{b}_qmax_q50\']=float(np.max(f)/max(abs(np.median(f)),np.median(ee)))\n        out[f\'phys_{b}_q90_q50\']=float(np.quantile(f,.9)/max(abs(np.median(f)),np.median(ee)))\n        out[f\'phys_{b}_excess_variance\']=float((np.var(f)-np.mean(ee**2))/peak_flux**2)\n        out[f\'phys_{b}_von_neumann\']=float(np.mean(np.diff(f)**2)/max(np.var(f),1e-15)) if len(f)>2 else np.nan\n        # Equal-time resampling within observing seasons, never across >60-day gaps.\n        values=[]\n        for segment in np.split(np.arange(len(tt)),np.flatnonzero(np.diff(tt)>60/(1+z))+1):\n            if len(segment)<2:continue\n            grid=np.linspace(tt[segment[0]],tt[segment[-1]],max(3,min(100,int(np.ptp(tt[segment])/3)+1)))\n            values.extend(np.interp(grid,tt[segment],f[segment]))\n        if len(values)>3 and np.std(values)>1e-12:\n            out[f\'phys_{b}_resampled_skew\']=float(skew(values))\n            out[f\'phys_{b}_resampled_kurtosis\']=float(kurtosis(values))\n        for left,right in [(-80,-30),(-30,-10),(-10,10),(10,30),(30,60),(60,120),(120,300)]:\n            m=(tt>=left)&(tt<right)\n            out[f\'phys_{b}_bin{left}_{right}_mean\']=float(np.mean(f[m])) if m.any() else np.nan\n            out[f\'phys_{b}_bin{left}_{right}_norm\']=float(np.mean(f[m])/peak_flux) if m.any() else np.nan\n    # Cap computation deterministically, retaining time coverage and strong detections.\n    order=np.argsort(t,kind=\'stable\')\n    if len(order)>160:\n        strong=np.argsort(snr)[-40:]\n        uniform=order[np.linspace(0,len(order)-1,120).astype(int)]\n        order=np.unique(np.r_[strong,uniform])\n    gt,gw=t[order],wavelengths[band[order]]\n    scale=max(float(np.quantile(np.abs(flux),.95)),float(np.median(error)),1e-8)\n    gy=flux[order]/scale\n    variance=(error[order]/scale)**2+1e-6\n    def fit(log_tau,return_fit=False):\n        tau=np.exp(log_tau)\n        cov=kernel(gt,gw,gt,gw,tau);cov.flat[::len(cov)+1]+=variance\n        try:\n            chol=cho_factor(cov,lower=True,check_finite=False)\n            alpha=cho_solve(chol,gy,check_finite=False)\n            loss=.5*float(gy@alpha)+float(np.log(np.diag(chol[0])).sum())\n        except np.linalg.LinAlgError:\n            return np.inf\n        return (tau,chol,alpha,loss) if return_fit else loss\n    opt=minimize_scalar(fit,bounds=(np.log(2),np.log(500)),method=\'bounded\',options={\'xatol\':.1,\'maxiter\':20})\n    tau,chol,alpha,loss=fit(opt.x,True)\n    out.update(gp_log_timescale=float(np.log10(tau)),gp_nll_per_obs=float(loss/len(gt)))\n    def predict(qt,qb,uncertainty=False):\n        cov=kernel(gt,gw,qt,wavelengths[qb],tau)\n        mean=cov.T@alpha*scale\n        if not uncertainty:return mean\n        var=np.maximum(1-np.sum(cov*cho_solve(chol,cov,check_finite=False),axis=0),1e-8)\n        return mean,np.sqrt(var)*scale\n    grid=np.linspace(float(t.min()),float(t.max()),min(256,max(40,int(np.ptp(t)/3)+1)))\n    qtime=np.tile(grid,6);qband=np.repeat(np.arange(6),len(grid))\n    smooth=predict(qtime,qband).reshape(6,-1)\n    # Peak selected using supported g/r/i grid locations, avoiding unconstrained gaps.\n    support=np.min(abs(grid[:,None]-t[None,:]),axis=1)<30/(1+z)\n    combined=np.maximum(smooth[1:4],0).sum(axis=0)\n    combined[~support]=-np.inf\n    peak2=float(grid[int(np.argmax(combined))])\n    out[\'gp_peak_shift\']=peak2\n    phase_grid=grid-peak2\n    for bi,b in enumerate(BANDS):\n        s=smooth[bi];maximum=float(np.max(s));normal=max(maximum,1e-10)\n        out[f\'gp_{b}_peak\']=maximum\n        out[f\'gp_{b}_peak_phase\']=float(phase_grid[np.argmax(s)])\n        out[f\'gp_{b}_mean_norm\']=float(np.mean(s)/normal)\n        out[f\'gp_{b}_std_norm\']=float(np.std(s)/normal)\n        out[f\'gp_{b}_skew\']=float(skew(s)) if np.std(s)>1e-12 else np.nan\n        out[f\'gp_{b}_logLpeak\']=float(np.log10(4*np.pi*distance**2*maximum*1e-29/(1+z))) if maximum>0 and distance>0 else np.nan\n        for frac in [.2,.5,.8]:\n            above=phase_grid[s>frac*normal]\n            out[f\'gp_{b}_width{frac}\']=float(np.ptp(above)) if len(above)>1 else np.nan\n        # Is most of the integrated positive flux concentrated near a single peak?\n        near=abs(phase_grid)<60\n        out[f\'gp_{b}_concentration\']=float(np.maximum(s[near],0).sum()/max(np.maximum(s,0).sum(),1e-10))\n    qt=np.tile(np.array(PHASES)+peak2,6);qb=np.repeat(np.arange(6),len(PHASES))\n    means,errors=predict(qt,qb,True);means=means.reshape(6,-1);errors=errors.reshape(6,-1)\n    # Inflate very small conditional uncertainty by a representative measurement error.\n    errors=np.sqrt(errors**2+np.median(error)**2)\n    for bi,b in enumerate(BANDS):\n        for pi,phase in enumerate(PHASES):\n            out[f\'gp_{b}_p{phase}_norm\']=float(means[bi,pi]/max(np.max(smooth[bi]),1e-10))\n            out[f\'gp_{b}_p{phase}_snr\']=float(means[bi,pi]/errors[bi,pi])\n    for a,b in combinations(range(6),2):\n        colors=[]\n        for pi,phase in enumerate(PHASES):\n            reliable=(means[[a,b],pi]>2*errors[[a,b],pi]).all()\n            color=float(-2.5*np.log10(means[a,pi]/means[b,pi])) if reliable else np.nan\n            out[f\'gp_color_{BANDS[a]}{BANDS[b]}_p{phase}\']=color;colors.append(color)\n        out[f\'gp_color_{BANDS[a]}{BANDS[b]}_evolution\']=colors[5]-colors[2]\n    thermal=[]\n    for pi,phase in enumerate(PHASES):\n        pars=thermal_fit(means[:,pi],errors[:,pi],z,distance);thermal.append(pars)\n        for name,value in zip([\'logT\',\'logR\',\'logL\',\'chi2\'],pars):out[f\'bb_{name}_p{phase}\']=float(value)\n    for idx,name in enumerate([\'logT\',\'logR\',\'logL\']):\n        out[f\'bb_{name}_evolution10_30\']=float(thermal[5][idx]-thermal[3][idx])\n        out[f\'bb_{name}_evolution0_60\']=float(thermal[6][idx]-thermal[2][idx])\n    return object_id,out\n', 'src/mallorn/reporting.py': '"""Local evidence export; synthetic runs are never competition scores."""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom sklearn.metrics import precision_recall_curve\n\n\ndef write_json(path: Path, value):\n    path.write_text(json.dumps(value,indent=2,allow_nan=False)+"\\n",encoding="utf-8")\n\n\ndef report(output: Path, metrics: dict, oof):\n    figures=output/"figures"\n    figures.mkdir(parents=True,exist_ok=True)\n    synthetic=metrics["data_kind"]=="synthetic_fixture"\n    title="SYNTHETIC SOFTWARE TEST — NOT COMPETITION PERFORMANCE" if synthetic else "Local nested validation — not a Kaggle leaderboard score"\n    models=list(metrics["candidate_outer_metrics"])\n    values=[metrics["candidate_outer_metrics"][n]["f1"] for n in models]\n    with plt.rc_context({"font.size":10,"axes.spines.top":False,"axes.spines.right":False}):\n        fig,axes=plt.subplots(1,2,figsize=(12,4.5))\n        axes[0].barh(models,values,color="#177e89")\n        axes[0].set(xlim=(0,1),xlabel="F1 on outer held-out objects",title="Fixed candidates; inner-selected thresholds")\n        p,r,_=precision_recall_curve(oof.target,oof.selected_probability)\n        axes[1].plot(r,p,color="#6d4596",label="Nested selected workflow")\n        axes[1].axhline(oof.target.mean(),linestyle="--",color="gray",label="Positive prevalence")\n        axes[1].set(xlim=(0,1),ylim=(0,1.02),xlabel="Recall",ylabel="Precision",title="Cross-validated probability ranking")\n        axes[1].legend(fontsize=8)\n        fig.suptitle(title,fontsize=11)\n        fig.tight_layout()\n        fig.savefig(figures/"validation.svg",metadata={"Date":None})\n        plt.close(fig)\n    lines=["# Run report","",f"**{title}**","",\n           f"Primary nested F1: **{metrics[\'primary\'][\'f1\']:.4f}**; precision {metrics[\'primary\'][\'precision\']:.4f}; recall {metrics[\'primary\'][\'recall\']:.4f}.","",\n           "| Candidate | Outer F1 | Precision | Recall | Average precision |","| --- | ---: | ---: | ---: | ---: |"]\n    for name,m in metrics["candidate_outer_metrics"].items():\n        lines.append(f"| {name} | {m[\'f1\']:.4f} | {m[\'precision\']:.4f} | {m[\'recall\']:.4f} | {m[\'average_precision\']:.4f} |")\n    lines += ["","![Validation results](figures/validation.svg)","",\n              "Model and threshold selection are nested within each outer fold. Final deployment tuning uses all OOF labels; its F1 is not an unbiased performance estimate.","",\n              f"Grouping mode: `{metrics[\'grouping\']}`. Without real source-template groups, related simulations may cross folds; object-only CV does not resolve that risk.","",\n              "The train/test redshift measurement difference and repeated exploration of local results can reduce generalization. This report provides no claim of beating the private leaderboard.","",\n              "Exact scores: [metrics.json](metrics.json). Predictions: [oof.csv](oof.csv). IDs and fitting boundaries: [fold_audit.json](fold_audit.json)."]\n    (output/"RESULTS.md").write_text("\\n".join(lines)+"\\n",encoding="utf-8")\n\n\ndef bootstrap_interval(y, decisions, groups=None, seed=42, repeats=1000):\n    """Conditional bootstrap of fixed OOF decisions; does not refit models."""\n    rng=np.random.default_rng(seed)\n    y,decisions=np.asarray(y),np.asarray(decisions)\n    units=np.arange(len(y)) if groups is None else np.asarray(groups)\n    unique=np.unique(units)\n    blocks={unit:np.flatnonzero(units==unit) for unit in unique}\n    values=[]\n    for _ in range(repeats):\n        idx=np.concatenate([blocks[u] for u in rng.choice(unique,len(unique),replace=True)])\n        tp=np.sum((y[idx]==1)&(decisions[idx]==1))\n        denom=np.sum(y[idx])+np.sum(decisions[idx])\n        values.append(2*tp/denom if denom else 0.)\n    return {"ci95_low":float(np.quantile(values,.025)),"ci95_high":float(np.quantile(values,.975)),\n            "repeats":repeats,"unit":"source_group" if groups is not None else "object",\n            "scope":"Conditional on fixed OOF decisions; ignores model refitting and protocol selection uncertainty"}\n', 'src/mallorn/selected_ensemble.py': '"""Recovered cycle-5 method: training-only selection and three-seed averaging.\n\nReconstructed from the recorded implementation after a workspace rollback.\nThe original fitted cycle-5 checkpoints are not included.\n"""\nimport numpy as np\nfrom lightgbm import LGBMClassifier\n\nPARAMS = dict(n_estimators=900, learning_rate=.025, num_leaves=9,\n              min_child_samples=30, reg_lambda=10., colsample_bytree=.8,\n              subsample=.85, subsample_freq=1, deterministic=True,\n              force_col_wise=True, verbosity=-1)\nFORBIDDEN = {\'target\', \'SpecType\', \'Z_err\', \'object_id\'}\n\n\ndef labels_weights(meta, kind):\n    y = meta.target.to_numpy()\n    types = meta.SpecType.astype(str)\n    if kind == \'fine5\':\n        labels = np.select([y == 1, types.eq(\'AGN\'), types.eq(\'SN IIn\'), types.str.startswith(\'SLSN\')],\n                           [4, 0, 2, 3], default=1)\n        return labels, {0: 1, 1: 1, 2: 1, 3: 1, 4: 3}, 4\n    if kind == \'aux\':\n        return np.where(y == 1, 2, np.where(types.eq(\'AGN\'), 0, 1)), {0: 1, 1: 1, 2: 3}, 2\n    raise ValueError(\'Unknown auxiliary target\')\n\n\ndef select_columns(x, gain, limit=125, correlation=.95):\n    if FORBIDDEN & set(x.columns):\n        raise ValueError(\'Forbidden feature column\')\n    gain = np.asarray(gain, dtype=float)\n    if gain.shape != (x.shape[1],) or not np.isfinite(gain).all() or (gain < 0).any():\n        raise ValueError(\'Invalid importance vector\')\n    available = x.columns[x.nunique(dropna=True) > 1]\n    if not len(available):\n        raise ValueError(\'No variable fitting-fold features\')\n    values = x[available].fillna(x[available].median()).to_numpy(dtype=float)\n    centered = values-values.mean(axis=0)\n    unit = centered/np.maximum(np.linalg.norm(centered, axis=0), 1e-100)\n    positions = {name: i for i, name in enumerate(available)}\n    importance = dict(zip(x.columns, gain))\n    selected, indices = [], []\n    for name in sorted(available, key=lambda c: (-importance[c], c)):\n        i = positions[name]\n        if indices and np.any(np.abs(unit[:, indices].T @ unit[:, i]) >= correlation):\n            continue\n        selected.append(name)\n        indices.append(i)\n        if len(selected) >= limit:\n            break\n    return selected\n\n\ndef fit(x, meta, kind, seed, threads):\n    if FORBIDDEN & set(x.columns) or not x.index.equals(meta.index):\n        raise ValueError(\'Invalid training features or row alignment\')\n    labels, weights, positive = labels_weights(meta, kind)\n    gains = []\n    for offset in (10000, 20000):\n        selector = LGBMClassifier(**dict(PARAMS, n_estimators=600,\n            random_state=seed+offset, n_jobs=threads, class_weight=weights, importance_type=\'gain\'))\n        gain = selector.fit(x, labels).feature_importances_.astype(float)\n        gains.append(gain/max(gain.sum(), 1e-100))\n    columns = select_columns(x, np.mean(gains, axis=0))\n    models = [LGBMClassifier(**dict(PARAMS, random_state=seed+offset,\n        n_jobs=threads, class_weight=weights)).fit(x[columns], labels) for offset in (0, 1000, 2000)]\n    return {\'models\': models, \'selected_columns\': columns, \'positive\': positive}\n\n\ndef predict(fitted, x):\n    return np.mean([model.predict_proba(x[fitted[\'selected_columns\']])[:,\n        list(model.classes_).index(fitted[\'positive\'])] for model in fitted[\'models\']], axis=0)\n', 'scripts/audit_improvement.py': '"""Freeze a development-selected recipe, then run its one-time reserved audit.\n\nSeparate commands make the freeze reviewable before outcome inspection. A\nstarted audit can resume only with the identical recipe and implementation.\n"""\nimport argparse\nimport json\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nfrom improve import fit_candidate, predict_candidate\nfrom mallorn.data import sha256\nfrom mallorn.modeling import score\nfrom mallorn.reporting import bootstrap_interval, write_json\n\nROOT = Path(__file__).resolve().parents[1]\nOUT = ROOT / \'artifacts/improvement\'\nFEATURES = {\n    \'cached_features\': \'artifacts/official_001/features.csv\',\n    \'enhanced_features\': \'artifacts/improvement/enhanced_train.csv\',\n    \'physics_only\': \'artifacts/improvement/physics_train.csv\',\n    \'physics_advanced\': \'artifacts/improvement/physics_train.csv\',\n    \'complete_advanced\': \'artifacts/improvement/complete_train.csv\',\n    \'complete_selected\': \'artifacts/improvement/complete_train.csv\',\n    \'physical_focused\': \'artifacts/improvement/physical_view_train.csv\',\n}\n\n\ndef now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef implementation_hashes():\n    return {str(path.relative_to(ROOT)): sha256(path) for path in [\n        ROOT / \'scripts/improve.py\', Path(__file__).resolve(),\n        ROOT / \'src/mallorn/modeling.py\', ROOT / \'src/mallorn/reporting.py\']}\n\n\ndef freeze(comparison_path, destination):\n    if (OUT / \'audit/audit_started.json\').exists():\n        raise ValueError(\'The audit has started; no further recipe selection is allowed\')\n    comparison = json.loads(comparison_path.read_text())\n    chosen = comparison[\'results\'][0]\n    components = []\n    for member in chosen[\'members\']:\n        label, candidate = member.split(\'/\')\n        config = comparison[\'sources\'][label][\'config\']\n        path = FEATURES[label]\n        if sha256(ROOT / path) != config[\'features_sha256\']:\n            raise ValueError(\'Feature data changed after the screening run\')\n        components.append({\'name\': member, \'features\': path,\n                           \'features_sha256\': config[\'features_sha256\'],\n                           \'spec\': config[\'candidates\'][candidate], \'seed\': 42})\n    recipe = {\n        \'frozen_at\': now(), \'selection_rule\': \'Highest development F1; AP tie-break\',\n        \'comparison_sha256\': sha256(comparison_path), \'development_score\': chosen,\n        \'threshold\': chosen[\'threshold\'], \'components\': components,\n        \'ensemble\': \'Equal arithmetic mean of component probabilities\',\n        \'training\': \'Each component refitted on all development objects, seed 42\',\n        \'split_sha256\': sha256(OUT / \'split.json\'),\n        \'metadata_sha256\': sha256(ROOT / \'data/mallorn/train_log.csv\'),\n        \'implementation_sha256\': implementation_hashes(),\n        \'limitations\': [\n            \'Development selection scores are optimistic after model and threshold search.\',\n            \'The audit was reserved for this cycle but all labels were used by the old baseline.\',\n            \'No genuine source-template groups are available.\',\n            \'Audit F1 cannot establish equivalence to the Kaggle private leaderboard.\'],\n    }\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with destination.open(\'x\') as handle:\n        json.dump(recipe, handle, indent=2)\n        handle.write(\'\\n\')\n    print(\'FROZEN\', destination, sha256(destination))\n\n\ndef audit(recipe_path):\n    recipe = json.loads(recipe_path.read_text())\n    if recipe[\'implementation_sha256\'] != implementation_hashes():\n        raise ValueError(\'Implementation changed after the recipe freeze\')\n    if recipe[\'split_sha256\'] != sha256(OUT / \'split.json\'):\n        raise ValueError(\'Reserved split changed\')\n    if recipe[\'metadata_sha256\'] != sha256(ROOT / \'data/mallorn/train_log.csv\'):\n        raise ValueError(\'Training metadata changed\')\n    folder = OUT / \'audit\'\n    folder.mkdir(exist_ok=True)\n    if (folder / \'metrics.json\').exists():\n        raise ValueError(\'Audit already completed; read the recorded result without re-tuning\')\n    receipt = folder / \'audit_started.json\'\n    if receipt.exists():\n        if json.loads(receipt.read_text())[\'recipe_sha256\'] != sha256(recipe_path):\n            raise ValueError(\'A different recipe has already started the audit\')\n    else:\n        with receipt.open(\'x\') as handle:\n            json.dump({\'started_at\': now(), \'recipe_sha256\': sha256(recipe_path)}, handle, indent=2)\n    split = json.loads((OUT / \'split.json\').read_text())\n    dev, held = split[\'development\'], split[\'audit\']\n    if set(dev) & set(held):\n        raise ValueError(\'Development/audit overlap\')\n    meta = pd.read_csv(ROOT / \'data/mallorn/train_log.csv\').set_index(\'object_id\')\n    probabilities = []\n    for i, component in enumerate(recipe[\'components\']):\n        path = ROOT / component[\'features\']\n        if sha256(path) != component[\'features_sha256\']:\n            raise ValueError(\'Frozen feature bytes changed\')\n        x = pd.read_csv(path, index_col=\'object_id\')\n        if any(c in x for c in (\'target\', \'SpecType\', \'Z_err\')):\n            raise ValueError(\'Forbidden predictor column\')\n        saved = folder / f\'component_{i}.joblib\'\n        if saved.exists():\n            fitted = joblib.load(saved)\n        else:\n            fitted = fit_candidate(x.loc[dev], meta.loc[dev], component[\'spec\'], component[\'seed\'])\n            joblib.dump(fitted, saved, compress=3)\n        p = predict_candidate(fitted, x.loc[held])\n        np.save(folder / f\'component_{i}_audit.npy\', p)\n        probabilities.append(p)\n        print(\'FITTED\', component[\'name\'], flush=True)\n    p = np.mean(probabilities, axis=0)\n    decisions = p >= recipe[\'threshold\']\n    # First access to reserved outcomes occurs only after every prediction is fixed.\n    y = meta.loc[held, \'target\'].to_numpy()\n    result = score(y, p, decisions)\n    interval = bootstrap_interval(y, decisions, seed=20261005, repeats=5000)\n    interval[\'scope\'] = \'Conditional on fixed audit decisions; no refitting or source-group uncertainty\'\n    metrics = {\'completed_at\': now(), \'recipe_sha256\': sha256(recipe_path),\n               \'threshold\': recipe[\'threshold\'], \'audit\': result,\n               \'conditional_f1_interval\': interval, \'limitations\': recipe[\'limitations\']}\n    pd.DataFrame({\'object_id\': held, \'target\': y, \'probability\': p,\n                  \'prediction\': decisions.astype(int)}).to_csv(folder / \'predictions.csv\', index=False)\n    write_json(folder / \'metrics.json\', metrics)\n    print(json.dumps(metrics, indent=2), flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest=\'command\', required=True)\n    f = commands.add_parser(\'freeze\')\n    f.add_argument(\'--comparison\', type=Path, required=True)\n    f.add_argument(\'--output\', type=Path, default=OUT / \'frozen_recipe.json\')\n    a = commands.add_parser(\'audit\')\n    a.add_argument(\'--recipe\', type=Path, default=OUT / \'frozen_recipe.json\')\n    args = parser.parse_args()\n    if args.command == \'freeze\':\n        freeze(args.comparison, args.output)\n    else:\n        audit(args.recipe)\n', 'scripts/build_morphology_features.py': "import argparse,json,time\nfrom concurrent.futures import ProcessPoolExecutor\nfrom pathlib import Path\nimport pandas as pd\nimport numpy as np\nfrom mallorn.data import BANDS,load_dataset,sha256\nfrom mallorn.morphology_features import object_features\n\nif __name__=='__main__':\n    parser=argparse.ArgumentParser();parser.add_argument('--mode',choices=['train','test'],default='train');args=parser.parse_args()\n    root=Path(__file__).resolve().parents[1];dest=root/'artifacts/improvement'\n    train,test,photo,_=load_dataset(root/'data/mallorn');log=train if args.mode=='train' else test\n    meta=log.set_index('object_id')[['Z','EBV']];photo=photo[photo.object_id.isin(meta.index)]\n    tasks=[]\n    for oid,g in photo.groupby('object_id',sort=True):\n        row=meta.loc[oid]\n        tasks.append((oid,float(row.Z),float(row.EBV),g['Time (MJD)'].to_numpy(),g.Flux.to_numpy(),g.Flux_err.to_numpy(),g.Filter.map({b:i for i,b in enumerate(BANDS)}).to_numpy()))\n    rows={};start=time.monotonic()\n    with ProcessPoolExecutor(max_workers=3) as pool:\n        for n,(oid,f) in enumerate(pool.map(object_features,tasks,chunksize=10),1):\n            rows[oid]=f\n            if n%300==0:print(args.mode,n,len(tasks),'seconds',round(time.monotonic()-start,1),flush=True)\n    f=pd.DataFrame.from_dict(rows,orient='index').rename_axis('object_id').sort_index(axis=1).replace([np.inf,-np.inf],np.nan)\n    base=pd.read_csv(dest/f'enhanced_{args.mode}.csv',index_col='object_id')\n    base.join(f).to_csv(dest/f'complete_{args.mode}.csv',float_format='%.15g')\n    (dest/f'morphology_{args.mode}_manifest.json').write_text(json.dumps({'objects':len(f),'new_features':len(f.columns),'elapsed_seconds':time.monotonic()-start,'source_sha256':sha256(root/'src/mallorn/morphology_features.py'),'features_sha256':sha256(dest/f'complete_{args.mode}.csv')},indent=2)+'\\n')\n    print('DONE',f.shape,flush=True)\n", 'scripts/build_physical_view.py': '"""Compose a focused physical feature view without reading any class labels."""\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom mallorn.data import sha256\nfrom mallorn.reporting import write_json\n\n\ndef transform(frame):\n    out = frame.loc[:, frame.columns.str.startswith(\n        (\'phys_\', \'gp_\', \'bb_\', \'morph_\', \'flare_\'))].copy()\n    # Physical derivatives differ from fractional (logarithmic) derivatives:\n    # e.g. two equal fractional radius changes can imply different expansion speeds.\n    for parameter, scale in [(\'T\', 1e4), (\'R\', 1e15), (\'L\', 1e44)]:\n        for left, right in [(0, 10), (10, 30), (0, 60)]:\n            a = np.power(10., frame[f\'bb_log{parameter}_p{left}\']) / scale\n            b = np.power(10., frame[f\'bb_log{parameter}_p{right}\']) / scale\n            out[f\'derived_{parameter}_slope{left}_{right}\'] = (b - a) / (right - left)\n        out[f\'derived_{parameter}_slope_change\'] = (\n            out[f\'derived_{parameter}_slope10_30\'] - out[f\'derived_{parameter}_slope0_10\'])\n    for band in \'ugrizy\':\n        out[f\'derived_{band}_peak_tail_ratio\'] = (\n            frame[f\'gp_{band}_p30_norm\'] / frame[f\'gp_{band}_p90_norm\'].where(\n                frame[f\'gp_{band}_p90_norm\'] > .02))\n        out[f\'derived_{band}_width_ratio\'] = (\n            frame[f\'gp_{band}_width0.2\'] / frame[f\'gp_{band}_width0.8\'].where(\n                frame[f\'gp_{band}_width0.8\'] > 0))\n    return out.replace([np.inf, -np.inf], np.nan)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--mode\', choices=[\'train\', \'test\'], default=\'train\')\n    args = parser.parse_args()\n    root = Path(__file__).resolve().parents[1]\n    dest = root / \'artifacts/improvement\'\n    source = dest / f\'complete_{args.mode}.csv\'\n    result = dest / f\'physical_view_{args.mode}.csv\'\n    frame = transform(pd.read_csv(source, index_col=\'object_id\'))\n    frame.to_csv(result, float_format=\'%.15g\')\n    write_json(dest / f\'physical_view_{args.mode}_manifest.json\', {\n        \'source_sha256\': sha256(source), \'code_sha256\': sha256(Path(__file__)),\n        \'result_sha256\': sha256(result), \'objects\': len(frame), \'features\': frame.shape[1]})\n    print(frame.shape)\n', 'scripts/build_physics_features.py': '"""Build per-object physical features without reading target labels into extraction."""\nimport argparse\nimport json\nimport time\nfrom concurrent.futures import ProcessPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom mallorn.data import BANDS,load_dataset,sha256\nfrom mallorn.physics_features import object_features\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--mode\',choices=[\'train\',\'test\'],default=\'train\')\n    parser.add_argument(\'--workers\',type=int,default=3)\n    args=parser.parse_args()\n    root=Path(__file__).resolve().parents[1]\n    destination=root/\'artifacts/improvement\';destination.mkdir(parents=True,exist_ok=True)\n    train,test,photo,manifest=load_dataset(root/\'data/mallorn\')\n    log=train if args.mode==\'train\' else test\n    meta=log.set_index(\'object_id\')[[\'Z\',\'EBV\']]\n    photo=photo[photo.object_id.isin(meta.index)]\n    bandmap={b:i for i,b in enumerate(BANDS)}\n    tasks=[]\n    for oid,g in photo.groupby(\'object_id\',sort=True):\n        row=meta.loc[oid]\n        tasks.append((oid,float(row.Z),float(row.EBV),g[\'Time (MJD)\'].to_numpy(),\n                      g.Flux.to_numpy(),g.Flux_err.to_numpy(),g.Filter.map(bandmap).to_numpy()))\n    rows={};start=time.monotonic()\n    with ProcessPoolExecutor(max_workers=args.workers) as pool:\n        for n,(oid,features) in enumerate(pool.map(object_features,tasks,chunksize=10),1):\n            rows[oid]=features\n            if n%100==0: print(args.mode,n,len(tasks),\'seconds\',round(time.monotonic()-start,1),flush=True)\n    new=pd.DataFrame.from_dict(rows,orient=\'index\').sort_index(axis=1).rename_axis(\'object_id\')\n    new=new.replace([np.inf,-np.inf],np.nan)\n    base=pd.read_csv(root/\'artifacts/official_001/features.csv\',index_col=\'object_id\').loc[new.index]\n    for name,frame in [(\'physics\',new),(\'enhanced\',base.join(new))]:\n        frame.to_csv(destination/f\'{name}_{args.mode}.csv\',float_format=\'%.15g\')\n    (destination/f\'physics_{args.mode}_manifest.json\').write_text(json.dumps({\n        \'objects\':len(new),\'new_features\':len(new.columns),\'source_sha256\':sha256(root/\'src/mallorn/physics_features.py\'),\n        \'features_sha256\':sha256(destination/f\'enhanced_{args.mode}.csv\'),\n        \'elapsed_seconds\':time.monotonic()-start,\'label_free\':True,\n        \'approximations\':\'Matern32 time/log-wavelength GP; fixed wavelength scale; max160 observations; monochromatic blackbody; flat LCDM H0=70 Om=.3\'},indent=2)+\'\\n\')\n    print(\'DONE\',new.shape,flush=True)\n', 'scripts/compare_development.py': '"""Compare fixed, equal-weight ensembles on development OOF only.\n\nThis is candidate selection, not an independent evaluation. Every tried pair\nis retained; no audit predictions or audit labels are read.\n"""\nimport argparse\nimport itertools\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom mallorn.data import sha256\nfrom mallorn.modeling import best_threshold, score\nfrom mallorn.reporting import write_json\n\nROOT = Path(__file__).resolve().parents[1]\nOUT = ROOT / \'artifacts/improvement\'\n\n\ndef compare(labels, name):\n    split = json.loads((OUT / \'split.json\').read_text())\n    predictions, sources, target = {}, {}, None\n    for label in labels:\n        folder = OUT / label\n        config = json.loads((folder / \'config.json\').read_text())\n        if config[\'fold_seed\'] != 20261004 or config[\'folds\'] != 5:\n            raise ValueError(\'Incompatible OOF folds\')\n        frame = pd.read_csv(folder / \'oof.csv\').set_index(\'object_id\')\n        if set(frame.index) != set(split[\'development\']):\n            raise ValueError(\'Only the exact development objects are allowed\')\n        frame = frame.loc[split[\'development\']]\n        y = frame.target.to_numpy()\n        if target is None:\n            target = y\n        np.testing.assert_array_equal(y, target)\n        sources[label] = {\'oof_sha256\': sha256(folder / \'oof.csv\'), \'config\': config}\n        for column in config[\'candidates\']:\n            p = frame[column].to_numpy()\n            if not (np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()):\n                raise ValueError(\'Invalid probabilities\')\n            predictions[f\'{label}/{column}\'] = p\n    results = []\n    # Deliberately use equal weights, with no continuous weight optimizer.\n    recipes = [[key] for key in predictions]\n    recipes += list(itertools.combinations(predictions, 2))\n    recipes += [[key for key in predictions if key.startswith(label + \'/\')]\n                for label in labels]\n    recipes += [list(predictions)]\n    for members in recipes:\n        p = np.mean([predictions[key] for key in members], axis=0)\n        threshold, _ = best_threshold(target, p)\n        results.append({\'members\': list(members), \'threshold\': threshold,\n                        **score(target, p, p >= threshold)})\n    results.sort(key=lambda row: (row[\'f1\'], row[\'average_precision\']), reverse=True)\n    folder = OUT / name\n    folder.mkdir(exist_ok=True)\n    write_json(folder / \'comparison.json\', {\n        \'scope\': \'Development OOF candidate and threshold selection; optimistic tuning scores\',\n        \'sources\': sources, \'candidates_evaluated\': len(results), \'results\': results})\n    print(json.dumps({\'candidates_evaluated\': len(results), \'top\': results[:8]}, indent=2))\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--labels\', nargs=\'+\', required=True)\n    parser.add_argument(\'--name\', required=True)\n    args = parser.parse_args()\n    compare(args.labels, args.name)\n', 'scripts/cycle2.py': '"""Fixed second-cycle candidates with repeated, resumable object-level CV."""\nimport argparse\nimport json\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedKFold\n\nfrom improve import candidate_specs, fit_candidate, predict_candidate\nfrom mallorn.data import sha256\nfrom mallorn.modeling import best_threshold, score\nfrom mallorn.reporting import write_json\n\nROOT = Path(__file__).resolve().parents[1]\nOUT = ROOT / \'artifacts/cycle2\'\nCOMPONENTS = {\n    \'physics_aux\': (\'physics\', candidate_specs(\'advanced\')[\'cat_aux3_d5\']),\n    \'selected_binary\': (\'complete\', candidate_specs(\'selected\')[\'cat_d4_w3_top125\']),\n    \'selected_aux\': (\'complete\', candidate_specs(\'selected\')[\'cat_aux3_d5_top125\']),\n    \'hierarchical\': (\'physical_view\', candidate_specs(\'physical\')[\'cat_hierarchy_d4\']),\n}\nRECIPES = {\n    \'reference\': [\'physics_aux\', \'selected_binary\'],\n    \'selected_class\': [\'selected_binary\', \'selected_aux\'],\n    \'diverse\': list(COMPONENTS),\n}\n\n\ndef inputs(name):\n    view, spec = COMPONENTS[name]\n    path = ROOT / f\'artifacts/improvement/{view}_train.csv\'\n    meta = pd.read_csv(ROOT / \'data/mallorn/train_log.csv\').set_index(\'object_id\')\n    x = pd.read_csv(path, index_col=\'object_id\').loc[meta.index]\n    assert not set(x.columns) & {\'target\', \'SpecType\', \'Z_err\', \'object_id\'}\n    return x, meta, spec, path\n\n\ndef screen(name):\n    OUT.mkdir(exist_ok=True)\n    folder = OUT / name\n    folder.mkdir(exist_ok=True)\n    x, meta, spec, path = inputs(name)\n    config = {\'name\': name, \'spec\': spec, \'features_sha256\': sha256(path),\n              \'metadata_sha256\': sha256(ROOT / \'data/mallorn/train_log.csv\'),\n              \'fit_code_sha256\': sha256(ROOT / \'scripts/improve.py\'),\n              \'split_seeds\': [20261014, 20261015], \'folds\': 5, \'objects\': len(x)}\n    saved_config = folder / \'config.json\'\n    if saved_config.exists() and json.loads(saved_config.read_text()) != config:\n        raise ValueError(\'Configuration changed; use a separate experiment\')\n    write_json(saved_config, config)\n    y = meta.target.to_numpy()\n    for repeat, seed in enumerate(config[\'split_seeds\']):\n        file = folder / f\'oof_repeat{repeat}.npy\'\n        p = np.load(file) if file.exists() else np.full(len(y), np.nan)\n        start = time.monotonic()\n        for fold, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=seed).split(x, y)):\n            if np.isfinite(p[b]).all():\n                continue\n            fitted = fit_candidate(x.iloc[a], meta.iloc[a], spec, 42 + 100 * repeat + fold)\n            p[b] = predict_candidate(fitted, x.iloc[b])\n            write_json(folder / f\'repeat{repeat}_fold{fold}.json\', {\n                \'fitting_ids\': x.index[a].tolist(), \'validation_ids\': x.index[b].tolist(),\n                \'columns\': fitted[\'columns\'], \'seed\': 42 + 100 * repeat + fold})\n            temp = folder / \'checkpoint.npy\'\n            np.save(temp, p)\n            temp.replace(file)\n            print(name, \'repeat\', repeat, \'fold\', fold, \'seconds\', round(time.monotonic()-start, 1), flush=True)\n        assert np.isfinite(p).all()\n        threshold, _ = best_threshold(y, p)\n        print(name, \'repeat\', repeat, json.dumps(score(y, p, p >= threshold)), flush=True)\n\n\ndef stable_threshold(y, p):\n    rng = np.random.default_rng(20261016)\n    classes = [np.flatnonzero(y == k) for k in [0, 1]]\n    thresholds = []\n    for _ in range(500):\n        idx = np.concatenate([rng.choice(a, len(a), replace=True) for a in classes])\n        thresholds.append(best_threshold(y[idx], p[idx])[0])\n    return float(np.median(thresholds)), np.quantile(thresholds, [.1, .9]).tolist()\n\n\ndef compare():\n    _, meta, _, _ = inputs(\'physics_aux\')\n    y = meta.target.to_numpy()\n    ids = set(meta.index)\n    verification = {}\n    for name in COMPONENTS:\n        config = json.loads((OUT/name/\'config.json\').read_text())\n        assert config[\'metadata_sha256\'] == sha256(ROOT/\'data/mallorn/train_log.csv\')\n        assert config[\'fit_code_sha256\'] == sha256(ROOT/\'scripts/improve.py\')\n        for repeat in range(2):\n            seen = []\n            for fold in range(5):\n                audit = json.loads((OUT/name/f\'repeat{repeat}_fold{fold}.json\').read_text())\n                a, b = set(audit[\'fitting_ids\']), set(audit[\'validation_ids\'])\n                assert not a & b and a | b == ids\n                assert not set(audit[\'columns\']) & {\'target\', \'SpecType\', \'Z_err\', \'object_id\'}\n                seen.extend(audit[\'validation_ids\'])\n            assert len(seen) == len(ids) and set(seen) == ids\n        verification[name] = {\'repetitions\': 2, \'folds_per_repetition\': 5,\n                              \'validation_predictions_per_object\': 2,\n                              \'fitting_validation_overlap\': 0}\n    write_json(OUT/\'verification.json\', verification)\n    predictions = {name: np.stack([np.load(OUT/name/f\'oof_repeat{r}.npy\') for r in range(2)])\n                   for name in COMPONENTS}\n    assert all(p.shape == (2, len(y)) and np.isfinite(p).all() for p in predictions.values())\n    result = {}\n    for name, members in RECIPES.items():\n        repeated = np.mean([predictions[m] for m in members], axis=0)\n        p = repeated.mean(axis=0)\n        threshold, interval = stable_threshold(y, p)\n        tuned_threshold, tuned_f1 = best_threshold(y, p)\n        result[name] = {\'members\': members, \'threshold\': threshold,\n                        \'bootstrap_threshold_10_90\': interval,\n                        \'combined\': score(y, p, p >= threshold),\n                        \'repeat_scores_at_fixed_threshold\': [score(y, q, q >= threshold) for q in repeated],\n                        \'unsmoothed_tuning_threshold\': tuned_threshold,\n                        \'unsmoothed_tuning_f1\': tuned_f1}\n    reference = result[\'reference\'][\'combined\']\n    old_threshold = json.loads((ROOT/\'artifacts/improvement/frozen_recipe.json\').read_text())[\'threshold\']\n    old_recipe_repeated = np.mean([predictions[m] for m in RECIPES[\'reference\']], axis=0)\n    reference_at_old_threshold = {\n        \'threshold\': old_threshold,\n        \'combined\': score(y, old_recipe_repeated.mean(axis=0),\n                          old_recipe_repeated.mean(axis=0) >= old_threshold),\n        \'individual_repetitions\': [score(y, p, p >= old_threshold) for p in old_recipe_repeated],\n    }\n    eligible = [k for k in result if result[k][\'combined\'][\'f1\'] >= reference[\'f1\'] + .01\n                and result[k][\'combined\'][\'average_precision\'] >= reference[\'average_precision\'] - .005]\n    chosen = max(eligible, key=lambda k: result[k][\'combined\'][\'f1\']) if eligible else \'reference\'\n    record = {\'scope\': \'Repeated local CV selection; all historical labels reused, no pristine audit\',\n              \'recipes\': result, \'chosen\': chosen, \'deployment_seeds\': [42, 142],\n              \'reference_at_previously_submitted_threshold\': reference_at_old_threshold,\n              \'code_sha256\': sha256(Path(__file__)),\n              \'component_configs\': {k: json.loads((OUT/k/\'config.json\').read_text()) for k in COMPONENTS}}\n    path = OUT / \'frozen_comparison.json\'\n    if path.exists() and json.loads(path.read_text()) != record:\n        raise ValueError(\'Frozen comparison already exists with different values\')\n    write_json(path, record)\n    receipt = OUT/\'freeze_receipt.json\'\n    if not receipt.exists():\n        write_json(receipt, {\'frozen_at\': datetime.now(timezone.utc).isoformat(),\n                            \'frozen_comparison_sha256\': sha256(path),\n                            \'kaggle_score_observed_for_this_recipe\': False})\n    print(json.dumps(record, indent=2))\n\n\ndef deploy():\n    file = OUT / \'frozen_comparison.json\'\n    record = json.loads(file.read_text())\n    if record[\'code_sha256\'] != sha256(Path(__file__)):\n        raise ValueError(\'Implementation changed after freeze\')\n    recipe = record[\'recipes\'][record[\'chosen\']]\n    template = pd.read_csv(ROOT / \'data/mallorn/sample_submission.csv\')\n    assert template.columns.tolist() == [\'object_id\', \'prediction\']\n    folder = OUT / \'deployment\'\n    folder.mkdir(exist_ok=True)\n    all_p = []\n    for name in recipe[\'members\']:\n        x, meta, spec, train_path = inputs(name)\n        if sha256(train_path) != record[\'component_configs\'][name][\'features_sha256\']:\n            raise ValueError(\'Feature data changed\')\n        test_path = Path(str(train_path).replace(\'_train.csv\', \'_test.csv\'))\n        test_x = pd.read_csv(test_path, index_col=\'object_id\').loc[template.object_id]\n        assert x.columns.equals(test_x.columns)\n        for seed in record[\'deployment_seeds\']:\n            path = folder / f\'{name}_{seed}.joblib\'\n            if path.exists():\n                fitted = joblib.load(path)\n            else:\n                fitted = fit_candidate(x, meta, spec, seed)\n                joblib.dump(fitted, path, compress=3)\n            p = predict_candidate(fitted, test_x)\n            np.save(folder / f\'{name}_{seed}.npy\', p)\n            all_p.append(p)\n            print(\'FITTED\', name, seed, flush=True)\n    p = np.mean(all_p, axis=0)\n    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()\n    result = template.copy()\n    result.prediction = (p >= recipe[\'threshold\']).astype(int)\n    assert len(result) == 7135 and result.object_id.is_unique\n    result.to_csv(folder / \'submission.csv\', index=False)\n    np.save(folder / \'probabilities.npy\', p)\n    write_json(folder / \'manifest.json\', {\'recipe\': record[\'chosen\'],\n        \'frozen_comparison_sha256\': sha256(file), \'submission_sha256\': sha256(folder/\'submission.csv\'),\n        \'threshold\': recipe[\'threshold\'], \'positive_predictions\': int(result.prediction.sum()),\n        \'objects\': len(result), \'submitted_to_kaggle\': False})\n    print((folder / \'manifest.json\').read_text())\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'action\', choices=[\'screen\', \'compare\', \'deploy\'])\n    parser.add_argument(\'--component\', choices=list(COMPONENTS))\n    args = parser.parse_args()\n    if args.action == \'screen\':\n        if args.component is None:\n            parser.error(\'--component is required for screening\')\n        screen(args.component)\n    elif args.action == \'compare\':\n        compare()\n    else:\n        deploy()\n', 'scripts/cycle3.py': '"""Fixed LightGBM diversification experiment, driven by resume_research.py."""\nfrom __future__ import annotations\n\nfrom datetime import datetime, timezone\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport time\n\nimport joblib\nfrom lightgbm import LGBMClassifier\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedKFold\n\nfrom mallorn.checkpointing import atomic_json, atomic_model, freeze_json\nfrom mallorn.data import sha256\nfrom mallorn.evaluation import best_threshold, score\n\nROOT = Path(__file__).resolve().parents[1]\nOUT = ROOT / \'artifacts/cycle3\'\nCOMPONENTS = {\n    \'lgb_physics_aux\': (\'physics\', \'aux\'),\n    \'lgb_physical_hierarchy\': (\'physical_view\', \'hierarchy\'),\n    \'lgb_complete_binary\': (\'complete\', \'binary\'),\n}\nPARAMS = dict(n_estimators=900, learning_rate=.025, num_leaves=9,\n              min_child_samples=30, reg_lambda=10., colsample_bytree=.8,\n              subsample=.85, subsample_freq=1, deterministic=True,\n              force_col_wise=True, verbosity=-1)\nFORBIDDEN = {\'target\', \'SpecType\', \'Z_err\', \'object_id\'}\n\n\ndef now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef initialize():\n    files = [\'data/mallorn/train_log.csv\', \'scripts/cycle3.py\',\n             \'src/mallorn/checkpointing.py\', \'src/mallorn/evaluation.py\',\n             \'docs/THIRD_CYCLE_PROTOCOL.md\', \'artifacts/cycle2/frozen_comparison.json\']\n    files += [f\'artifacts/improvement/{view}_train.csv\' for view, _ in COMPONENTS.values()]\n    reference = json.loads((ROOT / files[5]).read_text())\n    receipt = json.loads((ROOT/\'artifacts/cycle2/freeze_receipt.json\').read_text())\n    assert sha256(ROOT/files[5]) == receipt[\'frozen_comparison_sha256\']\n    assert reference[\'chosen\'] == \'diverse\'\n    for name in reference[\'recipes\'][\'diverse\'][\'members\']:\n        for repeat in range(2):\n            files.append(f\'artifacts/cycle2/{name}/oof_repeat{repeat}.npy\')\n    manifest = {\'schema_version\': 1, \'components\': COMPONENTS, \'params\': PARAMS,\n                \'split_seeds\': [20261014, 20261015], \'folds\': 5,\n                \'threads\': int(os.environ.get(\'MALLORN_THREADS\', 4)),\n                \'packages\': {k: importlib.metadata.version(k) for k in\n                             [\'lightgbm\', \'numpy\', \'pandas\', \'scikit-learn\', \'joblib\']},\n                \'files\': {f: sha256(ROOT/f) for f in files},\n                \'publishing_allowed\': False}\n    # Normalize tuples to their on-disk JSON representation before comparison.\n    manifest = json.loads(json.dumps(manifest))\n    freeze_json(OUT/\'manifest.json\', manifest)\n    if not (OUT/\'start_receipt.json\').exists():\n        atomic_json(OUT/\'start_receipt.json\', {\'started_at\': now(),\n                    \'manifest_sha256\': sha256(OUT/\'manifest.json\')})\n    return manifest\n\n\ndef inputs(name):\n    view, _ = COMPONENTS[name]\n    meta = pd.read_csv(ROOT/\'data/mallorn/train_log.csv\').set_index(\'object_id\')\n    x = pd.read_csv(ROOT/f\'artifacts/improvement/{view}_train.csv\', index_col=\'object_id\')\n    assert x.index.is_unique and set(x.index) == set(meta.index)\n    x = x.loc[meta.index]\n    assert not FORBIDDEN & set(x.columns) and not np.isinf(x.to_numpy()).any()\n    return x, meta\n\n\ndef fit(x, meta, kind, seed, threads):\n    def model(weights):\n        return LGBMClassifier(**PARAMS, random_state=seed, n_jobs=threads, class_weight=weights)\n    y = meta.target.to_numpy()\n    if kind == \'aux\':\n        labels = np.where(meta.SpecType.eq(\'AGN\'), 0, np.where(y == 1, 2, 1))\n        return {\'model\': model({0: 1, 1: 1, 2: 3}).fit(x, labels), \'positive\': 2}\n    if kind == \'hierarchy\':\n        transient = ~meta.SpecType.eq(\'AGN\').to_numpy()\n        return {\'gate\': model({0: 1, 1: 1}).fit(x, transient.astype(int)),\n                \'model\': model({0: 1, 1: 3}).fit(x.loc[transient], y[transient]), \'positive\': 1}\n    return {\'model\': model({0: 1, 1: 3}).fit(x, y), \'positive\': 1}\n\n\ndef predict(fitted, x):\n    model = fitted[\'model\']\n    p = model.predict_proba(x)[:, list(model.classes_).index(fitted[\'positive\'])]\n    if \'gate\' in fitted:\n        gate = fitted[\'gate\']\n        p *= gate.predict_proba(x)[:, list(gate.classes_).index(1)]\n    return p\n\n\ndef fold_path(name, repeat, fold):\n    return OUT/name/f\'repeat{repeat}_fold{fold}.joblib\'\n\n\ndef verify_fold(record, signature, ids, a, b):\n    if record[\'manifest_sha256\'] != signature:\n        raise ValueError(\'Checkpoint belongs to a different experiment\')\n    if record[\'fitting_ids\'] != ids[a].tolist() or record[\'validation_ids\'] != ids[b].tolist():\n        raise ValueError(\'Checkpoint fold assignment differs\')\n    if set(record[\'fitting_ids\']) & set(record[\'validation_ids\']):\n        raise ValueError(\'Fitting/validation overlap\')\n    p = np.asarray(record[\'probabilities\'])\n    if p.shape != (len(b),) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():\n        raise ValueError(\'Invalid checkpoint probabilities\')\n    return p\n\n\ndef run_folds(manifest, deadline=float(\'inf\'), max_new=None):\n    signature = sha256(OUT/\'manifest.json\')\n    completed = 0\n    for name, (_, kind) in COMPONENTS.items():\n        x, meta = inputs(name)\n        y = meta.target.to_numpy()\n        for repeat, split_seed in enumerate(manifest[\'split_seeds\']):\n            cv = StratifiedKFold(5, shuffle=True, random_state=split_seed)\n            for fold, (a, b) in enumerate(cv.split(x, y)):\n                path = fold_path(name, repeat, fold)\n                if path.exists():\n                    record = joblib.load(path)\n                    verify_fold(record, signature, x.index, a, b)\n                    continue\n                if time.monotonic() >= deadline or (max_new is not None and completed >= max_new):\n                    return False\n                start = time.monotonic()\n                seed = 42 + 100*repeat + fold\n                atomic_json(OUT/\'progress.json\', {\'state\': \'training\', \'component\': name,\n                    \'repeat\': repeat, \'fold\': fold, \'started_at\': now()})\n                fitted = fit(x.iloc[a], meta.iloc[a], kind, seed, manifest[\'threads\'])\n                p = predict(fitted, x.iloc[b])\n                record = {\'manifest_sha256\': signature, \'fitting_ids\': x.index[a].tolist(),\n                          \'validation_ids\': x.index[b].tolist(), \'probabilities\': p,\n                          \'model\': fitted, \'columns\': x.columns.tolist(), \'seed\': seed,\n                          \'seconds\': time.monotonic()-start, \'completed_at\': now()}\n                verify_fold(record, signature, x.index, a, b)\n                atomic_model(path, record)\n                completed += 1\n                print(f\'{name} repeat={repeat} fold={fold} committed ({record["seconds"]:.1f}s)\', flush=True)\n    return True\n\n\ndef stable_threshold(y, p):\n    rng = np.random.default_rng(20261016)\n    classes = [np.flatnonzero(y == k) for k in [0, 1]]\n    values = []\n    for _ in range(500):\n        idx = np.concatenate([rng.choice(a, len(a), replace=True) for a in classes])\n        values.append(best_threshold(y[idx], p[idx])[0])\n    return float(np.median(values))\n\n\ndef paired_interval(y, decisions, reference):\n    rng = np.random.default_rng(20261017)\n    classes = [np.flatnonzero(y == k) for k in [0, 1]]\n    def f1(a, b):\n        return 2*np.sum(a & b)/max(int(a.sum()+b.sum()), 1)\n    differences = []\n    for _ in range(2000):\n        idx = np.concatenate([rng.choice(a, len(a), replace=True) for a in classes])\n        differences.append(f1(y[idx], decisions[idx])-f1(y[idx], reference[idx]))\n    return np.quantile(differences, [.025, .975]).tolist()\n\n\ndef compare(manifest):\n    _, meta = inputs(\'lgb_physics_aux\')\n    y = meta.target.to_numpy()\n    signature = sha256(OUT/\'manifest.json\')\n    predictions = {}\n    replay = {}\n    for name in COMPONENTS:\n        x, _ = inputs(name)\n        repeated = np.full((2, len(y)), np.nan)\n        for repeat, seed in enumerate(manifest[\'split_seeds\']):\n            seen = np.zeros(len(y), dtype=int)\n            for fold, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=seed).split(x, y)):\n                record = joblib.load(fold_path(name, repeat, fold))\n                p = verify_fold(record, signature, x.index, a, b)\n                assert record[\'columns\'] == x.columns.tolist()\n                replay_p = predict(record[\'model\'], x.iloc[b])\n                delta = float(np.max(np.abs(replay_p-p)))\n                if delta > 1e-12:\n                    raise ValueError(\'Saved model replay mismatch\')\n                replay[f\'{name}/{repeat}/{fold}\'] = delta\n                repeated[repeat, b] = p\n                seen[b] += 1\n            assert np.all(seen == 1)\n        predictions[name] = repeated\n    old = json.loads((ROOT/\'artifacts/cycle2/frozen_comparison.json\').read_text())\n    members = old[\'recipes\'][\'diverse\'][\'members\']\n    ref = np.mean([np.stack([np.load(ROOT/f\'artifacts/cycle2/{m}/oof_repeat{r}.npy\')\n                            for r in range(2)]) for m in members], axis=0)\n    assert ref.shape == (2, len(y)) and np.isfinite(ref).all()\n    recipes = {\'reference\': ref}\n    for name, p in predictions.items():\n        recipes[\'blend_\'+name] = .5*(ref+p)\n    lgb_mean = np.mean(list(predictions.values()), axis=0)\n    recipes[\'lgb_ensemble\'] = lgb_mean\n    recipes[\'blend_lgb_ensemble\'] = .5*(ref+lgb_mean)\n    results = {}\n    for name, repeated in recipes.items():\n        p = repeated.mean(axis=0)\n        threshold = stable_threshold(y, p)\n        results[name] = {\'threshold\': threshold, \'combined\': score(y, p, p >= threshold),\n            \'repeat_scores\': [score(y, q, q >= threshold) for q in repeated]}\n    reference = results[\'reference\']\n    assert np.isclose(reference[\'threshold\'], old[\'recipes\'][\'diverse\'][\'threshold\'])\n    assert np.isclose(reference[\'combined\'][\'f1\'], old[\'recipes\'][\'diverse\'][\'combined\'][\'f1\'])\n    eligible = [name for name, r in results.items() if name != \'reference\'\n                and r[\'combined\'][\'f1\'] >= reference[\'combined\'][\'f1\']+.005\n                and r[\'combined\'][\'average_precision\'] >= reference[\'combined\'][\'average_precision\']\n                and all(a[\'f1\'] >= b[\'f1\']-.005 for a, b in zip(r[\'repeat_scores\'], reference[\'repeat_scores\']))]\n    chosen = max(eligible, key=lambda name: results[name][\'combined\'][\'f1\']) if eligible else \'reference\'\n    ref_decisions = ref.mean(axis=0) >= reference[\'threshold\']\n    for name in results:\n        decisions = recipes[name].mean(axis=0) >= results[name][\'threshold\']\n        results[name][\'paired_f1_difference_95pct_descriptive\'] = paired_interval(y, decisions, ref_decisions)\n    component_scores = {}\n    for name, repeated in predictions.items():\n        p = repeated.mean(axis=0)\n        threshold = stable_threshold(y, p)\n        component_scores[name] = score(y, p, p >= threshold)\n    record = {\'manifest_sha256\': signature, \'recipes\': results, \'chosen\': chosen,\n              \'component_diagnostics\': component_scores, \'model_replay_max_errors\': replay,\n              \'scope\': \'Repeated development selection; not an independent test\',\n              \'github_upload\': False, \'kaggle_upload\': False}\n    freeze_json(OUT/\'comparison.json\', record)\n    atomic_json(OUT/\'progress.json\', {\'state\': \'complete\', \'completed_at\': now(), \'chosen\': chosen})\n    print(json.dumps({k: v[\'combined\'] for k,v in results.items()}, indent=2), flush=True)\n    return record\n', 'scripts/cycle4.py': '"""Confusing-negative experiments; reuse the tested checkpoint engine unchanged."""\nfrom __future__ import annotations\n\nimport importlib.util\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom lightgbm import LGBMClassifier\n\nfrom mallorn.checkpointing import atomic_json, freeze_json\nfrom mallorn.data import sha256\nimport cycle3\n\nROOT = Path(__file__).resolve().parents[1]\nOUT = ROOT/\'artifacts/cycle4\'\nCOMPONENTS = {\'fine5\': (\'complete\', \'fine5\'),\n              \'hard_negative_binary\': (\'complete\', \'hard_negative_binary\')}\n\n\ndef fit(x, meta, kind, seed, threads):\n    y = meta.target.to_numpy()\n    types = meta.SpecType.astype(str)\n    params = dict(cycle3.PARAMS, random_state=seed, n_jobs=threads)\n    if kind == \'fine5\':\n        # Target takes precedence; all mappings use only this fitting fold.\n        labels = np.select([y == 1, types.eq(\'AGN\'), types.eq(\'SN IIn\'), types.str.startswith(\'SLSN\')],\n                           [4, 0, 2, 3], default=1)\n        model = LGBMClassifier(**params, class_weight={0: 1, 1: 1, 2: 1, 3: 1, 4: 3})\n        return {\'model\': model.fit(x, labels), \'positive\': 4}\n    weights = np.where(y == 1, 3., np.where(types.eq(\'SN IIn\') | types.str.startswith(\'SLSN\'), 2., 1.))\n    model = LGBMClassifier(**params)\n    return {\'model\': model.fit(x, y, sample_weight=weights), \'positive\': 1}\n\n\ndef engine():\n    """An isolated module instance keeps archived cycle-3 globals unchanged.\n\n    The same validated fold/checkpoint/comparison implementation is configured\n    for a new experiment. Its source hash is included in this cycle\'s manifest.\n    """\n    spec = importlib.util.spec_from_file_location(\'mallorn_cycle4_engine\', ROOT/\'scripts/cycle3.py\')\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    module.OUT = OUT\n    module.COMPONENTS = COMPONENTS\n    module.fit = fit\n    # Both candidates (including the engine\'s initial metadata read) use the\n    # complete feature view; preserve cycle-3 input validation and row ordering.\n    module.inputs = lambda name: cycle3.inputs(\'lgb_complete_binary\')\n    return module\n\n\ndef initialize(base_manifest):\n    # Do not call the cycle-3 initializer with different globals: it owns a\n    # distinct frozen protocol. Include every dependency of the reused engine.\n    manifest = json.loads(json.dumps(base_manifest))\n    manifest[\'experiment\'] = \'cycle4\'\n    manifest[\'components\'] = {k: list(v) for k,v in COMPONENTS.items()}\n    manifest[\'files\'][\'scripts/cycle4.py\'] = sha256(Path(__file__))\n    manifest[\'files\'][\'docs/FOURTH_CYCLE_PROTOCOL.md\'] = sha256(ROOT/\'docs/FOURTH_CYCLE_PROTOCOL.md\')\n    manifest[\'files\'][\'artifacts/cycle3/comparison.json\'] = sha256(ROOT/\'artifacts/cycle3/comparison.json\')\n    freeze_json(OUT/\'manifest.json\', manifest)\n    if not (OUT/\'start_receipt.json\').exists():\n        atomic_json(OUT/\'start_receipt.json\', {\'started_at\': cycle3.now(),\n                    \'manifest_sha256\': sha256(OUT/\'manifest.json\')})\n    return manifest\n', 'scripts/cycle5_recovered.py': '"""Rerun the recovered cycle-5 recipe in a NEW namespace, preserving evidence."""\nimport argparse\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport time\n\nimport cycle3\nfrom mallorn.selected_ensemble import fit, predict\nfrom mallorn.checkpointing import atomic_json, exclusive_lock, freeze_json\nfrom mallorn.data import sha256\n\nROOT = Path(__file__).resolve().parents[1]\nOUT = ROOT/\'artifacts/cycle5_recovered\'\nCOMPONENTS = {\'selected_complete_fine5\': (\'complete\',\'fine5\'),\n              \'selected_physical_aux\': (\'physical_view\',\'aux\')}\n\n\ndef inputs(name):\n    view = COMPONENTS[name][0] if name in COMPONENTS else \'complete\'\n    return cycle3.inputs(\'lgb_complete_binary\' if view == \'complete\' else \'lgb_physical_hierarchy\')\n\n\ndef run(seconds):\n    with exclusive_lock(ROOT/\'artifacts/research.lock\'):\n        manifest = json.loads(json.dumps(cycle3.initialize()))\n        manifest.update(experiment=\'cycle5_recovered\',components=COMPONENTS,\n                        feature_limit=125,correlation_limit=.95,\n                        selector_offsets=[10000,20000],final_offsets=[0,1000,2000])\n        for path in [\'scripts/cycle5_recovered.py\',\'src/mallorn/selected_ensemble.py\',\n                     \'docs/FIFTH_CYCLE_RECOVERY.md\']:\n            manifest[\'files\'][path] = sha256(ROOT/path)\n        manifest = json.loads(json.dumps(manifest))\n        freeze_json(OUT/\'manifest.json\',manifest)\n        if not (OUT/\'start_receipt.json\').exists():\n            atomic_json(OUT/\'start_receipt.json\',{\'started_at\':cycle3.now(),\'recovered_source\':True})\n        spec = importlib.util.spec_from_file_location(\'recovered_engine\',ROOT/\'scripts/cycle3.py\')\n        worker = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(worker)\n        worker.OUT, worker.COMPONENTS = OUT, COMPONENTS\n        worker.inputs, worker.fit, worker.predict = inputs, fit, predict\n        complete = worker.run_folds(manifest,time.monotonic()+seconds)\n        if complete and not (OUT/\'comparison.json\').exists():\n            worker.compare(manifest)\n        print(json.dumps({\'complete\':complete,\'output\':str(OUT)}))\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--max-seconds\',type=int,default=1500)\n    args = parser.parse_args()\n    if args.max_seconds < 1: parser.error(\'Time budget must be positive\')\n    run(args.max_seconds)\n', 'scripts/deploy_improvement.py': '"""Refit an audited recipe and prepare local predictions; never upload them."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nfrom audit_improvement import implementation_hashes\nfrom improve import fit_candidate, predict_candidate\nfrom mallorn.data import sha256\nfrom mallorn.reporting import write_json\n\nROOT = Path(__file__).resolve().parents[1]\nOUT = ROOT / \'artifacts/improvement\'\n\n\ndef deploy(recipe_path, predict_only):\n    recipe = json.loads(recipe_path.read_text())\n    audit = json.loads((OUT / \'audit/metrics.json\').read_text())\n    if audit[\'recipe_sha256\'] != sha256(recipe_path):\n        raise ValueError(\'Audit does not belong to this frozen recipe\')\n    if implementation_hashes() != recipe[\'implementation_sha256\']:\n        raise ValueError(\'Fitting implementation changed after the audit\')\n    folder = OUT / \'deployment\'\n    folder.mkdir(exist_ok=True)\n    receipt = {\'recipe_sha256\': sha256(recipe_path),\n               \'training_scope\': \'All 3043 labeled competition objects after the one-time audit\'}\n    saved_receipt = folder / \'recipe.json\'\n    if saved_receipt.exists() and json.loads(saved_receipt.read_text()) != receipt:\n        raise ValueError(\'Deployment directory already belongs to a different recipe\')\n    write_json(saved_receipt, receipt)\n    meta = pd.read_csv(ROOT / \'data/mallorn/train_log.csv\').set_index(\'object_id\')\n    test = pd.read_csv(ROOT / \'data/mallorn/test_log.csv\').set_index(\'object_id\')\n    template = pd.read_csv(ROOT / \'data/mallorn/sample_submission.csv\')\n    if template.columns.tolist() != [\'object_id\', \'prediction\']:\n        raise ValueError(\'Unexpected official submission schema\')\n    probabilities, artifacts = [], []\n    for i, component in enumerate(recipe[\'components\']):\n        source = ROOT / component[\'features\']\n        if sha256(source) != component[\'features_sha256\']:\n            raise ValueError(\'Training features changed after the recipe freeze\')\n        test_source = Path(str(source).replace(\'_train.csv\', \'_test.csv\'))\n        train_x = pd.read_csv(source, index_col=\'object_id\').loc[meta.index]\n        test_x = pd.read_csv(test_source, index_col=\'object_id\').loc[test.index]\n        if not train_x.columns.equals(test_x.columns):\n            raise ValueError(\'Train and inference feature schemas differ\')\n        model_path = folder / f\'component_{i}.joblib\'\n        if model_path.exists():\n            fitted = joblib.load(model_path)\n        elif predict_only:\n            raise ValueError(\'Predict-only requires all fitted component artifacts\')\n        else:\n            fitted = fit_candidate(train_x, meta, component[\'spec\'], component[\'seed\'])\n            joblib.dump(fitted, model_path, compress=3)\n        probabilities.append(predict_candidate(fitted, test_x))\n        artifacts.append({\'component\': component[\'name\'], \'model_sha256\': sha256(model_path),\n                          \'test_features_sha256\': sha256(test_source),\n                          \'features_used\': len(fitted[\'columns\'])})\n        print(\'READY\', component[\'name\'], flush=True)\n    p = np.mean(probabilities, axis=0)\n    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()\n    predictions = (p >= recipe[\'threshold\']).astype(int)\n    submission = pd.DataFrame({\'object_id\': test.index, \'prediction\': predictions})\n    submission = submission.set_index(\'object_id\').loc[template.object_id].reset_index()\n    assert len(submission) == 7135 and submission.object_id.is_unique\n    path = folder / \'submission.csv\'\n    if predict_only:\n        pd.testing.assert_frame_equal(pd.read_csv(path), submission)\n        np.testing.assert_allclose(np.load(folder / \'probabilities.npy\'), p, rtol=0, atol=1e-12)\n    else:\n        submission.to_csv(path, index=False)\n        np.save(folder / \'probabilities.npy\', p)\n    result = {**receipt, \'threshold\': recipe[\'threshold\'], \'test_objects\': len(submission),\n              \'predicted_positive\': int(predictions.sum()), \'components\': artifacts,\n              \'submission_sha256\': sha256(path), \'prediction_replay_checked\': predict_only,\n              \'published_to_github\': False, \'submitted_to_kaggle\': False,\n              \'notice\': \'Local submission candidate; no organizer test-set score is available\'}\n    write_json(folder / \'manifest.json\', result)\n    print(json.dumps(result, indent=2))\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--recipe\', type=Path, default=OUT / \'frozen_recipe.json\')\n    parser.add_argument(\'--predict-only\', action=\'store_true\')\n    args = parser.parse_args()\n    deploy(args.recipe, args.predict_only)\n', 'scripts/improve.py': '"""Local development screening. The reserved audit outcomes are never scored here."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostClassifier\nfrom lightgbm import LGBMClassifier\nfrom sklearn.model_selection import StratifiedKFold, train_test_split\n\nfrom mallorn.data import sha256\nfrom mallorn.modeling import best_threshold, score\nfrom mallorn.reporting import write_json\n\nROOT = Path(__file__).resolve().parents[1]\nOUT = ROOT / "artifacts/improvement"\n\n\ndef split_ids(train):\n    OUT.mkdir(parents=True, exist_ok=True)\n    path = OUT / "split.json"\n    if path.exists():\n        split = json.loads(path.read_text())\n    else:\n        dev, audit = train_test_split(train.object_id.to_numpy(), test_size=.25,\n                                     stratify=train.target, random_state=20261003)\n        split = {"development": dev.tolist(), "audit": audit.tolist(), "seed": 20261003,\n                 "note": "Reserved for this cycle; all labels were previously used in baseline evaluation"}\n        write_json(path, split)\n    assert not set(split["development"]) & set(split["audit"])\n    assert set(split["development"]) | set(split["audit"]) == set(train.object_id)\n    return split\n\n\ndef estimator(spec, seed):\n    params = dict(spec)\n    kind = params.pop("kind")\n    params.pop("view", None)\n    params.pop("auxiliary", None)\n    params.pop("select_features", None)\n    params.pop("hierarchical", None)\n    if kind == "cat":\n        return CatBoostClassifier(**params, random_seed=seed, thread_count=int(os.environ.get(\'MALLORN_THREADS\',6)),\n                                  allow_writing_files=False, verbose=False)\n    return LGBMClassifier(**params, random_state=seed, n_jobs=int(os.environ.get(\'MALLORN_THREADS\',6)), verbosity=-1,\n                          deterministic=True, force_col_wise=True)\n\n\ndef candidate_specs(batch):\n    cat = dict(kind="cat", iterations=1000, learning_rate=.035, l2_leaf_reg=6, loss_function="Logloss")\n    lgb = dict(kind="lgb", n_estimators=700, learning_rate=.025, reg_lambda=5,\n               colsample_bytree=.85, min_child_samples=20)\n    specs = {\n        "cat_d5": dict(cat, depth=5),\n        "cat_d4_w3": dict(cat, depth=4, class_weights=[1, 3]),\n        "cat_d6": dict(cat, depth=6),\n        "lgb_l9_w3": dict(lgb, num_leaves=9, scale_pos_weight=3),\n        "lgb_l15": dict(lgb, num_leaves=15),\n    }\n    if batch == "compact":\n        specs = {k:v for k,v in specs.items() if k in ("cat_d5", "cat_d4_w3", "lgb_l9_w3")}\n    elif batch == "advanced":\n        specs = {\n            "cat_d4_long":dict(cat,depth=4,iterations=1800,l2_leaf_reg=8),\n            "cat_d5_w3":dict(cat,depth=5,iterations=1200,class_weights=[1,3]),\n            "cat_aux3_d5":dict(cat,depth=5,iterations=1200,loss_function=\'MultiClass\',\n                               class_weights=[1,1,3],auxiliary=True),\n            "lgb_l9_long":dict(lgb,num_leaves=9,n_estimators=1200),\n            "lgb_l15_w3":dict(lgb,num_leaves=15,scale_pos_weight=3),\n        }\n    elif batch == "selected":\n        specs = {\n            "cat_d4_w3_top125": dict(cat, depth=4, class_weights=[1,3], select_features=125),\n            "cat_d5_w3_top250": dict(cat, depth=5, class_weights=[1,3], select_features=250),\n            "cat_aux3_d5_top125": dict(cat, depth=5, iterations=1200,\n                                      loss_function=\'MultiClass\', class_weights=[1,1,3],\n                                      auxiliary=True, select_features=125),\n        }\n    elif batch == "physical":\n        specs = {\n            \'cat_d4_w3\': dict(cat, depth=4, class_weights=[1,3]),\n            \'cat_hierarchy_d4\': dict(cat, depth=4, class_weights=[1,3], hierarchical=True),\n            \'cat_aux6_d4\': dict(cat, depth=4, iterations=1200, loss_function=\'MultiClass\',\n                              class_weights=[1,1,1,1,1,3], auxiliary=\'six\'),\n        }\n    return specs\n\n\ndef training_columns(x, y, spec, seed):\n    """Rank and decorrelate columns using the fitting fold only."""\n    count = spec.get(\'select_features\')\n    if count is None:\n        return list(x.columns)\n    selector = estimator(dict(kind=\'cat\', iterations=500, depth=4,\n                             learning_rate=.04, loss_function=\'Logloss\',\n                             class_weights=[1,3], l2_leaf_reg=8), seed)\n    selector.fit(x, y)\n    order = np.argsort(-selector.feature_importances_, kind=\'stable\')\n    candidates = list(x.columns[order[:min(len(order), count * 3)]])\n    # Pairwise training-only correlation, including missing values as missing.\n    corr = x[candidates].corr().abs()\n    selected = []\n    for column in candidates:\n        if not selected or not (corr.loc[column, selected] > .95).any():\n            selected.append(column)\n        if len(selected) == count:\n            break\n    return selected\n\n\ndef fit_candidate(x, meta, spec, seed):\n    y = meta.target.to_numpy()\n    columns = training_columns(x, y, spec, seed)\n    x = x[columns]\n    model = estimator(spec, seed)\n    if spec.get(\'hierarchical\'):\n        transient = ~meta.SpecType.eq(\'AGN\').to_numpy()\n        gate = estimator(dict(spec, class_weights=[1,1]), seed)\n        gate.fit(x, transient.astype(int))\n        model.fit(x.loc[transient], y[transient])\n        return {\'model\': model, \'gate\': gate, \'columns\': columns, \'positive\': 1}\n    if spec.get(\'auxiliary\') == \'six\':\n        # AGN, Ia family, IIn, other type-II, other transient, TDE.\n        types = meta.SpecType.astype(str)\n        labels = np.select([types.eq(\'AGN\'), types.str.startswith(\'SN Ia\'),\n                            types.eq(\'SN IIn\'), types.str.startswith(\'SN II\'), y == 1],\n                           [0, 1, 2, 3, 5], default=4)\n        positive = 5\n    elif spec.get(\'auxiliary\'):\n        labels = np.where(meta.SpecType.eq(\'AGN\'), 0, np.where(y == 1, 2, 1))\n        positive = 2\n    else:\n        labels, positive = y, 1\n    model.fit(x, labels)\n    return {\'model\': model, \'columns\': columns, \'positive\': positive}\n\n\ndef predict_candidate(fitted, x):\n    x = x[fitted[\'columns\']]\n    model = fitted[\'model\']\n    p = model.predict_proba(x)[:, list(model.classes_).index(fitted[\'positive\'])]\n    if \'gate\' in fitted:\n        gate = fitted[\'gate\']\n        p *= gate.predict_proba(x)[:, list(gate.classes_).index(1)]\n    return p\n\n\ndef screen(features_path, label, batch):\n    train = pd.read_csv(ROOT / "data/mallorn/train_log.csv")\n    split = split_ids(train)\n    x = pd.read_csv(features_path, index_col="object_id").loc[split["development"]]\n    meta = train.set_index("object_id").loc[x.index]\n    y = meta.target.to_numpy()\n    folder = OUT / label\n    folder.mkdir(exist_ok=True)\n    config = {"features_sha256": sha256(features_path), "development_objects": len(x),\n              "feature_count": x.shape[1], "fold_seed": 20261004, "folds": 5,\n              "scope": "Development OOF candidate/threshold tuning, not an unbiased final score",\n              "candidates": candidate_specs(batch)}\n    config_path = folder / "config.json"\n    if config_path.exists() and json.loads(config_path.read_text()) != config:\n        raise ValueError("Existing experiment configuration differs")\n    write_json(config_path, config)\n    cv = list(StratifiedKFold(5, shuffle=True, random_state=20261004).split(x, y))\n    table = pd.DataFrame({"object_id": x.index, "target": y})\n    results = {}\n    for name, spec in config["candidates"].items():\n        saved = folder / f"{name}.npy"\n        start = time.monotonic()\n        if saved.exists():\n            p = np.load(saved)\n        else:\n            checkpoint = folder / f\'{name}_partial.npy\'\n            p = np.load(checkpoint) if checkpoint.exists() else np.full(len(y), np.nan)\n            for fold, (a,b) in enumerate(cv):\n                if np.isfinite(p[b]).all():\n                    continue\n                fitted = fit_candidate(x.iloc[a], meta.iloc[a], spec, 42+fold)\n                if spec.get(\'select_features\'):\n                    write_json(folder / f\'{name}_fold{fold}_columns.json\', fitted[\'columns\'])\n                p[b] = predict_candidate(fitted, x.iloc[b])\n                temporary = folder / f\'{name}_checkpoint.npy\'\n                np.save(temporary, p)\n                temporary.replace(checkpoint)\n                print(label, name, "fold", fold+1, "seconds", round(time.monotonic()-start,1), flush=True)\n            np.save(saved, p)\n            checkpoint.unlink(missing_ok=True)\n        threshold, _ = best_threshold(y,p)\n        results[name] = dict(score(y,p,p>=threshold), threshold=threshold,\n                             elapsed_seconds=round(time.monotonic()-start,2))\n        table[name] = p\n        write_json(folder / "scores.json", results)\n        print(label, name, json.dumps(results[name]), flush=True)\n    base_names = list(config["candidates"])\n    for i, left in enumerate(base_names):\n        for right in base_names[i+1:]:\n            name = f"blend_{left}_{right}"\n            p = .5*table[left].to_numpy()+.5*table[right].to_numpy()\n            threshold, _ = best_threshold(y,p)\n            results[name] = dict(score(y,p,p>=threshold), threshold=threshold)\n    table.to_csv(folder / "oof.csv", index=False, float_format="%.15g")\n    write_json(folder / "scores.json", results)\n    print("BEST",label,json.dumps(sorted(results.items(),key=lambda kv:kv[1][\'f1\'],reverse=True)[:5]),flush=True)\n\n\nif __name__ == "__main__":\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--features",type=Path,required=True)\n    parser.add_argument("--label",required=True)\n    parser.add_argument("--batch",choices=["full","compact","advanced","selected","physical"],default="full")\n    args=parser.parse_args()\n    screen(args.features,args.label,args.batch)\n', 'scripts/notebook_smoke.py': '"""Portable synthetic feature/model/replay check; no competition score claim."""\nimport argparse\nfrom pathlib import Path\nimport tempfile\nimport joblib\nfrom lightgbm import LGBMClassifier\nimport numpy as np\nimport pandas as pd\nfrom mallorn.fixtures import make_fixture\nfrom mallorn.data import load_dataset, validate_submission\nfrom mallorn.features import extract_features\nfrom mallorn.checkpointing import atomic_json, atomic_model\n\n\ndef run(output):\n    if output.exists() and any(output.iterdir()):\n        raise ValueError(\'Choose an empty output directory for this software check\')\n    output.mkdir(parents=True,exist_ok=True)\n    with tempfile.TemporaryDirectory() as temporary:\n        root=Path(temporary);make_fixture(root)\n        train,test,photo,_=load_dataset(root)\n        x=extract_features(pd.concat([train,test],ignore_index=True),photo)\n        assert not {\'target\',\'SpecType\',\'Z_err\'} & set(x.columns)\n        model=LGBMClassifier(n_estimators=12,num_leaves=5,min_child_samples=5,\n            n_jobs=1,random_state=42,verbosity=-1,deterministic=True,force_col_wise=True)\n        model.fit(x.loc[train.object_id],train.target)\n        p=model.predict_proba(x.loc[test.object_id])[:,1]\n        assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all()\n        atomic_model(output/\'synthetic_model.joblib\',model)\n        np.testing.assert_array_equal(p,joblib.load(output/\'synthetic_model.joblib\').predict_proba(x.loc[test.object_id])[:,1])\n        frame=pd.DataFrame({\'object_id\':test.object_id,\'prediction\':(p>=.5).astype(int)})\n        validate_submission(frame,test.object_id)\n        frame.to_csv(output/\'synthetic_submission.csv\',index=False)\n        atomic_json(output/\'software_check.json\',{\'data_kind\':\'synthetic_fixture\',\n            \'not_competition_performance\':True,\'training_objects\':len(train),\'test_objects\':len(test),\n            \'feature_count\':x.shape[1],\'model_replay_exact\':True,\'submission_schema_checked\':True})\n        print(\'Synthetic feature extraction, LightGBM fit, saved-model replay and schema check passed.\')\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    run(parser.parse_args().output)\n', 'scripts/refit_frozen.py': '"""Crash-safe full-data refit of the unchanged cycle-2 recipe; local output only."""\nfrom __future__ import annotations\n\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport tempfile\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nfrom mallorn.checkpointing import atomic_json, atomic_model, exclusive_lock, freeze_json\nfrom mallorn.data import sha256\n\nROOT = Path(__file__).resolve().parents[1]\nOUT = ROOT/\'artifacts/cycle2/deployment\'\n\n\ndef atomic_csv(path, frame):\n    fd, tmp = tempfile.mkstemp(prefix=path.name, suffix=\'.pending\', dir=path.parent)\n    try:\n        with os.fdopen(fd, \'w\') as handle:\n            frame.to_csv(handle, index=False)\n            handle.flush()\n            os.fsync(handle.fileno())\n        os.replace(tmp, path)\n    finally:\n        if os.path.exists(tmp):\n            os.unlink(tmp)\n\n\ndef run():\n    # Fail before importing native CatBoost in an incompatible Linux runtime.\n    if sys.platform.startswith(\'linux\'):\n        with open(\'/proc/self/statm\') as handle:\n            handle.read(100)\n    from cycle2 import inputs\n    from improve import fit_candidate, predict_candidate\n    frozen = ROOT/\'artifacts/cycle2/frozen_comparison.json\'\n    record = json.loads(frozen.read_text())\n    if sha256(ROOT/\'scripts/cycle2.py\') != record[\'code_sha256\']:\n        raise ValueError(\'Frozen cycle-2 implementation changed\')\n    recipe = record[\'recipes\'][record[\'chosen\']]\n    template = pd.read_csv(ROOT/\'data/mallorn/sample_submission.csv\')\n    assert template.columns.tolist() == [\'object_id\', \'prediction\']\n    assert len(template) == 7135 and template.object_id.is_unique\n    with exclusive_lock(OUT/\'refit.lock\'):\n        source_files = [\'scripts/refit_frozen.py\', \'src/mallorn/checkpointing.py\',\n                        \'data/mallorn/train_log.csv\', \'data/mallorn/sample_submission.csv\']\n        for name in recipe[\'members\']:\n            x, meta, spec, train_path = inputs(name)\n            config = record[\'component_configs\'][name]\n            assert sha256(train_path) == config[\'features_sha256\']\n            assert sha256(ROOT/\'data/mallorn/train_log.csv\') == config[\'metadata_sha256\']\n            assert sha256(ROOT/\'scripts/improve.py\') == config[\'fit_code_sha256\']\n            source_files += [str(train_path.relative_to(ROOT)), str(train_path.relative_to(ROOT)).replace(\'_train.csv\', \'_test.csv\')]\n        spec_record = {\'frozen_recipe_sha256\': sha256(frozen),\n                       \'files\': {f: sha256(ROOT/f) for f in source_files},\n                       \'packages\': {k: importlib.metadata.version(k) for k in\n                                    [\'catboost\', \'numpy\', \'pandas\', \'scikit-learn\', \'joblib\']},\n                       \'threads\': int(os.environ.get(\'MALLORN_THREADS\', 6)),\n                       \'threshold\': recipe[\'threshold\'], \'seeds\': record[\'deployment_seeds\']}\n        freeze_json(OUT/\'resumable_manifest.json\', spec_record)\n        signature = sha256(OUT/\'resumable_manifest.json\')\n        all_probabilities = []\n        for name in recipe[\'members\']:\n            x, meta, spec, train_path = inputs(name)\n            test_x = pd.read_csv(Path(str(train_path).replace(\'_train.csv\', \'_test.csv\')),\n                                 index_col=\'object_id\').loc[template.object_id]\n            assert x.columns.equals(test_x.columns)\n            for seed in record[\'deployment_seeds\']:\n                path = OUT/f\'checkpoints/{name}_{seed}.joblib\'\n                if path.exists():\n                    checkpoint = joblib.load(path)\n                    if checkpoint[\'manifest_sha256\'] != signature:\n                        raise ValueError(\'Refit checkpoint configuration differs\')\n                    fitted = checkpoint[\'model\']\n                    p = predict_candidate(fitted, test_x)\n                    if not np.allclose(p, checkpoint[\'probabilities\'], rtol=0, atol=1e-12):\n                        raise ValueError(\'Saved-model replay differs\')\n                else:\n                    fitted = fit_candidate(x, meta, spec, seed)\n                    p = predict_candidate(fitted, test_x)\n                    checkpoint = {\'manifest_sha256\': signature, \'model\': fitted, \'probabilities\': p}\n                    atomic_model(path, checkpoint)\n                assert p.shape == (len(template),) and np.isfinite(p).all()\n                assert ((p >= 0) & (p <= 1)).all()\n                all_probabilities.append(p)\n                print(\'CHECKPOINTED\', name, seed, flush=True)\n        p = np.mean(all_probabilities, axis=0)\n        result = template.copy()\n        result.prediction = (p >= recipe[\'threshold\']).astype(int)\n        atomic_csv(OUT/\'submission.csv\', result)\n        atomic_model(OUT/\'probabilities.joblib\', p)\n        atomic_json(OUT/\'manifest.json\', {\'recipe\': record[\'chosen\'],\n            \'frozen_comparison_sha256\': sha256(frozen),\n            \'resumable_manifest_sha256\': signature,\n            \'submission_sha256\': sha256(OUT/\'submission.csv\'),\n            \'threshold\': recipe[\'threshold\'], \'positive_predictions\': int(result.prediction.sum()),\n            \'objects\': len(result), \'models\': len(all_probabilities), \'submitted_to_kaggle\': False})\n        print(\'LOCAL_REFIT_COMPLETE; no upload performed\', flush=True)\n\n\nif __name__ == \'__main__\':\n    run()\n', 'scripts/report_research.py': '"""Render aggregate local experiment evidence without per-object private data."""\nimport json\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef render(cycle):\n    source = ROOT/f\'artifacts/cycle{cycle}/comparison.json\'\n    if not source.exists():\n        return\n    record = json.loads(source.read_text())\n    folder = ROOT/f\'reports/cycle{cycle}_20261004\'\n    folder.mkdir(exist_ok=True)\n    names = {\n        \'reference\': \'Retained cycle-2 reference\',\n        \'blend_lgb_physics_aux\': \'Reference + physics three-class LightGBM\',\n        \'blend_lgb_physical_hierarchy\': \'Reference + hierarchical LightGBM\',\n        \'blend_lgb_complete_binary\': \'Reference + complete binary LightGBM\',\n        \'blend_fine5\': \'Reference + five-class LightGBM\',\n        \'blend_hard_negative_binary\': \'Reference + weighted confusing negatives\',\n        \'lgb_ensemble\': \'New LightGBM ensemble\',\n        \'blend_lgb_ensemble\': \'Reference + new LightGBM ensemble\',\n    }\n    ref = record[\'recipes\'][\'reference\']\n    best = max(record[\'recipes\'], key=lambda k: record[\'recipes\'][k][\'combined\'][\'f1\'])\n    chosen = record[\'chosen\']\n    text = [f\'# Cycle {cycle} — local development results\', \'\',\n        f\'**Selected by the frozen rule: {names[chosen]}.** Highest pooled development F1: \'\n        f\'**{record["recipes"][best]["combined"]["f1"]:.6f}** ({names[best]}).\', \'\',\n        \'These scores use all 3,043 labeled objects (148 TDEs), two stratified five-fold \'\n        \'repetitions, and the mean of each object\\\'s held-out probabilities. Thresholds \'\n        \'are tuned using 500 stratified bootstraps. This is repeated development \'\n        \'selection on historically reused labels, not independent test evidence.\', \'\',\n        \'| Recipe | F1 | Precision | Recall | AP | Repeat 1 F1 | Repeat 2 F1 |\',\n        \'| --- | ---: | ---: | ---: | ---: | ---: | ---: |\']\n    for name, result in record[\'recipes\'].items():\n        m = result[\'combined\']\n        reps = result[\'repeat_scores\']\n        text.append(f\'| {names[name]} | {m["f1"]:.6f} | {m["precision"]:.6f} | \'\n                    f\'{m["recall"]:.6f} | {m["average_precision"]:.6f} | \'\n                    f\'{reps[0]["f1"]:.6f} | {reps[1]["f1"]:.6f} |\')\n    text += [\'\', \'## Selection and uncertainty\', \'\',\n        \'An alternative must gain at least 0.005 pooled F1, avoid any AP decrease, \'\n        \'and avoid lowering either repetition\\\'s fixed-threshold F1 by more than 0.005. \'\n        \'The rule was written before this cycle\\\'s new fits.\']\n    for name, result in record[\'recipes\'].items():\n        if name == \'reference\':\n            continue\n        m = result[\'combined\']\n        failures = []\n        if m[\'f1\'] < ref[\'combined\'][\'f1\']+.005:\n            failures.append(\'F1 gain below 0.005\')\n        if m[\'average_precision\'] < ref[\'combined\'][\'average_precision\']:\n            failures.append(\'AP decreased\')\n        if any(a[\'f1\'] < b[\'f1\']-.005 for a,b in zip(result[\'repeat_scores\'], ref[\'repeat_scores\'])):\n            failures.append(\'repetition stability failed\')\n        lo, hi = result[\'paired_f1_difference_95pct_descriptive\']\n        text.append(f\'\\n- {names[name]}: {"; ".join(failures) if failures else "eligible"}. \'\n                    f\'Descriptive paired-bootstrap F1 difference interval: [{lo:.4f}, {hi:.4f}].\')\n    text += [\'\', \'The intervals keep fitted models and thresholds fixed. They do not account \'\n        \'for historical selection, threshold tuning uncertainty, or unknown shared \'\n        \'simulation sources. They are not proof of a population-level improvement.\', \'\',\n        \'## Verification and continuation\', \'\',\n        f\'All {len(record["model_replay_max_errors"])} saved fold models were reloaded and replayed; \'\n        f\'maximum probability difference: **{max(record["model_replay_max_errors"].values()):.1g}**. \'\n        \'Every checkpoint verifies its manifest and exact fitting/validation IDs. \'\n        \'Each object is held out exactly once in each repetition. SpecType is used \'\n        \'only as auxiliary training supervision or training sample weights, never as a feature.\', \'\',\n        \'See [restart instructions](../../docs/RESUME_RESEARCH.md). The worker resumes \'\n        \'completed folds without retraining them and rejects duplicate workers. \'\n        \'Eight checkpoint tests cover failed atomic writes, stale configuration, \'\n        \'invalid fold records, concurrency and lock release after a killed worker.\', \'\',\n        \'**No GitHub or Kaggle upload was made.** Verified private F1 remains **0.6648**, \'\n        \'versus the top private benchmark **0.6824**. New local development scores \'\n        \'cannot establish that the top-ranked entry has been surpassed.\']\n    (folder/\'RESULTS.md\').write_text(\'\\n\'.join(text)+\'\\n\')\n    (folder/\'comparison.json\').write_bytes(source.read_bytes())\n    (folder/\'start_receipt.json\').write_bytes((source.parent/\'start_receipt.json\').read_bytes())\n\n\nif __name__ == \'__main__\':\n    for number in (3, 4):\n        render(number)\n', 'scripts/resume_research.py': '"""Resume the finite, local-only MALLORN experiment queue; never publishes."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport traceback\n\nfrom mallorn.checkpointing import AlreadyRunning, atomic_json, exclusive_lock\nimport cycle3\nimport cycle4\nfrom report_research import render\n\nROOT = Path(__file__).resolve().parents[1]\nSTATE = ROOT/\'artifacts/research_state.json\'\n\n\ndef catboost_access():\n    if sys.platform.startswith(\'linux\'):\n        try:\n            with open(\'/proc/self/statm\') as handle:\n                handle.read(100)\n        except OSError:\n            return False\n    return True\n\n\ndef status():\n    return {\'cycle3_completed_folds\': sum(1 for _ in (ROOT/\'artifacts/cycle3\').glob(\'*/repeat*_fold*.joblib\')),\n            \'cycle3_total_folds\': 30,\n            \'cycle3_compared\': (ROOT/\'artifacts/cycle3/comparison.json\').exists(),\n            \'cycle4_completed_folds\': sum(1 for _ in (ROOT/\'artifacts/cycle4\').glob(\'*/repeat*_fold*.joblib\')),\n            \'cycle4_total_folds\': 20,\n            \'cycle4_compared\': (ROOT/\'artifacts/cycle4/comparison.json\').exists(),\n            \'cycle2_refit_complete\': (ROOT/\'artifacts/cycle2/deployment/manifest.json\').exists(),\n            \'catboost_process_statistics_access\': catboost_access(),\n            \'publishing_allowed\': False,\n            \'previous_run\': json.loads(STATE.read_text()) if STATE.exists() else None}\n\n\ndef run(seconds, max_folds=None):\n    deadline = time.monotonic()+seconds\n    with exclusive_lock(ROOT/\'artifacts/research.lock\'):\n        atomic_json(STATE, {\'state\': \'running\', \'started_at\': cycle3.now(), \'publishing_allowed\': False})\n        try:\n            manifest = cycle3.initialize()\n            # Verify committed folds even when rerun after successful completion.\n            complete = cycle3.run_folds(manifest, deadline, max_folds)\n            if complete and not (cycle3.OUT/\'comparison.json\').exists():\n                cycle3.compare(manifest)\n            if complete:\n                render(3)\n            cycle4_complete = False\n            if complete and time.monotonic() < deadline:\n                fourth_manifest = cycle4.initialize(manifest)\n                fourth = cycle4.engine()\n                cycle4_complete = fourth.run_folds(fourth_manifest, deadline, max_folds)\n                if cycle4_complete and not (cycle4.OUT/\'comparison.json\').exists():\n                    fourth.compare(fourth_manifest)\n                if cycle4_complete:\n                    render(4)\n            queue_complete = complete and cycle4_complete\n            blocked = []\n            cat_done = (ROOT/\'artifacts/cycle2/deployment/manifest.json\').exists()\n            if not cat_done:\n                if not catboost_access():\n                    blocked.append(\'cycle2_refit: CatBoost cannot read /proc/self/statm\')\n                elif queue_complete and max_folds is None and time.monotonic() < deadline:\n                    # Atomic full-data checkpoints preserve completed siblings if\n                    # a timeout interrupts training or serialization of one model.\n                    with (ROOT/\'artifacts/cycle2-resume-deployment.log\').open(\'a\') as log:\n                        subprocess.run([sys.executable, \'scripts/refit_frozen.py\'], cwd=ROOT,\n                                       stdout=log, stderr=subprocess.STDOUT,\n                                       timeout=max(1, deadline-time.monotonic()), check=True)\n                    cat_done = True\n            state = \'complete\' if queue_complete and cat_done else \'blocked\' if queue_complete and blocked else \'checkpointed\'\n            result = {\'state\': state, \'updated_at\': cycle3.now(), \'cycle3_complete\': complete,\n                      \'cycle4_complete\': cycle4_complete,\n                      \'cycle2_refit_complete\': cat_done, \'blocked\': blocked, \'publishing_allowed\': False}\n            atomic_json(STATE, result)\n            return result\n        except BaseException as exc:\n            atomic_json(STATE, {\'state\': \'interrupted\' if isinstance(exc, (KeyboardInterrupt, subprocess.TimeoutExpired)) else \'failed\',\n                              \'updated_at\': cycle3.now(), \'error\': str(exc), \'publishing_allowed\': False})\n            raise\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'action\', choices=[\'run\', \'status\'])\n    parser.add_argument(\'--max-seconds\', type=int, default=1500)\n    parser.add_argument(\'--max-new-folds\', type=int)\n    args = parser.parse_args()\n    if args.max_seconds < 1 or (args.max_new_folds is not None and args.max_new_folds < 1):\n        parser.error(\'Limits must be positive\')\n    try:\n        print(json.dumps(status() if args.action == \'status\' else run(args.max_seconds, args.max_new_folds), indent=2))\n    except AlreadyRunning:\n        print(json.dumps({\'state\': \'already_running\', \'action\': \'No duplicate worker started\'}))\n    except Exception:\n        traceback.print_exc()\n        sys.exit(1)\n', 'scripts/verify_improvement.py': '"""Check improvement artifacts and replay a selected model\'s first OOF fold."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedKFold\n\nfrom improve import fit_candidate, predict_candidate\nfrom mallorn.data import sha256\nfrom mallorn.reporting import write_json\n\nROOT = Path(__file__).resolve().parents[1]\nOUT = ROOT / \'artifacts/improvement\'\n\n\ndef verify(replay):\n    split = json.loads((OUT / \'split.json\').read_text())\n    dev = split[\'development\']\n    assert len(dev) == len(set(dev)) == 2282\n    assert len(split[\'audit\']) == len(set(split[\'audit\'])) == 761\n    assert not set(dev) & set(split[\'audit\'])\n    details = {\'development_objects\': len(dev), \'audit_objects\': len(split[\'audit\']),\n               \'audit_outcomes_read\': False, \'schemas\': {}, \'oof\': {}}\n    for view in [\'physics\', \'enhanced\', \'complete\', \'physical_view\']:\n        a = pd.read_csv(OUT / f\'{view}_train.csv\', index_col=\'object_id\')\n        b = pd.read_csv(OUT / f\'{view}_test.csv\', index_col=\'object_id\')\n        assert a.columns.equals(b.columns), view\n        assert len(a) == 3043 and len(b) == 7135\n        assert a.index.is_unique and b.index.is_unique\n        assert not set(a.index) & set(b.index)\n        assert not set(a.columns) & {\'SpecType\', \'target\', \'Z_err\', \'split\', \'object_id\'}\n        assert not np.isinf(a.to_numpy()).any() and not np.isinf(b.to_numpy()).any()\n        details[\'schemas\'][view] = {\'features\': a.shape[1], \'test_objects\': len(b)}\n    for path in OUT.glob(\'*/oof.csv\'):\n        config = json.loads((path.parent / \'config.json\').read_text())\n        frame = pd.read_csv(path).set_index(\'object_id\')\n        assert frame.index.tolist() == dev\n        p = frame[list(config[\'candidates\'])].to_numpy()\n        assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()\n        details[\'oof\'][path.parent.name] = {\'models\': p.shape[1], \'sha256\': sha256(path)}\n    if replay:\n        name = \'cat_aux3_d5_top125\'\n        folder = OUT / \'complete_selected\'\n        config = json.loads((folder / \'config.json\').read_text())\n        x = pd.read_csv(OUT / \'complete_train.csv\', index_col=\'object_id\').loc[dev]\n        # Only development metadata is supplied to the fit.\n        meta = pd.read_csv(ROOT / \'data/mallorn/train_log.csv\').set_index(\'object_id\').loc[dev]\n        a, b = next(StratifiedKFold(5, shuffle=True, random_state=20261004).split(x, meta.target))\n        fitted = fit_candidate(x.iloc[a], meta.iloc[a], config[\'candidates\'][name], 42)\n        expected_columns = json.loads((folder / f\'{name}_fold0_columns.json\').read_text())\n        assert fitted[\'columns\'] == expected_columns\n        p = predict_candidate(fitted, x.iloc[b])\n        expected = np.load(folder / f\'{name}.npy\')[b]\n        np.testing.assert_allclose(p, expected, rtol=1e-10, atol=1e-12)\n        details[\'replay\'] = {\'candidate\': name, \'fold\': 0, \'objects\': len(b),\n                             \'max_probability_error\': float(np.max(np.abs(p - expected)))}\n    write_json(OUT / \'verification.json\', details)\n    print(json.dumps(details, indent=2))\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--replay\', action=\'store_true\')\n    verify(parser.parse_args().replay)\n', 'scripts/verify_run.py': '"""Independently audit a completed local run; load only your own model artifact."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import f1_score\n\nfrom mallorn.data import sha256, validate_submission\nfrom mallorn.modeling import predict_bundle\nfrom mallorn.reporting import write_json\n\n\ndef verify(data: Path, run: Path):\n    metrics = json.loads((run / "metrics.json").read_text())\n    manifest = json.loads((run / "manifest.json").read_text())\n    audit = json.loads((run / "fold_audit.json").read_text())\n    train = pd.read_csv(data / "train_log.csv").set_index("object_id")\n    test = pd.read_csv(data / "test_log.csv")\n    sample = pd.read_csv(data / "sample_submission.csv")\n    oof = pd.read_csv(run / "oof.csv")\n    assert not oof.object_id.duplicated().any()\n    assert set(oof.object_id) == set(train.index)\n    assert not oof.isna().any().any()\n    np.testing.assert_array_equal(oof.target, train.loc[oof.object_id, "target"])\n    np.testing.assert_allclose(f1_score(oof.target, oof.selected_prediction), metrics["primary"]["f1"], atol=1e-14)\n    held_out = []\n    for fold in audit:\n        fit, held = set(fold["fit_ids"]), set(fold["test_ids"])\n        assert not fit & held and fit | held == set(train.index)\n        held_out.extend(fold["test_ids"])\n        inner_held_out = []\n        for inner in fold["inner_splits"]:\n            a, b = set(inner["fit_ids"]), set(inner["validation_ids"])\n            assert not a & b and a | b == fit and not (a | b) & held\n            inner_held_out.extend(inner["validation_ids"])\n        assert len(inner_held_out) == len(set(inner_held_out)) == len(fit)\n        rows = oof[oof.outer_fold.eq(fold["fold"])]\n        assert set(rows.object_id) == held\n        chosen = fold["selected"]\n        np.testing.assert_array_equal(rows.selected_prediction, rows[f"{chosen}_prediction"])\n        np.testing.assert_allclose(rows.selected_probability, rows[f"{chosen}_probability"])\n    assert len(held_out) == len(set(held_out)) == len(train)\n    for item in manifest["files"]:\n        assert sha256(data / item["path"]) == item["sha256"]\n    assert sha256(run / "features.csv") == manifest["feature_matrix_sha256"]\n    features = pd.read_csv(run / "features.csv", index_col="object_id")\n    assert not np.isinf(features.to_numpy()).any()\n    bundle = joblib.load(run / "model.joblib")\n    probabilities, predictions = predict_bundle(bundle, features.loc[test.object_id])\n    saved = pd.read_csv(run / "test_probabilities.csv")\n    assert saved.object_id.tolist() == test.object_id.tolist()\n    np.testing.assert_allclose(probabilities, saved.probability, rtol=1e-8, atol=1e-10)\n    submission = pd.read_csv(run / "submission.csv")\n    validate_submission(submission, test.object_id)\n    assert submission.object_id.tolist() == sample.object_id.tolist()\n    expected = pd.Series(predictions, index=test.object_id).loc[sample.object_id]\n    np.testing.assert_array_equal(submission.prediction, expected)\n    result = {"status": "passed", "train_objects": len(train), "test_objects": len(test),\n              "outer_folds": len(audit), "input_hashes_verified": len(manifest["files"]),\n              "feature_matrix_hash_verified": True, "fold_partitions_verified": True,\n              "oof_labels_and_f1_verified": True, "saved_model_replay_verified": True,\n              "submission_ids_order_and_decisions_verified": True,\n              "test_predicted_positives": int(submission.prediction.sum()),\n              "submission_sha256": sha256(run / "submission.csv")}\n    write_json(run / "verification.json", result)\n    print(json.dumps(result, indent=2))\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--data", type=Path, required=True)\n    parser.add_argument("--run", type=Path, required=True)\n    args = parser.parse_args()\n    verify(args.data, args.run)\n', 'tests/test_checkpointing.py': '"""Failure-mode tests: interruption, conflicting work and invalid fold reuse."""\nimport ast\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nfrom mallorn.checkpointing import (AlreadyRunning, atomic_json, atomic_model,\n                                   exclusive_lock, freeze_json)\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]/\'scripts\'))\nfrom cycle3 import verify_fold\nfrom refit_frozen import atomic_csv\n\n\nclass CheckpointTests(unittest.TestCase):\n    def test_metrics_match_frozen_implementation_without_native_import(self):\n        src = Path(__file__).resolve().parents[1]/\'src/mallorn\'\n        names = {\'check_probability\', \'best_threshold\', \'score\'}\n        def functions(path):\n            return {node.name: ast.dump(node, include_attributes=False)\n                    for node in ast.parse(path.read_text()).body\n                    if isinstance(node, ast.FunctionDef) and node.name in names}\n        self.assertEqual(functions(src/\'modeling.py\'), functions(src/\'evaluation.py\'))\n\n    def test_failed_replace_preserves_previous_checkpoint(self):\n        with tempfile.TemporaryDirectory() as directory:\n            path = Path(directory)/\'state.json\'\n            atomic_json(path, {\'completed\': 4})\n            with patch(\'mallorn.checkpointing.os.replace\', side_effect=OSError(\'interrupted\')):\n                with self.assertRaises(OSError):\n                    atomic_json(path, {\'completed\': 5})\n            self.assertEqual(json.loads(path.read_text()), {\'completed\': 4})\n            self.assertEqual(list(Path(directory).glob(\'*.pending\')), [])\n\n    def test_freeze_rejects_changed_experiment(self):\n        with tempfile.TemporaryDirectory() as directory:\n            path = Path(directory)/\'manifest.json\'\n            freeze_json(path, {\'seed\': 42})\n            freeze_json(path, {\'seed\': 42})\n            with self.assertRaises(ValueError):\n                freeze_json(path, {\'seed\': 43})\n            self.assertEqual(json.loads(path.read_text()), {\'seed\': 42})\n\n    def test_lock_blocks_second_process_and_releases(self):\n        with tempfile.TemporaryDirectory() as directory:\n            path = Path(directory)/\'lock\'\n            program = (\n                \'from mallorn.checkpointing import exclusive_lock, AlreadyRunning\\n\'\n                \'import sys\\n\'\n                \'try:\\n\'\n                \' with exclusive_lock(sys.argv[1]): pass\\n\'\n                \'except AlreadyRunning: sys.exit(7)\\n\')\n            with exclusive_lock(path):\n                p = subprocess.run([sys.executable, \'-c\', program, str(path)], capture_output=True)\n                self.assertEqual(p.returncode, 7, p.stderr)\n            p = subprocess.run([sys.executable, \'-c\', program, str(path)], capture_output=True)\n            self.assertEqual(p.returncode, 0, p.stderr)\n\n    def test_atomic_model_round_trip(self):\n        with tempfile.TemporaryDirectory() as directory:\n            path = Path(directory)/\'fold.joblib\'\n            atomic_model(path, {\'probabilities\': np.array([.1, .9]), \'fold\': 2})\n            result = joblib.load(path)\n            np.testing.assert_array_equal(result[\'probabilities\'], [.1, .9])\n            self.assertEqual(result[\'fold\'], 2)\n\n    def test_interrupted_csv_write_preserves_previous_output(self):\n        class InterruptedFrame:\n            def to_csv(self, handle, index):\n                handle.write(\'incomplete\')\n                raise OSError(\'interrupted\')\n        with tempfile.TemporaryDirectory() as directory:\n            path = Path(directory)/\'submission.csv\'\n            frame = pd.DataFrame({\'object_id\': [\'a\'], \'prediction\': [1]})\n            atomic_csv(path, frame)\n            before = path.read_bytes()\n            with self.assertRaises(OSError):\n                atomic_csv(path, InterruptedFrame())\n            self.assertEqual(path.read_bytes(), before)\n            self.assertEqual(list(Path(directory).glob(\'*.pending\')), [])\n\n    def test_killed_worker_releases_lock(self):\n        with tempfile.TemporaryDirectory() as directory:\n            path = Path(directory)/\'lock\'\n            ready = Path(directory)/\'ready\'\n            program = (\n                \'from mallorn.checkpointing import exclusive_lock\\n\'\n                \'from pathlib import Path\\n\'\n                \'import sys\\n\'\n                \'with exclusive_lock(sys.argv[1]):\\n\'\n                \' Path(sys.argv[2]).write_text("ready")\\n\'\n                \' sys.stdin.read()\\n\')\n            proc = subprocess.Popen([sys.executable, \'-c\', program, str(path), str(ready)],\n                                    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE)\n            try:\n                import time\n                deadline = time.monotonic()+10\n                while not ready.exists() and proc.poll() is None and time.monotonic() < deadline:\n                    time.sleep(.01)\n                self.assertTrue(ready.exists())\n                with self.assertRaises(AlreadyRunning):\n                    with exclusive_lock(path):\n                        pass\n                proc.kill()\n                proc.communicate(timeout=10)\n                with exclusive_lock(path):\n                    pass\n            finally:\n                if proc.poll() is None:\n                    proc.kill()\n                proc.communicate(timeout=10)\n\n    def test_invalid_fold_is_never_reused(self):\n        ids = pd.Index([\'a\', \'b\', \'c\'])\n        a, b = np.array([0, 1]), np.array([2])\n        record = {\'manifest_sha256\': \'frozen\', \'fitting_ids\': [\'a\', \'b\'],\n                  \'validation_ids\': [\'c\'], \'probabilities\': np.array([.2])}\n        np.testing.assert_array_equal(verify_fold(record, \'frozen\', ids, a, b), [.2])\n        for bad in [dict(record, manifest_sha256=\'changed\'),\n                    dict(record, validation_ids=[\'b\']),\n                    dict(record, probabilities=[np.nan]),\n                    dict(record, probabilities=[1.1]),\n                    dict(record, probabilities=[.1, .2])]:\n            with self.assertRaises(ValueError):\n                verify_fold(bad, \'frozen\', ids, a, b)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'tests/test_integrity.py': "from __future__ import annotations\n\nimport json\nimport os\nfrom pathlib import Path\nimport tempfile\nimport unittest\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom pandas.testing import assert_frame_equal\nfrom sklearn.metrics import f1_score\n\nfrom mallorn.cli import prepare,run\nfrom mallorn.data import load_dataset,load_groups,validate_log,validate_photometry,validate_submission\nfrom mallorn.features import (FeatureConfig,band_features,dust_scale,extract_features,interpolate_observed)\nfrom mallorn.fixtures import make_fixture\nfrom mallorn.modeling import (TrainConfig,best_threshold,folds,predict_bundle,score,fit_model,predict_model,fit_candidates,predict_candidates)\nfrom mallorn.reporting import bootstrap_interval\n\n\nclass IntegrityTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        cls.temporary=tempfile.TemporaryDirectory()\n        cls.root=Path(cls.temporary.name)\n        make_fixture(cls.root,train_count=48,test_count=8)\n        cls.train,cls.test,cls.photo,cls.manifest=load_dataset(cls.root)\n        cls.metadata=pd.concat([cls.train,cls.test],ignore_index=True)\n        cls.features=extract_features(cls.metadata,cls.photo)\n\n    @classmethod\n    def tearDownClass(cls):\n        cls.temporary.cleanup()\n\n    def test_official_layout_and_hashes(self):\n        self.assertEqual(self.manifest['train_objects'],48)\n        self.assertEqual(self.manifest['positive_objects'],12)\n        self.assertEqual(len(self.manifest['files']),7)\n        self.assertTrue(all(len(f['sha256'])==64 for f in self.manifest['files']))\n\n    def test_missing_competition_data_fails_clearly(self):\n        with tempfile.TemporaryDirectory() as d:\n            with self.assertRaisesRegex(FileNotFoundError,'official data'):\n                load_dataset(Path(d))\n\n    def test_duplicate_metadata_rejected(self):\n        with self.assertRaises(ValueError):\n            validate_log(pd.concat([self.train,self.train.iloc[[0]]]),True)\n\n    def test_metadata_target_must_be_binary(self):\n        bad=self.train.copy();bad.loc[0,'target']=2\n        with self.assertRaises(ValueError):validate_log(bad,True)\n\n    def test_invalid_redshift_rejected(self):\n        bad=self.train.copy();bad.loc[0,'Z']=-1\n        with self.assertRaises(ValueError):validate_log(bad,True)\n\n    def test_target_not_required_for_test_log(self):\n        validate_log(self.test,False)\n\n    def test_negative_photometric_redshift_preserved_and_features_floored(self):\n        meta=self.test.iloc[[0]].copy();meta['Z']=-0.01751\n        validated=validate_log(meta,False)\n        self.assertEqual(validated.Z.iloc[0],-0.01751)\n        photo=self.photo[self.photo.object_id.eq(meta.object_id.iloc[0])]\n        negative=extract_features(validated,photo)\n        meta['Z']=0.\n        assert_frame_equal(negative,extract_features(meta,photo))\n\n    def test_nonfinite_redshift_and_negative_extinction_rejected(self):\n        for col,value in [('Z',np.nan),('Z',np.inf),('EBV',-.01)]:\n            bad=self.test.copy();bad.loc[0,col]=value\n            with self.assertRaises(ValueError):validate_log(bad,False)\n\n    def test_duplicate_photometry_rejected(self):\n        with self.assertRaisesRegex(ValueError,'Duplicate'):\n            validate_photometry(pd.concat([self.photo,self.photo.iloc[[0]]]))\n\n    def test_nonpositive_uncertainty_rejected(self):\n        bad=self.photo.copy();bad.loc[0,'Flux_err']=0\n        with self.assertRaises(ValueError):validate_photometry(bad)\n\n    def test_nonfinite_flux_rejected(self):\n        bad=self.photo.copy();bad.loc[0,'Flux']=np.inf\n        with self.assertRaises(ValueError):validate_photometry(bad)\n\n    def test_missing_flux_removed_without_imputing(self):\n        raw=self.photo.iloc[:20].copy();raw.iloc[0,raw.columns.get_loc('Flux')]=np.nan\n        clean=validate_photometry(raw)\n        self.assertEqual(len(clean),19)\n        self.assertFalse(clean.Flux.isna().any())\n        with self.assertRaises(ValueError):validate_photometry(raw.assign(Flux=np.nan))\n\n    def test_missing_flux_counts_audited_by_loader(self):\n        with tempfile.TemporaryDirectory() as d:\n            root=Path(d);make_fixture(root,train_count=48,test_count=8)\n            path=next(root.rglob('train_full_lightcurves.csv'))\n            raw=pd.read_csv(path);raw.loc[0,'Flux']=np.nan;raw.to_csv(path,index=False)\n            train,_,_,manifest=load_dataset(root)\n            self.assertEqual(len(train),48)\n            self.assertEqual(manifest['cleaning']['train']['missing_flux_rows_removed'],1)\n\n    def test_unknown_filter_rejected(self):\n        bad=self.photo.copy();bad.loc[0,'Filter']='X'\n        with self.assertRaises(ValueError):validate_photometry(bad)\n\n    def test_negative_flux_allowed(self):\n        bad=self.photo.iloc[:20].copy();bad['Flux']=-2.\n        self.assertTrue((validate_photometry(bad).Flux<0).all())\n\n    def test_features_exclude_label_metadata(self):\n        forbidden={'target','SpecType','split','Z_err','English Translation','object_id'}\n        self.assertFalse(forbidden & set(self.features.columns))\n        altered=self.metadata.copy()\n        altered['target']=1-altered['target'];altered['SpecType']='leak';altered['split']='changed'\n        assert_frame_equal(self.features,extract_features(altered,self.photo))\n\n    def test_feature_order_invariant(self):\n        shuffled=self.photo.sample(frac=1,random_state=19)\n        assert_frame_equal(self.features,extract_features(self.metadata,shuffled))\n\n    def test_object_features_do_not_depend_on_other_objects(self):\n        meta=self.metadata.iloc[[4]];photo=self.photo[self.photo.object_id.eq(meta.iloc[0].object_id)]\n        assert_frame_equal(self.features.loc[meta.object_id],extract_features(meta,photo))\n\n    def test_missing_band_has_stable_columns(self):\n        self.assertEqual(self.features.loc['synthetic_train_0000','u__n'],0)\n        self.assertTrue(np.isnan(self.features.loc['synthetic_train_0000','u__max']))\n        self.assertFalse(np.isinf(self.features.to_numpy()).any())\n\n    def test_dust_scale_preserves_snr(self):\n        self.assertAlmostEqual(dust_scale(0,'g'),1)\n        factor=dust_scale(.1,'g')\n        self.assertGreater(factor,1)\n        self.assertAlmostEqual((5*factor)/(2*factor),2.5)\n\n    def test_interpolation_no_extrapolation_or_season_bridge(self):\n        t=np.array([0.,10.,200.]);f=np.array([10.,20.,30.]);e=np.ones(3)\n        self.assertEqual(interpolate_observed(t,f,e,5,60,2),15)\n        self.assertTrue(np.isnan(interpolate_observed(t,f,e,-1,60,2)))\n        self.assertTrue(np.isnan(interpolate_observed(t,f,e,50,60,2)))\n\n    def test_interpolation_requires_both_endpoints_detected(self):\n        self.assertTrue(np.isnan(interpolate_observed(np.array([0.,10.]),np.array([-1.,10.]),np.ones(2),5,60,2)))\n\n    def test_rest_frame_duration(self):\n        t=np.array([0.,10.,20.]);f=np.array([10.,20.,10.]);e=np.ones(3)\n        result=band_features(t,f,e,10,1)\n        self.assertEqual(result['span'],10)\n        self.assertEqual(result['width_snr5'],10)\n\n    def test_phase_features_are_invariant_to_calendar_offset(self):\n        shifted=self.photo.copy();shifted['Time (MJD)']+=1000\n        np.testing.assert_allclose(self.features,extract_features(self.metadata,shifted),rtol=1e-7,atol=1e-7,equal_nan=True)\n\n    def test_threshold_matches_bruteforce(self):\n        y=np.array([0,1,0,1,1,0]);p=np.array([.1,.4,.4,.6,.8,.9])\n        threshold,value=best_threshold(y,p)\n        self.assertAlmostEqual(value,max(f1_score(y,p>=t) for t in np.unique(p)))\n        self.assertAlmostEqual(value,f1_score(y,p>=threshold))\n\n    def test_threshold_ties_keep_equal_probabilities_together(self):\n        threshold,value=best_threshold([0,1,0,1],[.3,.3,.3,.3])\n        self.assertAlmostEqual(threshold,.3)\n        self.assertAlmostEqual(value,2/3)\n\n    def test_threshold_rejects_bad_probabilities(self):\n        for p in ([.1,np.nan],[-.1,.8],[.1,1.1]):\n            with self.assertRaises(ValueError):best_threshold([0,1],p)\n\n    def test_group_folds_disjoint(self):\n        y=np.tile([0,1],24);groups=np.repeat(np.arange(24),2)\n        for a,b in folds(y,groups,3,42):\n            self.assertFalse(set(groups[a]) & set(groups[b]))\n\n    def test_insufficient_minority_fails(self):\n        with self.assertRaises(ValueError):folds(np.array([0,0,0,1]),None,3,42)\n\n    def test_group_mapping_reorders_by_object(self):\n        ids=self.train.object_id\n        mapping=pd.DataFrame({'object_id':ids,'group_id':[f'g{i//2}' for i in range(len(ids))]})\n        path=self.root/'groups.csv';mapping.iloc[::-1].to_csv(path,index=False)\n        actual=load_groups(path,ids)\n        self.assertEqual(actual[0],'g0');self.assertEqual(actual[-1],'g23')\n\n    def test_submission_exact_ids_and_binary_values(self):\n        good=pd.DataFrame({'object_id':self.test.object_id,'prediction':0})\n        validate_submission(good,self.test.object_id)\n        bad=good.copy();bad['prediction']=.5\n        with self.assertRaises(ValueError):validate_submission(bad,self.test.object_id)\n        with self.assertRaises(ValueError):validate_submission(good.iloc[:-1],self.test.object_id)\n\n    def test_confusion_matrix_and_f1(self):\n        actual=score([0,0,1,1],[.1,.9,.2,.8],[0,1,0,1])\n        self.assertEqual(actual['confusion_matrix_tn_fp_fn_tp'],[1,1,1,1])\n        self.assertEqual(actual['f1'],.5)\n\n    def test_bootstrap_is_repeatable_and_bounded(self):\n        a=bootstrap_interval([0,1,1,0],[0,1,0,0],repeats=30)\n        b=bootstrap_interval([0,1,1,0],[0,1,0,0],repeats=30)\n        self.assertEqual(a,b)\n        self.assertTrue(0<=a['ci95_low']<=a['ci95_high']<=1)\n\n    def test_lightgbm_fits_a_nonconstant_model_and_replays(self):\n        rng=np.random.default_rng(94)\n        x=pd.DataFrame({'g__mean':rng.normal(size=160),'color_g_r__p0':rng.normal(size=160)})\n        y=(x['g__mean']>0).astype(int).to_numpy()\n        model=fit_model('lgb_full',x,y,TrainConfig(iterations=20,threads=1),42)\n        p=predict_model(model,x)\n        self.assertGreater(np.ptp(p),.1)\n        path=self.root/'temporary_test_model.joblib';joblib.dump(model,path)\n        np.testing.assert_allclose(p,predict_model(joblib.load(path),x),rtol=0,atol=0)\n\n    @unittest.skipUnless(os.environ.get('MALLORN_TEST_CATBOOST')=='1','CatBoost needs process-statistics access; enable on a compatible host')\n    def test_optional_hybrid_backend_fit_and_blend(self):\n        rng=np.random.default_rng(94)\n        x=pd.DataFrame({'g__mean':rng.normal(size=160),'color_g_r__p0':rng.normal(size=160)})\n        y=(x['g__mean']>0).astype(int).to_numpy()\n        models=fit_candidates(x,y,TrainConfig(iterations=12,threads=1,backend='hybrid'),42)\n        p=predict_candidates(models,x)\n        self.assertGreater(np.ptp(p['cat_full']),.01)\n        np.testing.assert_allclose(p['blend'],.5*p['cat_full']+.5*p['lgb_full'])\n\n\nclass EndToEndTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        cls.temporary=tempfile.TemporaryDirectory()\n        cls.root=Path(cls.temporary.name)/'data';cls.output=Path(cls.temporary.name)/'run'\n        make_fixture(cls.root,train_count=48,test_count=8)\n        cls.metrics=run(cls.root,cls.output,TrainConfig(3,2,8,42,1),FeatureConfig())\n\n    @classmethod\n    def tearDownClass(cls):cls.temporary.cleanup()\n\n    def test_run_is_labeled_synthetic(self):\n        self.assertEqual(self.metrics['data_kind'],'synthetic_fixture')\n        self.assertFalse((self.output/'submission.csv').exists())\n        self.assertIn('NOT COMPETITION PERFORMANCE',(self.output/'RESULTS.md').read_text())\n\n    def test_outer_and_inner_ids_do_not_leak(self):\n        audit=json.loads((self.output/'fold_audit.json').read_text())\n        tested=[]\n        for fold in audit:\n            train,test=set(fold['fit_ids']),set(fold['test_ids'])\n            self.assertFalse(train&test);tested+=fold['test_ids']\n            for inner in fold['inner_splits']:\n                fit,val=set(inner['fit_ids']),set(inner['validation_ids'])\n                self.assertFalse(fit&val)\n                self.assertEqual(fit|val,train)\n                self.assertFalse((fit|val)&test)\n        self.assertEqual(len(tested),len(set(tested)))\n        self.assertEqual(len(tested),48)\n\n    def test_oof_metrics_independently_recompute(self):\n        oof=pd.read_csv(self.output/'oof.csv')\n        self.assertFalse(oof.isna().any().any())\n        self.assertAlmostEqual(f1_score(oof.target,oof.selected_prediction),self.metrics['primary']['f1'])\n        self.assertFalse(self.metrics['deployment']['tuning_f1_is_performance_estimate'])\n\n    def test_saved_model_replays_and_submission_follows_sample_order(self):\n        _,test,x,_=prepare(self.root,FeatureConfig())\n        bundle=joblib.load(self.output/'model.joblib')\n        p,pred=predict_bundle(bundle,x.loc[test.object_id])\n        saved=pd.read_csv(self.output/'test_probabilities.csv')\n        np.testing.assert_allclose(p,saved.probability,rtol=1e-10,atol=1e-10)\n        submission=pd.read_csv(self.output/'synthetic_submission.csv')\n        sample=pd.read_csv(self.root/'sample_submission.csv')\n        self.assertEqual(submission.object_id.tolist(),sample.object_id.tolist())\n        self.assertEqual(submission.prediction.tolist(),pd.Series(pred,index=test.object_id).loc[sample.object_id].tolist())\n\n    def test_inference_schema_guard(self):\n        _,test,x,_=prepare(self.root,FeatureConfig())\n        with self.assertRaises(ValueError):\n            predict_bundle(joblib.load(self.output/'model.joblib'),x.loc[test.object_id].iloc[:,:-1])\n\n    def test_existing_run_is_not_overwritten(self):\n        with self.assertRaises(FileExistsError):run(self.root,self.output,TrainConfig(),FeatureConfig())\n\n\nif __name__=='__main__':unittest.main()\n", 'tests/test_morphology_features.py': "import unittest\n\nimport numpy as np\n\nfrom mallorn.morphology_features import object_features\n\n\nclass MorphologyTests(unittest.TestCase):\n    def task(self):\n        phase = np.tile(np.linspace(-80, 220, 101), 3)\n        band = np.repeat([1, 2, 3], 101)\n        shape = np.exp(-.5 * (np.minimum(phase, 0) / 20) ** 2)\n        shape *= (1 + np.maximum(phase, 0) / 50) ** (-5 / 3)\n        flux = (20 + 5 * band) * shape\n        return ('synthetic', .2, 0., 50000 + 1.2 * phase,\n                flux, np.ones(len(flux)) * .3, band)\n\n    def test_powerlaw_fit_recovers_rest_frame_scales(self):\n        _, values = object_features(self.task())\n        self.assertAlmostEqual(values['flare_powerlaw_rise'], 20., places=3)\n        self.assertAlmostEqual(values['flare_powerlaw_fall'], 50., places=3)\n        self.assertLess(values['flare_powerlaw_chi2'], 1e-8)\n        self.assertLess(values['flare_powerlaw_exp_chi2_ratio'], 1e-6)\n\n    def test_calendar_translation_does_not_change_features(self):\n        task = self.task()\n        _, original = object_features(task)\n        _, shifted = object_features((*task[:3], task[3] + 20000, *task[4:]))\n        self.assertEqual(original.keys(), shifted.keys())\n        np.testing.assert_allclose(list(original.values()), list(shifted.values()),\n                                   rtol=1e-5, atol=1e-5, equal_nan=True)\n        self.assertFalse(np.isinf(list(original.values())).any())\n\n\nif __name__ == '__main__':\n    unittest.main()\n", 'tests/test_physics_features.py': "import unittest\nimport numpy as np\n\nfrom mallorn.physics_features import object_features,thermal_fit,luminosity_distance_cm\nfrom mallorn.features import WAVELENGTHS\nfrom mallorn.data import BANDS\n\n\nclass PhysicsTests(unittest.TestCase):\n    def task(self):\n        t=np.tile(np.linspace(0,180,18),6)\n        band=np.repeat(np.arange(6),18)\n        flux=(1+.1*band)*20*np.exp(-.5*((t-70)/25)**2)\n        error=np.ones(len(t))*.8\n        return ('fixture',.2,.02,t,flux,error,band)\n\n    def test_gp_features_are_calendar_translation_invariant(self):\n        task=self.task();_,a=object_features(task)\n        shifted=(*task[:3],task[3]+40000,*task[4:])\n        _,b=object_features(shifted)\n        self.assertEqual(set(a),set(b))\n        np.testing.assert_allclose(list(a.values()),list(b.values()),rtol=1e-6,atol=1e-6,equal_nan=True)\n        self.assertFalse(np.isinf(list(a.values())).any())\n\n    def test_thermal_fit_recovers_synthetic_temperature_and_radius(self):\n        z=.15;distance=luminosity_distance_cm(z);temperature=20000.;radius=1e15\n        nu=2.99792458e18/np.array([WAVELENGTHS[b] for b in BANDS])*(1+z)\n        bb=2*6.62607015e-27*nu**3/2.99792458e10**2/np.expm1(6.62607015e-27*nu/(1.380649e-16*temperature))\n        flux=(1+z)*np.pi*bb*(radius/distance)**2/1e-29\n        fitted=thermal_fit(flux,.03*flux,z,distance)\n        self.assertLess(abs(fitted[0]-np.log10(temperature)),.02)\n        self.assertLess(abs(fitted[1]-np.log10(radius)),.04)\n\n    def test_thermal_fit_rejects_unsupported_measurements(self):\n        self.assertTrue(np.isnan(thermal_fit(np.ones(6),np.ones(6),.2,luminosity_distance_cm(.2))).all())\n\n    def test_distance_small_redshift_limit(self):\n        z=.0001\n        self.assertAlmostEqual(luminosity_distance_cm(z)/(299792.458/70*z*3.085677581491367e24),1.,places=3)\n\n\nif __name__=='__main__':unittest.main()\n", 'tests/test_selected_ensemble.py': "import tempfile\nimport unittest\nfrom pathlib import Path\nfrom unittest.mock import patch\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom mallorn import selected_ensemble as s\n\n\nclass RecoveredSelectionTests(unittest.TestCase):\n    def test_correlation_pruning_and_empty_columns(self):\n        x = pd.DataFrame({'a':np.arange(20.),'duplicate':-2*np.arange(20.),\n                          'b':np.tile([0.,1.],10),'empty':np.nan,'constant':1.})\n        self.assertEqual(s.select_columns(x,[10,9,8,100,100]),['a','b'])\n\n    def test_forbidden_labels(self):\n        with self.assertRaises(ValueError): s.select_columns(pd.DataFrame({'target':[0,1]}),[1])\n\n    def test_auxiliary_target_precedence(self):\n        meta = pd.DataFrame({'target':[1,0,0,0,0],'SpecType':['AGN','AGN','SN IIn','SLSN-I','SN Ia']})\n        np.testing.assert_array_equal(s.labels_weights(meta,'fine5')[0],[4,0,2,3,1])\n\n    def test_training_boundary_and_replay(self):\n        rng=np.random.default_rng(7)\n        x=pd.DataFrame(rng.normal(size=(100,8)),columns=[f'x{i}' for i in range(8)])\n        meta=pd.DataFrame({'target':(x.x0>.8).astype(int),'SpecType':np.where(x.x1>0,'AGN','SN Ia')})\n        held=pd.DataFrame(rng.normal(size=(15,8)),columns=x.columns,index=np.arange(100,115))\n        seen=[]\n        original=s.select_columns\n        def select(frame,gain):\n            seen.extend(frame.index)\n            return original(frame,gain)\n        with patch.object(s,'select_columns',side_effect=select),patch.dict(s.PARAMS,{'n_estimators':8,'min_child_samples':5}):\n            model=s.fit(x,meta,'aux',42,1)\n        self.assertEqual(set(seen),set(x.index))\n        self.assertFalse(set(seen)&set(held.index))\n        p=s.predict(model,held)\n        with tempfile.TemporaryDirectory() as directory:\n            path=Path(directory)/'model.joblib'\n            joblib.dump(model,path)\n            np.testing.assert_array_equal(p,s.predict(joblib.load(path),held))\n", 'docs/DATA.md': "# Data setup and execution\n\n## Obtain the authorized files\n\nOpen the [official data page](https://www.kaggle.com/competitions/mallorn-astronomical-classification-challenge/data), sign in, review and accept the competition rules, and download the files. Rule acceptance is a user action. Do not share passwords or API credentials in a chat or commit them to GitHub.\n\nExtract the archive so `train_log.csv`, `test_log.csv` and `sample_submission.csv` are directly inside `data/mallorn/`. The light-curve files may be in the official `split_01` through `split_20` folders; the loader discovers them recursively. Each folder contains `train_full_lightcurves.csv` and `test_full_lightcurves.csv`. A `split` field, if present in metadata, must match the parent folder name exactly.\n\nThe loader deliberately fails on missing files, unknown filters, duplicate observations, infinite flux, nonfinite times/errors, nonpositive errors, overlapping train/test IDs and inconsistent object coverage. Official observations with missing flux are omitted and counted by train/test partition in the manifest; no flux is imputed. Every metadata object must still have observations. Negative measured flux is retained.\n\nThe supplied archive contains one slightly negative test photometric redshift (`-0.01751`). Raw files and metadata remain unchanged. Feature extraction uses `max(Z, 0)` consistently for redshift features, rest-frame time and colour queries. The same fixed rule applies to train and test, requires no fitting, and was introduced before model evaluation. Z must be finite and greater than -1; EBV must be finite and nonnegative.\n\n## Input schema\n\n| File | Required columns | Treatment |\n| --- | --- | --- |\n| `train_log.csv` | `object_id`, `Z`, `EBV`, `target` | Binary target; finite Z > -1 and EBV >= 0; negative Z floored only in features |\n| `test_log.csv` | `object_id`, `Z`, `EBV` | No target used |\n| Light curves | `object_id`, `Time (MJD)`, `Flux`, `Flux_err`, `Filter` | Times in days, flux/errors in microjansky; filters u,g,r,i,z,y; missing flux observations omitted and audited |\n| `sample_submission.csv` | `object_id`, `prediction` | All test IDs exactly once; original order preserved |\n| Optional group file | `object_id`, `group_id` | Exactly one genuine source-template identifier per training object |\n\n`SpecType`, the translation of the object name, file `split`, object ID, and `Z_err` are never classifier inputs. In particular, file splits are storage partitions, not assumed source-template families.\n\n## Train and evaluate\n\n```bash\nmallorn run --data data/mallorn --output artifacts/run_001 --threads 2\n```\n\nIf the organizers provide the true template/source mapping, enable group-aware splitting at both levels:\n\n```bash\nmallorn run --data data/mallorn --groups data/source_groups.csv --output artifacts/grouped_001\n```\n\nThe group mapping must include exactly the training objects. Do not invent groups from the random object names or claim that unique object IDs solve template leakage.\n\nFor a separately labelled diagnostic run, parameters include `--iterations`, `--outer-folds`, `--inner-folds`, `--seed` and `--no-deredden`. These change the experiment and must be recorded; comparing many variants makes the existing local evaluation a development set. Keep a genuinely untouched period/dataset or organizer evaluation for final claims.\n\nThe locally verified backend is `--backend lightgbm` (default). `--backend hybrid` includes CatBoost and an equal-weight blend. CatBoost requires a host where its native runtime can read process statistics; it could not fit in this authoring sandbox. Before using hybrid results, run its optional integration check on a compatible host:\n\n```bash\n# Linux/macOS:\nMALLORN_TEST_CATBOOST=1 python -m unittest discover -s tests -v\n```\n\nFor PowerShell, set `$env:MALLORN_TEST_CATBOOST='1'` before the test command. This flag enables an additional genuine CatBoost fit/replay test; it does not replace CatBoost with a mock.\n\n## Validate and replay\n\n```bash\nmallorn check-submission --submission artifacts/run_001/submission.csv --test-log data/mallorn/test_log.csv\nmallorn predict --data data/mallorn --model artifacts/run_001/model.joblib --output artifacts/replayed_submission.csv\npython scripts/verify_run.py --data data/mallorn --run artifacts/run_001\n```\n\n`verify_run.py` checks input hashes, fold boundaries, held-out labels/F1, saved-model replay from the hashed feature table, and exact submission order/decisions. It writes `verification.json`. `predict` reuses the saved feature configuration and does not fit a model. The current loader expects the original dataset layout, including training logs, for its object-coverage contract. Only test rows are sent to the saved estimator. `joblib` uses pickle: load only artifacts you produced and trust.\n\nThere is no Kaggle submission call, credential handling, automatic rule acceptance, or notebook publication in the package. Local outputs stay under the git-ignored `artifacts/` directory.\n\n## Provenance and storage\n\nThe manifest records SHA-256 hashes and sizes of all inputs, feature names/configuration, the feature CSV hash, and an optional group-map hash. Metrics record package versions and the training configuration. Input data, feature tables, models and prediction files are ignored by git. Publish aggregate findings only after confirming the data-use terms. Exact replay needs the same files, core package versions and configuration; floating-point differences can occur across hardware.\n", 'docs/FIFTH_CYCLE_RECOVERY.md': "# Recovered cycle-5 method\n\nThe new `selected_ensemble.py` reconstructs the recorded modeling method:\ntwo training-fold LightGBM selectors (600 trees; seeds +10000/+20000), normalized\ngain averaging, 125 features pruned at absolute Pearson correlation .95, and\nthree 900-tree final models (seeds +0/+1000/+2000). Median filling is used only\nfor training-fold correlation. The model receives missing values directly.\n\nComplete-feature five-class and physical-view three-class components use the\nsame fixed LightGBM parameters and auxiliary class weights as the historical\ncycle. `cycle5_recovered.py` reuses cycle 3's two five-fold partitions, threshold\nmethod, five fixed blends, checkpoint audits and unchanged acceptance gates.\n\nThis is recovered source, not an assertion that the missing original model\nfiles were recovered or byte-for-byte identical source was restored. A rerun\nwrites a separate `artifacts/cycle5_recovered` namespace. It needs the local\nauthorized feature caches and cycle-2 reference OOF predictions, which are not\ndistributed in the public repository. Do not modify or fabricate old manifests.\n\n```bash\nMPLCONFIGDIR=artifacts/mpl OPENBLAS_NUM_THREADS=1 MALLORN_THREADS=4 \\\n  python scripts/cycle5_recovered.py --max-seconds 1500\n```\n\nThe historical cycle-5 measurements are preserved with their provenance in\n[the recovery note](../reports/RECOVERY_20261005.md). They remain development\nselection evidence, not independent validation or Kaggle scores.\n", 'docs/FOURTH_CYCLE_PROTOCOL.md': "# Fourth cycle: confusing negative classes\n\nMotivation, observed after cycle 3: the retained cycle-2 reference produces 79\nfalse positives, including 33 AGN, 18 of the 41 SN IIn objects, and 10 of the\n25 superluminous supernovae. Fine-grained class supervision and explicit\nattention to those confusing negatives are concrete next hypotheses. This is\nadaptive development on historically reused labels, not a new independent test.\n\nBefore fitting, fix two LightGBM candidates on the 1,126 complete features:\n\n- `fine5`: five classes: AGN, ordinary supernovae, SN IIn, SLSN, and TDE. Use\n  weights 1, 1, 1, 1, 3. Preserve the original binary target for all scoring.\n- `hard_negative_binary`: binary target; sample weights 3 for TDE, 2 for SN IIn\n  or SLSN, and 1 otherwise. These labels affect fitting-fold sample weights only.\n\nUse the exact cycle-3 LightGBM hyperparameters and the same two five-fold\npartitions. No extra hyperparameter trials or early stopping on validation.\nSeeds remain 42 + 100*repeat + fold. All class construction, sample weights and\ntraining operate on fitting rows only; SpecType is never a predictor.\n\nCompare five fixed recipes: the cycle-2 diverse reference; its 50:50 blend with\neach new candidate; the equal-weight new-candidate ensemble; and its 50:50 blend\nwith the reference. Apply cycle 3's unchanged threshold method and selection\nrule: F1 improvement >=0.005, no AP decrease, and no individual repetition's F1\ndecrease greater than 0.005. Report all failures and uncertainty. Do not promote\ncycle 3's 0.7030-F1 blend merely because it has a larger pooled tuning score.\n\nReuse the atomic fold checkpoint and model-replay checks. This cycle's source,\nprotocol, packages, inputs and reference predictions must be frozen separately.\nNo GitHub or Kaggle uploads. Stop the finite queue after comparison.\n", 'docs/IMPROVEMENT_METHODS.md': "# Local improvement cycle\n\nThe public baseline remains unchanged while this experiment is evaluated.\nThe target is approximately 0.68 F1. Precision and recall are reported\nseparately; a local F1 is not a Kaggle private-test score.\n\n## Features\n\nStarting from the original 414 features, the physical extractor adds 471\nmeasurements. A two-dimensional Gaussian process smooths flux over rest-frame\ntime and log wavelength. Its summaries include peak widths, time scales,\nphase-dependent colors and their evolution. Approximate monochromatic\nblackbody fits summarize temperature, radius and luminosity at several phases.\nPer-band luminosity proxies and season-aware resampled shapes complement these\nfeatures. The GP uses at most 160 observations per object, retaining both\nstrong detections and time coverage.\n\nThe morphology extractor adds 241 measurements: detection spans, flux/SNR\ndistribution shape, band ratios, and Gaussian-rise flare fits with exponential\nor power-law decay. A focused physical view adds 24 derived measurements,\nincluding temperature/radius/luminosity changes in physical units. These are\ndescriptive approximations, not precise astrophysical parameter estimates.\n\nEvery extractor works on one object's photometry, redshift and extinction.\nIt does not use targets, spectral class labels, other objects' labels, or\ncompetitor feature tables. Negative measured flux is retained.\n\n## Models and supervised feature selection\n\nCatBoost and LightGBM are compared using the same five development folds.\nBinary classifiers are supplemented by three-class supervision (AGN, other\ntransients, TDE), a six-class variant, and a two-stage classifier that separates\nAGN before discriminating TDE from other transients. Spectral class is used\nonly as a training-fold target, never as a prediction-time input.\n\nSelected-feature candidates fit a preliminary CatBoost model on the training\nfold, rank its feature importances, and retain 125 or 250 columns while pruning\ntraining-fold correlations above 0.95. Validation objects never participate in\nranking or correlation estimation. Equal-weight probability ensembles are\ncompared; every tried recipe and optimized threshold is recorded.\n\n## Evaluation boundaries\n\nSee [IMPROVEMENT_PROTOCOL.md](IMPROVEMENT_PROTOCOL.md). Development has 2,282\nobjects, including 111 TDEs; the reserved audit has 761 objects. Development\nscores include candidate and threshold selection and therefore are optimistic.\nThe audit is reserved for this cycle, but the earlier published baseline had\nalready used all training labels. Real source/template groups are unavailable.\n\n`audit_improvement.py freeze` writes a recipe with feature, split, metadata and\nimplementation hashes. The separate `audit` command fits on development only,\nfixes predictions at the frozen threshold, and then scores reserved outcomes.\nIt cannot replace a completed audit with another tuned recipe. Interrupted\naudits can resume only with the identical recipe.\n\n## Reproduce locally\n\nUse the pinned project dependencies and place the official competition data\nunder `data/mallorn`. The published baseline feature cache must first exist.\n\n```bash\nexport MPLCONFIGDIR=artifacts/mpl\nexport OPENBLAS_NUM_THREADS=1\nexport MALLORN_THREADS=3\npython scripts/build_physics_features.py --mode train\npython scripts/build_morphology_features.py --mode train\npython scripts/build_physical_view.py --mode train\npython scripts/improve.py --features artifacts/improvement/complete_train.csv \\\n  --label complete_selected --batch selected\n```\n\nOther recorded batches and feature paths appear in each experiment's\n`config.json`. Build test features using the same scripts with `--mode test`.\nTraining requires CatBoost to read its own process memory statistics; some\nrestricted runtimes hide that file. Do not bypass runtime access controls.\n\n## Method references\n\n- [Second-place write-up](https://www.kaggle.com/competitions/mallorn-astronomical-classification-challenge/writeups/mallorn-2nd-place-solution)\n- [Third-place write-up](https://www.kaggle.com/competitions/mallorn-astronomical-classification-challenge/writeups/3rd-place-write-up-catboost-with-threshold-tuning)\n- [Avocado paper](https://arxiv.org/abs/1907.04690)\n\nThese motivated the feature families and model comparisons. This code is an\nindependent implementation and does not reproduce their full pipelines.\n", 'docs/IMPROVEMENT_PROTOCOL.md': "# Local improvement protocol — 3 October 2026\n\nNo GitHub publication during model development. The target is approximately\n0.68 F1, reflecting the leaders' reported private-test F1, not classification\nprecision alone. Local results cannot establish leaderboard equivalence.\n\nThe existing official_001 baseline is frozen. It has already evaluated all\ntraining labels, so no subset of these data can be called historically unseen.\nFor this improvement cycle, reserve 25% of objects using stratified splitting\nwith seed 20261003. Do not inspect its outcomes while developing features,\nmodels or thresholds. The remaining 75% is the development set. Use five-fold\nstratified development OOF predictions (seed 20261004) for candidate and\nthreshold selection; label these scores explicitly as tuning estimates.\n\nInvestigate label-free light-curve shape, luminosity proxies, season-aware\ninterpolation, colors, analytic flare models and Gaussian-process summaries,\nguided by the published second- and third-place solutions. Compare CatBoost,\nLightGBM, class weighting and fixed probability ensembles. Never infer source\nfamilies from names or storage splits. No competition test labels, pseudo-labels\nor competitor feature files enter fitting. If auxiliary class labels are used,\nthey are training-fold supervision only, never predictor features.\n\nFreeze the chosen feature recipe, model configuration, ensemble and threshold\nbefore evaluating the reserved audit subset once. Report precision, recall,\nF1, counts and uncertainty even if the target is missed. Do not repeatedly tune\nagainst the audit outcome. Any later cycle must disclose that reuse and cannot\ncall it independent confirmation. Record all candidate results and failures.\n\nNo result is guaranteed. All original-source independence concerns remain;\nan organizer test-set evaluation is necessary for competition-level claims.\n", 'docs/INTERVIEW_GUIDE.md': '# Explain the project accurately\n\nThis repository was created with OpenAI Codex assistance. Run it, inspect the tests and understand the design before presenting it as evidence of your skills. Distinguish assistance from your own later experiments and contributions.\n\n## One-minute explanation\n\n“This project builds a reproducible classifier for rare astronomical tidal disruption events from irregular light curves. It uses features inspired by published leading MALLORN solutions and compares boosted-tree models. I focused on separating feature engineering, model selection, threshold tuning and evaluation, because optimizing F1 on the same predictions used to report performance can give an optimistic result. The code exports every fold and prediction so the experiment can be audited. On the official data, the fixed nested object-level evaluation achieved F1 0.5254, precision 0.4515 and recall 0.6284. Source-template independence is unresolved, and no Kaggle leaderboard score has been measured.”\n\nThe measured result and limitations are recorded in [the experiment report](../reports/official_001/RESULTS.md).\n\n## Decisions to understand\n\n- **Why F1?** TDEs are rare; an all-negative model can have high accuracy but finds none. Precision and recall must be reported alongside F1.\n- **Why nested folds?** A threshold is a learned decision, just like a model parameter. The outer test labels cannot choose it.\n- **Why object groups may not be enough?** Multiple simulated curves can originate from the same real transient. Genuine parent-source groups are needed to test independence at that level.\n- **Why not interpolate every colour?** A seasonal gap or noisy negative flux can make a magnitude colour unsupported. Missing values are preferable to confident invented measurements.\n- **Why no winner claim?** Local validation and the Kaggle private test set are different evaluations. Software fixtures measure neither.\n- **Why boosting first?** Leading write-ups support strong engineered features and boosted trees. The first independent implementation is deliberately small enough to inspect; the neural/GP components of those solutions are not reproduced.\n\nStart reading in this order: `data.py`, `features.py`, `modeling.py`, `cli.py`, then `tests/test_integrity.py`.\n', 'docs/METHODOLOGY.md': "# Experiment protocol\n\nThe model, feature families and nested selection protocol were written before competition-data training. The task is binary full-light-curve classification: TDE is 1, all other types are 0. F1 is the primary metric. The first official-data evaluation is documented in [the run report](../reports/official_001/RESULTS.md).\n\nBefore fitting, inspection of the supplied official archive required two data-contract corrections: omit observations with missing flux (891 train, 2,022 test) and floor the single negative test photometric redshift to zero during feature extraction. Input files remain unchanged; cleaning counts and hashes are recorded. No candidate settings or feature families were changed after viewing model scores.\n\n## Feature extraction\n\nEach object is processed independently. No labels, population imputation, feature selection or train/test distribution fitting occurs during extraction. Missing bands retain a fixed schema. Each learned imputer is fitted inside its own model's fitting fold.\n\n1. Validate the official photometry and metadata contracts; omit and audit missing-flux observations. Use `max(Z, 0)` for all redshift-dependent features, while preserving raw metadata.\n2. Optionally de-redden flux and its error using Fitzpatrick (1999), Rv=3.1, evaluated at fixed representative band wavelengths. Scaling both preserves SNR. This is a monochromatic approximation, not a passband-integrated SED correction.\n3. Reference time is the highest-SNR observation in g/r/i, falling back to all bands if none exist. Time relative to this reference is divided by 1+Z. It is a practical phase proxy, not an estimate of the true physical peak with fitted uncertainty.\n4. Compute per-band count, time coverage, gaps, flux quantiles and shape, inverse-variance mean, constant-flux chi-square, SNR and detection-width statistics. Negative flux is retained.\n5. Bin normalized flux and SNR by fixed rest-frame phase intervals, retaining counts to expose sparsity. The bins are fixed before fitting.\n6. For positive detections after phase +5 days, compute a weighted descriptive log-flux/log-time slope and its unweighted fit R-squared. It is not a physical TDE likelihood fit.\n7. Compute a weighted floating-mean Lomb–Scargle summary where coverage permits. No claim of physical periodicity is inferred from the largest sampled periodogram value.\n8. Compute pairwise AB colour differences from positive flux at shared phases -10, 0, +15, +30 and +60 days. Linear interpolation is bounded by observed samples, requires both endpoint SNRs >=2, and never bridges an observer-frame gap over 60 days. Unsupported colours remain missing.\n\nRedshift error is excluded because the official train and test sets have different availability. The archive confirms that Z_err is missing for every training object and present for every test object. Median Z is 0.4818 in training and 0.4842 in test; similar marginal distributions do not remove the spectroscopic/photometric measurement shift. Synthetic redshift noise is not introduced.\n\n## Fixed candidate set\n\n| Candidate | Features | Model |\n| --- | --- | --- |\n| `prior` | None | Training positive-class fraction |\n| `lgb_basic` | Statistics, SNR and timing; no phase bins, colours, decline or period features | LightGBM |\n| `lgb_full` | All features | LightGBM |\n| `cat_full` (hybrid only) | All features | CatBoost |\n| `blend` (hybrid only) | All features | Equal mean of cat_full and lgb_full probabilities |\n\nAll-negative and all-positive baselines are also reported. CatBoost uses depth 5, learning rate .04 and L2=6. LightGBM uses 15 leaves, learning rate .035, L2=5, minimum child samples=20 and column fraction .85. Both default to 500 iterations. Full settings are in `modeling.py` and the exported configuration. No early stopping sees an outer test fold. No SMOTE, pseudo-labels or test data are used for training.\n\nThreshold tuning addresses the F1/imbalance tradeoff; synthetic oversampling and class weighting are not presumed beneficial. The blend weights are fixed, not optimized on test results. The basic/full LightGBM comparison is an ablation of the added features. `lightgbm` is the default backend. The optional `hybrid` backend adds CatBoost, whose local execution was blocked by unavailable `/proc/self/statm` in the authoring sandbox. No CatBoost score or successful fit is claimed from that environment.\n\n## Nested evaluation\n\nDefault: five outer folds and three inner folds, stratified by target. With genuine source-group IDs, both levels use stratified group splits and validate group disjointness. If no such mapping exists, reports explicitly say `object_only`; they do not claim source-template independence.\n\nFor each outer fold:\n\n1. Keep its test objects out of every fitting and selection operation.\n2. Generate inner out-of-fold probabilities for every candidate using only the outer training portion. Fit all missing-value imputers on the corresponding inner fit rows.\n3. For each candidate, choose the threshold maximizing inner F1. Equal F1 thresholds select the higher threshold. Candidate ties prefer the earlier, simpler candidate in the declared order.\n4. Refit on the outer training portion, freeze the selected candidate and threshold, and score the outer test portion once.\n5. Export every split membership, tuning choice, probability and binary decision.\n\nThe primary reported F1 pools the outer test decisions from the complete nested selection procedure. Per-candidate outer scores evaluate their inner-selected thresholds. PR-AUC (average precision), precision, recall, ROC AUC, prevalence and confusion counts supplement F1.\n\nThe conditional bootstrap resamples fixed OOF decisions by object, or by source group when supplied. It does not rerun training or account for previous human/agent choices, feature exploration, or model selection across repeated experiments. It is descriptive, not proof of leaderboard superiority.\n\n## Deployment recipe\n\nUse candidate OOF probabilities across the training set to choose a final candidate and threshold; explicitly label that optimum as a tuning score. Refit that recipe on all labelled training objects and predict the test set. OOF-to-full-refit probability distribution changes are a known threshold-transfer risk.\n\nSubmission is exactly `object_id,prediction`, with binary integers and the official sample order. The package never submits it to Kaggle. Test labels are unavailable, so private leaderboard performance cannot be computed locally.\n\n## Interpretation\n\nLocal object-level validation may overestimate generalization when several simulated objects share an original source. Sparse cadence, redshift shifts, tiny positive counts and repeated development also matter. This first protocol prioritizes auditability and a tractable boosted-tree baseline inspired by top solutions. Adding pretrained neural networks, Gaussian processes, physical fitting or larger ensembles would be a new, separately evaluated experiment.\n", 'docs/PUBLICATION_CHECKPOINT.md': "# Publication checkpoint — 5 October 2026\n\nUser authorization: update the MALLORN project on GitHub and upload its code as\na Kaggle notebook. This supersedes the earlier pause on code publication.\nIt does not request another competition prediction submission.\n\n## Verified starting state\n\n- GitHub: `Ehsan-Ch/mallorn-tde-classifier`, branch `main`, commit\n  `d05fdd34816264360a9050bb24f7d79d2b929d19`.\n- Kaggle account: `ehsan1993`, display name Ehsan.Ch. Your Work / Code was empty.\n- Existing prediction submission: `mallorn-improvement-submission.csv`, private\n  F1 0.6648, public F1 0.6825, completed after the competition deadline.\n- The workspace was restored to local commit `5c71f33`, containing research\n  through cycles 3 and 4. The later cycle-5 fitted models and the 438,808,626-byte\n  `mallorn-recovery-20261005.zip` are not present in this restored workspace.\n  Do not claim those artifacts have been recovered. Their earlier reported\n  outcomes remain historical session evidence, not freshly replayed results.\n\n## Publication steps\n\n1. Prepare a self-contained Kaggle notebook and current README/status notes.\n2. Validate notebook structure, embedded source and a synthetic smoke run.\n3. Publish GitHub files atomically as one commit based on the current main tip.\n   Verify remote tree hashes before recording GitHub as complete.\n4. Inspect Kaggle Your Work before creating anything. If the notebook exists,\n   update that exact notebook; do not create duplicates after an interruption.\n5. Upload `notebooks/mallorn_research.ipynb`, save a version, verify its page\n   and version status, and record its exact URL.\n6. Publish final links and version receipts to this repository.\n\n## Resume after the user says continue\n\nRead this file and `publication/state.json` in the remote GitHub repository\nfirst. Remote state is authoritative if the local workspace rolled back.\nVerify the existing Kaggle notebook URL/version before any new upload. An\nin-flight save must be inspected, never assumed failed. Continue only incomplete\nsteps. Keep historical model and leaderboard scores distinct. Never insert a\nreported score into a notebook's execution outputs as if that run computed it.\n\nToken-reset detection is unavailable. The user gives the continuation signal.\nStored source and publication receipts survive through GitHub; running processes\nand unuploaded model artifacts cannot be guaranteed across workspace resets.\n", 'docs/RESUME_RESEARCH.md': "# Restartable local research\n\nCurrent scope: research runners operate locally and never publish automatically.\nOn 5 October 2026 the user separately authorized a GitHub source update and a\nKaggle code notebook. See [publication checkpoint](PUBLICATION_CHECKPOINT.md).\nThis publication does not request another prediction submission or paid compute.\nNo Excel output. The competitive target is private F1 above 0.6824, but only\na future organizer-scored evaluation can confirm that. Best verified private\nF1 remains 0.6648. Local development results must retain their own labels.\n\n## Resume command\n\nFrom the repository root, with dependencies installed:\n\n```bash\nMPLCONFIGDIR=artifacts/mpl OPENBLAS_NUM_THREADS=1 MALLORN_THREADS=4 \\\n  .venv/bin/python scripts/resume_research.py run --max-seconds 1500\n```\n\nInspect progress without starting work:\n\n```bash\n.venv/bin/python scripts/resume_research.py status\n```\n\nIn a fresh environment, create the virtual environment and install the pinned\nproject dependencies first. Do not change the frozen package versions to make a\ncheckpoint load. On Windows, set the environment variables with PowerShell and\nuse `.venv\\Scripts\\python.exe` instead. The runner's filesystem lock has Linux\nand Windows implementations; only the Linux path has been exercised here.\n\n## What continues automatically\n\nThe worker completes the fixed cycle-3 and cycle-4 queues without needing another chat\nmessage for each fold. It saves each fold atomically and then moves to the next.\nThe wall-time budget is checked at fold boundaries, so a running fold may\nfinish slightly after that budget. On restart, completed checkpoints are\nverified and skipped; only an interrupted, uncommitted fold is retrained.\n\nAn OS lock prevents two workers from writing the same experiment. The lock is\nreleased if a process crashes. A changed configuration, package version, source\nhash, input hash or validation assignment causes a visible error. Never delete\nthe manifest or edit it to force incompatible checkpoints to load.\n\nThe compatible LightGBM work runs before the blocked cycle-2 CatBoost full-data\nrefit. If CatBoost's process-statistics access becomes available, the worker\nattempts that refit locally. Otherwise it records the blocker and exits. It\ndoes not modify sandbox permissions or emulate the missing process file.\nThe new `scripts/refit_frozen.py` preserves the original fitting recipe while\nsaving each complete model and its probabilities atomically. It verifies saved\npredictions on replay and writes the final output manifest last. This full-data\npath is implemented but cannot be exercised until native CatBoost is available.\n\nThe bounded queue stops when complete. Repeated scheduled calls do not launch\nmore hyperparameter trials or change frozen thresholds. New research needs a\nnew written protocol and output directory, preserving all earlier outcomes.\n\n## Limits of unattended continuation\n\nThere is no token-reset event exposed to this code. A scheduled chat check can\nattempt continuation once usage allows, at most hourly with the available\nscheduler. It does not bypass usage limits, keep this compute environment\nalive, or guarantee access to this workspace in another run. Web scheduled\ntasks do not preserve local folders between runs; see the [official scheduled\ntasks documentation](https://learn.chatgpt.com/docs/automations).\n\nIf the workspace or its authorized data is unavailable, the scheduled task must\nreport the blocker and stop, rather than recreate results or read unrelated\nfiles. Do not repeatedly notify the user about an unchanged blocker. A saved\nexperiment archive preserves expensive outputs; it does not by itself provide\na running machine or an automatic restore service.\n\n## Continuation records\n\n- `artifacts/research_state.json`: latest worker outcome and concrete blocker.\n- `artifacts/cycle3/manifest.json`: frozen code, input hashes and package versions.\n- `artifacts/cycle3/progress.json`: active fold or completion.\n- `artifacts/cycle3/*/repeat*_fold*.joblib`: atomic fold model, probabilities and audit.\n- `artifacts/cycle3/comparison.json`: all six prespecified recipes and selection.\n- `artifacts/cycle4/`: the separate confusing-negative experiment, with the same\n  atomic checkpoint structure and five fixed recipe comparisons.\n- `artifacts/cycle2/frozen_comparison.json`: unchanged preceding candidate.\n\nDo not load arbitrary untrusted joblib files. These checkpoints are produced\nby this project from the user-authorized competition data.\n", 'docs/SECOND_CYCLE_PROTOCOL.md': "# Second improvement cycle — 4 October 2026\n\nReference: the unchanged first submitted recipe scored private F1 0.6648 and\npublic F1 0.6825. Its reserved audit has been opened. This cycle uses all 3,043\nlabeled objects, and makes no claim to have a new untouched local test set.\n\nBefore looking at new results, fix two stratified five-fold repetitions with\nsplit seeds 20261014 and 20261015. Each object receives one held-out prediction\nper repetition. Features and any supervised column selection remain confined\nto each fitting fold. No fitting uses competition test labels.\n\nCompare four already motivated components: the submitted physics-only\nthree-class CatBoost model, the submitted selected-feature binary CatBoost\nmodel, selected-feature three-class CatBoost, and the physical-feature\nhierarchical AGN/transient classifier. Do not add new hyperparameter trials\nwithin this cycle. Compare exactly three equal-weight recipes:\n\n1. Reference: the two submitted component types.\n2. Selected-class ensemble: selected binary plus selected three-class model.\n3. Diverse ensemble: all four component types.\n\nAverage each object's two held-out predictions. Choose each recipe's threshold\nas the median of 500 stratified-bootstrap F1-optimal development thresholds,\nwith bootstrap seed 20261016. Record both repeat-specific and combined scores,\nprecision, recall and average precision. These remain tuning/selection scores.\n\nPrefer an alternative recipe only if its combined F1 exceeds the reference by\nat least 0.01 and its average precision is no more than 0.005 lower. Otherwise\nretain the reference component types. Refit the chosen recipe with two fixed\nseeds (42 and 142) on all training objects and average the resulting probabilities.\nFreeze the recipe and threshold before any new organizer-scored submission.\n\nThis experiment tests whether use of all labeled objects, repeated validation,\nseed averaging and a less brittle threshold improve the prior result. The\nfirst Kaggle result is a reference, not a source for per-object label inference\nor a threshold sweep. The private leaderboard is now observed and cannot be\nclaimed as a never-used research test after further submissions.\n", 'docs/SOURCES.md': '# Sources and attribution\n\nUpdated 3 October 2026. These sources inform the design; no competitor code, pretrained weights, raw data, feature matrices or prediction files were copied.\n\n| Source | What informed this implementation | Difference |\n| --- | --- | --- |\n| [Official competition](https://www.kaggle.com/competitions/mallorn-astronomical-classification-challenge) and [data page](https://www.kaggle.com/competitions/mallorn-astronomical-classification-challenge/data) | Binary TDE target, F1 metric, six-band light curves and submission schema | Official archive supplied by the user and evaluated locally; no organizer score obtained |\n| [Suxing: second-place solution](https://www.kaggle.com/competitions/mallorn-astronomical-classification-challenge/writeups/mallorn-2nd-place-solution) | Shape, phase-binned and colour features; rest-frame timing, feature views and boosted-tree ensembles | Fixed CatBoost/LightGBM candidates and equal-weight blends; no pretrained CNN or post-processing zones |\n| [Sigrid Nissen: third-place solution](https://www.kaggle.com/competitions/mallorn-astronomical-classification-challenge/writeups/3rd-place-write-up-catboost-with-threshold-tuning) and [author repository](https://github.com/sigridhn/MALLORN-solution-sigrid) | Physics-informed light-curve characterization, GP/thermal features, CatBoost and threshold tuning | Independently implemented capped Matérn GP and monochromatic blackbody approximations; no SALT2 or echo-state network |\n| [Avocado](https://arxiv.org/abs/1907.04690) | Joint time/wavelength Gaussian-process light-curve representation | Lightweight feature extractor; no full Avocado pipeline or external augmentation |\n| [Magill et al.: MALLORN paper](https://arxiv.org/abs/2512.04946) | Simulated light curves based on real source observations and the train/test redshift distinction | No claim that object-level folds remove common-source dependence |\n| [Fitzpatrick99 implementation](https://extinction.readthedocs.io/en/latest/api/extinction.fitzpatrick99.html) | Dust extinction correction in the `extinction` library | Representative wavelength approximation, not integrated photometry |\n| [SciPy Lomb–Scargle](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.lombscargle.html) | Weighted periodogram for uneven sampling | Descriptive features only; no significance or physical-period claim |\n| [CatBoost API](https://catboost.ai/docs/en/concepts/python-reference_catboostclassifier) | CPU classifier configuration | Independently fixed settings |\n\nThe finalized leaderboard and later write-up titles can describe different rankings. A write-up titled “2nd Place Private LB (0.6804)” also identifies itself as a fifth-place solution with a late-submission result. It was not used as evidence of the official second-place finish. This project does not claim to reproduce an unverified first-place method.\n\nNested model/threshold selection, source hashes, strict file contracts, synthetic-only integrity fixtures and the evidence export are implementation choices for this repository. Their usefulness must be separated from empirical performance: no feature set is called superior without running the corresponding experiment on authorized data.\n', 'docs/THIRD_CYCLE_PROTOCOL.md': "# Third cycle: model diversity, fixed before training\n\nThe goal is to improve the second cycle's local F1 0.684058 and average\nprecision 0.677698. All labels have historical exposure; these are development\nexperiments, not untouched validation. The verified Kaggle private score remains\n0.6648; the top private benchmark is 0.6824. Neither a higher development score\nnor repeated tuning establishes that the benchmark has been surpassed.\n\nUse the same two stratified five-fold partitions (20261014, 20261015) and\nobject order as cycle 2. Freeze three LightGBM components:\n\n1. Three-class AGN/other-transient/TDE supervision on 471 physics features.\n2. An AGN/transient gate multiplied by a conditional TDE classifier on the\n   736 physical-view features; fit the second stage only on training transients.\n3. Binary classification on the 1,126 complete features.\n\nEach estimator uses 900 trees, learning rate 0.025, nine leaves, at least\n30 objects per leaf, L2 penalty 10, column sampling 0.8, row sampling 0.85\nevery iteration, and class weights 1:3 for TDE versus other classes. The gate\nuses equal class weights. Seed is 42 + 100*repeat + fold. No early stopping on\nouter validation labels, feature selection, extra trials or hyperparameter sweep.\n\nEvaluate exactly six recipes: the frozen cycle-2 diverse reference; a 50:50\nblend of that reference with each of the three new components; the equal-weight\nthree-LightGBM ensemble; and a 50:50 blend of that ensemble with the reference.\nAverage repeated held-out probabilities per object. Threshold selection uses\nthe same 500 stratified bootstraps as cycle 2 (seed 20261016).\n\nSelect an alternative only if combined F1 improves by at least 0.005, average\nprecision does not decrease, and neither repetition's F1 at the fixed threshold\nis more than 0.005 below the reference's corresponding repetition. If none\nqualifies, retain the reference. Break equal-F1 ties by recipe declaration order.\nReport every recipe, not just the winner. Record bootstrap paired F1 differences\nat frozen thresholds as descriptive uncertainty; these intervals do not correct\nfor historical tuning or simulated-source dependence.\n\nSave each complete fitted fold, held-out probabilities, row IDs and input/source\nhashes atomically. An incomplete fold is rerun; committed folds are verified and\nreused. An OS lock prevents concurrent writers. Never silently reuse a checkpoint\nwith a changed specification, code, data, package version or validation assignment.\n\nOnly local research is authorized in this cycle. Do not upload to GitHub or\nKaggle. A completed queue must stop; failed work records an error rather than\nretrying indefinitely. CatBoost's blocked full-data refit is a separate pending\ntask and must not prevent compatible LightGBM work.\n", 'pyproject.toml': '[build-system]\nrequires = ["setuptools>=69"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "mallorn-tde-classifier"\nversion = "0.1.0"\ndescription = "Reproducible MALLORN light-curve classification with nested F1 threshold selection"\nrequires-python = ">=3.11"\ndependencies = [\n  "numpy==2.3.5", "pandas==2.2.3", "scipy==1.17.0", "scikit-learn==1.8.0",\n  "catboost==1.2.8", "lightgbm==4.6.0", "extinction==0.4.8",\n  "matplotlib==3.10.8", "joblib==1.5.3"\n]\n\n[project.scripts]\nmallorn = "mallorn.cli:main"\n\n[tool.setuptools.packages.find]\nwhere = ["src"]\n', 'LICENSE': 'MIT License\n\nCopyright (c) 2026 Ehsan Cheraghi\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n'}
SOURCE_HASHES = {'src/mallorn/__init__.py': 'ccdbdabbc1911dd7275f3dbeb771be8cf2deab15a52bd12901ff1b622542672a', 'src/mallorn/checkpointing.py': 'ac795168654edb852e1e6847ec4a01689f774c36f0e4b6e4463b29b9934c3541', 'src/mallorn/cli.py': 'ca30853fcf88a7e00baaea4bebb11342e739490998fcbaeabaa55b17ec825eca', 'src/mallorn/data.py': 'ad3dfde8ee4b2a78f7e051bbe125005abdf00ce9ef800b765d48441b620716dc', 'src/mallorn/evaluation.py': '208d94fc64e92c3c98caa6a2f18919a0f2b1a6935be8d2f27931022b17be1604', 'src/mallorn/features.py': '03129c595e3849cfe3fd56a3b410375c2b173c2ca24ff471a0020e4ed535d6f4', 'src/mallorn/fixtures.py': '422f89651e423efebf783aa27af9240c3b539a0ff7b71a2b66b2d8fe5fb3bd3c', 'src/mallorn/modeling.py': '39720c592a2fe4ba7d9b294d8e2f1d2b780261b261aaf14135f02e1d94e8b7c8', 'src/mallorn/morphology_features.py': 'ddcbacf3ddf646732de9fa3944e52687ef42051d98478aa7c55e62035afd686a', 'src/mallorn/physics_features.py': '6d55dd01336f7162de2f7d60520f02476e9d6a176b787a32552ead3931581776', 'src/mallorn/reporting.py': 'af5aa9077d54b9eb99830b94942c91b060f65d92713ea7a37cbdfb8c50e79f06', 'src/mallorn/selected_ensemble.py': '41d4752b162a9bde5baa1a8ca45a595cbc5fc8865a93c6360ff3cf7382aa08e9', 'scripts/audit_improvement.py': 'a7fb78d4aede2555956aef0c07971ad21f4ee2e7784fdf51261f3bc2cef4e3b4', 'scripts/build_morphology_features.py': 'e0cd0228a042edb601da1444ba3095bba101271b860e166df1e9bf8e1b648f80', 'scripts/build_physical_view.py': '81282152f581d16edb6f8c96d7eea498cab68456ce963f1d6b7de0e61745d796', 'scripts/build_physics_features.py': 'cbd8e66074d8f5cf17df8646fa906b29c8184e6ff99381a00ab01d84d7b99f5a', 'scripts/compare_development.py': '71054085d8cff9087965d1610ae5e8c24c7cddd53989afcc7ed1aefe7030b701', 'scripts/cycle2.py': '096913a256a2a80902e44fa21ea3371f1749f43ed517b956380b364c3e06b386', 'scripts/cycle3.py': 'f96804f7691b43e744822a3e98ebb410d1f2ac8cf69da3f13b006ea7d041c3a4', 'scripts/cycle4.py': 'abba851225374745cd077a0ecb9bfe03eca3b7a5008dc6d993d5241c6e9858db', 'scripts/cycle5_recovered.py': '9fa19081b8f7dd5a5ab8f039fd1571ebbbd72b2251632717e230d77d027eaae7', 'scripts/deploy_improvement.py': 'd2f2c7081e112b0103540652a35eddf019f219846e618e501fb1a2c91f8d5e88', 'scripts/improve.py': '20f60f539a8b7d4ca24db63978328906efa47b4a8468543b9d3293bf76424f70', 'scripts/notebook_smoke.py': '77d517f0c2e95b393f5dc3c1c62cc839d62d3c3ce52fb55e6a66b99c9411c448', 'scripts/refit_frozen.py': 'efde68b85a828ab96fe43d11a06167ed18cec9d3214f936576a08165283bff87', 'scripts/report_research.py': '41537b441c4b22be09b6cf4d1e6e35469ac2a3aa8a0be7d5829c59615925435e', 'scripts/resume_research.py': 'f4ee54957cbeac264b70862622e69c7300c3720d1b2d229740d189936df02ace', 'scripts/verify_improvement.py': 'e49314c16d90aa680b48b2a5a0a1e79ac7274102c343dec600cfe5f8374bcecf', 'scripts/verify_run.py': 'b46b63852d9121ac4259226dd4956bde04679f92e1413974703d5a4664d5cc8b', 'tests/test_checkpointing.py': '58f334263cb1709eb5c0e4ffdb943498a367cf07587393360eb3e384c37233b1', 'tests/test_integrity.py': '76288bfe38844487a4625e226a6c92e90197156c4a1706d3d5554e4bd11ab53a', 'tests/test_morphology_features.py': '45327c458b97477f92591b84ef70d0eff27313d0bc1bea5df06ba36342cc7f79', 'tests/test_physics_features.py': 'cb0d9eecf84d4951831049b25cb8f56b3512a649983cc6d9c32e4691fb5eb87b', 'tests/test_selected_ensemble.py': '453d21ffefecb3753f5f2c9fd0e5a81f0f63ef53d6cda51d2fa2a44c0c665b4f', 'docs/DATA.md': '10598e729a86644b6c60ad3cbda049661e704438c786c83c20daa3352fd89623', 'docs/FIFTH_CYCLE_RECOVERY.md': 'f836bd3e65c42a1558657123d678eb5e81da1d0b13737bcd7df1dd77299274a9', 'docs/FOURTH_CYCLE_PROTOCOL.md': '6ed4378a05c41beee584a77a1bedf513081f6732b7a5562e126a79dc1c0f027a', 'docs/IMPROVEMENT_METHODS.md': 'caed50512195641d7b00ec48cb03ce16ae30e0559847486f24fe4482f21dd5b7', 'docs/IMPROVEMENT_PROTOCOL.md': 'f85dfae018080f053c33c5a4fe40eb888fd9b43309ea2fae265296b5818a08e8', 'docs/INTERVIEW_GUIDE.md': 'f5085762459b1155af8295d4a400cf009dc1e19ffb71e0b6f4ff3f30710af103', 'docs/METHODOLOGY.md': 'e80ca0f21cafe9cf242913077bd5d9d5f1406ba22b64a393770ef093f91fe98e', 'docs/PUBLICATION_CHECKPOINT.md': 'ae833c014fa5d476df459e1b50d172b77f5021fc6b8c3e47d83abf5508ea17a2', 'docs/RESUME_RESEARCH.md': 'a1d902b26055a7e707e9df20df2e339f6498ce1da3131f59c459daf8fc263380', 'docs/SECOND_CYCLE_PROTOCOL.md': '9685b5fe2ffc6489a6ba8735287fe300db32619b63bf01d54c3fd6f7e694390e', 'docs/SOURCES.md': '347ca726334347598a5d3c6185f2abebadd79af06d247c84048e4aabbfd4268f', 'docs/THIRD_CYCLE_PROTOCOL.md': 'ea15c77f084881e996865a2167f8f66f6771fe65784dcd6ebd7f41b88e09e95f', 'pyproject.toml': '1810deb32748c929c2382ae6bbd99f48e00c17dea45cf7b4e6960ac5fa60af45', 'LICENSE': '4235a55b305728c51e2a49c3cddf99aaf383a2fb82187eff82c47bf0570d8f85'}
from pathlib import Path
import hashlib, json, os, subprocess, sys
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
PROJECT = WORK / 'mallorn_project'
PROJECT.mkdir(parents=True, exist_ok=True)
for name, source in SOURCE_FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.read_text() != source:
        raise RuntimeError(f'Existing source differs: {name}. Use a new work directory.')
    path.write_text(source)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == SOURCE_HASHES[name]
(PROJECT / 'source_manifest.json').write_text(json.dumps(SOURCE_HASHES, indent=2))
ENV = dict(os.environ, MPLCONFIGDIR=str(PROJECT/'artifacts/mpl'), OPENBLAS_NUM_THREADS='1', MALLORN_THREADS='2')
import importlib.metadata, tomllib
requirements = tomllib.loads(SOURCE_FILES['pyproject.toml'])['project']['dependencies']
def versions_match():
    try:
        return all(importlib.metadata.version(r.split('==')[0]) == r.split('==')[1] for r in requirements)
    except importlib.metadata.PackageNotFoundError:
        return False
PYTHON = Path(sys.executable)
if not versions_match():
    PYTHON = PROJECT / '.venv/bin/python'
    if not PYTHON.exists():
        subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(PROJECT/'.venv')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '--python', str(PYTHON), 'install', '--no-cache-dir', '-q', *requirements], check=True, env=ENV)
ENV['PYTHONPATH'] = str(PROJECT/'src')
def command(*args):
    subprocess.run([str(PYTHON), *map(str,args)], cwd=PROJECT, env=ENV, check=True)
print(f'Checked {len(SOURCE_FILES)} source files; pinned dependencies installed.')


## 1. Software check

The following run uses invented light curves to test software behavior.
Its score is **not competition performance**. A completed matching check is reused.


In [ ]:
SMOKE = PROJECT / 'artifacts/kaggle_smoke'
receipt = SMOKE / 'completed_source_hashes.json'
if receipt.exists():
    assert json.loads(receipt.read_text()) == SOURCE_HASHES
    print('Reusing a completed matching synthetic software check.')
else:
    if SMOKE.exists() and any(SMOKE.iterdir()):
        raise RuntimeError('An interrupted smoke directory exists. Inspect it and choose a new output name.')
    command('scripts/notebook_smoke.py','--output',SMOKE)
    receipt.write_text(json.dumps(SOURCE_HASHES,indent=2))
print('Synthetic software check completed. This is not a Kaggle score.')


## 2. Locate attached official data

This reads the competition input only. Official fitting is opt-in below.


In [ ]:
INPUT = Path('/kaggle/input')
matches = sorted(INPUT.rglob('train_log.csv')) if INPUT.exists() else []
DATA = next((p.parent for p in matches if (p.parent/'test_log.csv').exists() and (p.parent/'sample_submission.csv').exists()), None)
print('Official data:', DATA if DATA else 'Not attached. Add MALLORN via Add Input to enable training.')


## 3. Optional official nested baseline

Set `RUN_OFFICIAL = True` to train the original nested-validation workflow.
This writes predictions locally but does not submit them. The output folder
must be fresh; a completed matching run is reused by its receipt.


In [ ]:
RUN_OFFICIAL = False
OFFICIAL = PROJECT / 'artifacts/official_001'
if RUN_OFFICIAL:
    if DATA is None: raise RuntimeError('Attach the official MALLORN competition data first.')
    official_receipt = OFFICIAL/'completed_source_hashes.json'
    if official_receipt.exists():
        assert json.loads(official_receipt.read_text()) == SOURCE_HASHES
        print('Reusing completed matching official run.')
    else:
        command('-m','mallorn.cli','run','--data',DATA,'--output',OFFICIAL,'--threads','2')
        official_receipt.write_text(json.dumps(SOURCE_HASHES,indent=2))
else:
    print('Official model fitting is disabled for this publication run.')


## 4. Optional physical features and research code

After an official baseline, the builders can compute GP/thermal, flare and
physical feature views. A symlink references attached data without copying it.
Original cycles 2–4 and the recovered cycle-5 runner are included under `scripts/`.
Historical replay requires the original private checkpoint/reference artifacts;
the public notebook does not invent them. See `docs/RESUME_RESEARCH.md`.


In [ ]:
BUILD_PHYSICAL_FEATURES = False
if BUILD_PHYSICAL_FEATURES:
    if DATA is None or not (OFFICIAL/'features.csv').exists():
        raise RuntimeError('Complete the official baseline first.')
    (PROJECT/'data').mkdir(exist_ok=True)
    target = PROJECT/'data/mallorn'
    if not target.exists(): target.symlink_to(DATA, target_is_directory=True)
    if target.resolve() != DATA.resolve(): raise RuntimeError('Data link points at another dataset.')
    for mode in ('train','test'):
        command('scripts/build_physics_features.py','--mode',mode,'--workers','2')
        command('scripts/build_morphology_features.py','--mode',mode)
        command('scripts/build_physical_view.py','--mode',mode)
else:
    print('Physical feature extraction is disabled for this publication run.')


## Resume and limitations

Saved Kaggle versions preserve the notebook; export/download output checkpoints
before ending a session if you need them later. Local notebook folders alone
are not guaranteed to survive a new runtime. Source hashes and model receipts
must match before reusing artifacts. A usage reset cannot be detected by Python;
the assistant resumes publication after the user says **continue** and checks
`publication/state.json` on GitHub before creating or updating remote resources.

Object-level folds cannot establish source-template independence. Labels have
been repeatedly reused for later development selection. Accurate training
redshifts differ from noisy test redshifts. F1 thresholds may transfer poorly.

Code: MIT. Official data remain under Kaggle competition rules. Created with
OpenAI Codex assistance for Ehsan Cheraghi. Method references and distinctions:
https://github.com/Ehsan-Ch/mallorn-tde-classifier/blob/main/docs/SOURCES.md
